<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/NLP-2026/Lecture_4/%D0%9B%D0%B5%D0%BA%D1%86%D0%B8%D1%8F_4_1_%D0%92%D0%B2%D0%B5%D0%B4%D0%B5%D0%BD%D0%B8%D0%B5_%D0%B2_%D1%82%D0%BE%D0%BD%D0%BA%D1%83%D1%8E_%D0%BD%D0%B0%D1%81%D1%82%D1%80%D0%BE%D0%B9%D0%BA%D1%83_LLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лекция 4.1. Введение в тонкую настройку LLM

## Тема 1. Фундаментальные основы тонкой настройки: от задачи до выбора метода

В предыдущих лекциях мы научились общаться с большими языковыми моделями через промпты: задавали вопросы, давали примеры (few-shot), строили цепочки рассуждений (Chain‑of‑Thought) и даже подключали внешние базы знаний (RAG). Но у всех этих методов есть потолок: модель остаётся *той же самой* – она не запоминает ваш стиль, не усваивает узкоспециализированную терминологию и не может научиться новым форматам ответов, если они не умещаются в контекст.

**Цель этой лекции** – познакомить вас с тонкой настройкой (fine‑tuning) – процессом, который превращает универсальную модель в узкого эксперта, адаптированного под вашу задачу. Мы шаг за шагом решим следующие вопросы:

- **Что такое тонкая настройка и чем она отличается от предобучения?** – проведём чёткую грань между «высшим образованием» модели и её «стажировкой» под ваш проект.
- **Когда она действительно нужна, а когда можно обойтись промптами или RAG?** – выработаем системный подход к принятию решений, чтобы не тратить ресурсы впустую.
- **Какие существуют методы (полная настройка, LoRA, QLoRA) и как их сравнить?** – построим иерархию затрат и качества.
- **С какими проблемами мы столкнёмся (катастрофическое забывание, галлюцинации, переобучение) и как их избежать?** – дадим конкретные приёмы, которые спасут ваши эксперименты.

В этой лекции мы **не будем писать код обучения** – это будет в следующих темах. Но мы заложим фундамент, на котором вы сможете осознанно выбирать инструменты и избегать типичных ошибок. Каждый раздел будет сопровождаться расчётами, таблицами и реальными примерами из индустрии.

---

### 1.0. Подготовка: что мы уже умеем и чего нам не хватает

Прежде чем говорить о тонкой настройке, давайте честно оценим наши текущие возможности. Мы умеем:

- **Zero‑shot** – давать инструкцию без примеров. *Быстро, дёшево, но модель отвечает так, как «выучила» на предобучении, без адаптации.*
- **Few‑shot** – вставлять в промпт 2–5 эталонных пар. *Улучшает форматирование, но съедает контекст и не меняет веса.*
- **RAG** – подгружать релевантные документы из внешней базы. *Даёт актуальные факты, но не учит модель «думать» в вашей предметной области.*

А что, если задача требует:

- **Единого корпоративного тона** – например, ответы технической поддержки должны быть вежливыми, структурированными и содержать внутренние аббревиатуры?
- **Нового формата вывода** – не просто текст, а строгий JSON с фиксированными полями, который ни разу не нарушается?
- **Снижения затрат** – мы хотим заменить дорогой GPT‑4 на маленькую открытую модель, но она пока не умеет так же хорошо следовать инструкциям?

Вот тут и вступает тонкая настройка – мы берём предобученную модель и дообучаем её на наших собственных примерах, изменяя её веса (или часть весов) так, чтобы она начинала вести себя нужным образом.

---

### 1.1. Определение тонкой настройки и её место в жизненном цикле модели

**Тонкая настройка** (fine‑tuning) – это процесс продолжения обучения предварительно обученной модели на целевом датасете, который отражает вашу конкретную задачу. Формально, если предобучение решало задачу минимизации потерь $\mathcal{L}_{pre}(\Theta)$ на общем корпусе $\mathcal{D}_{general}$ (триллионы токенов), то тонкая настройка решает:

$$
\Theta^* = \arg\min_{\Theta'} \mathcal{L}_{task}(\Theta') \quad \text{при инициализации} \quad \Theta'_0 = \Theta_{pre},
$$

где $\mathcal{L}_{task}$ – функция потерь для вашей задачи (обычно кросс‑энтропия для генерации), а скорость обучения берётся значительно ниже, чтобы не разрушить уже выученные знания.

**Аналогия из жизни.** Предобучение – это **фундаментальное высшее образование**: студент изучает математику, физику, историю, литературу, учится писать сочинения и решать уравнения. Он становится эрудированным генералистом. Тонкая настройка – это **стажировка в конкретной компании**: новичок узнаёт внутренние правила, жаргон, стандарты оформления отчётов, учится работать с корпоративной документацией. Он становится узким специалистом, который применяет свои знания в конкретной среде.

**Место в жизненном цикле.** Тонкая настройка – это не разовая акция, а один из этапов MLOps-конвейера:

1. **Сбор и очистка данных** – формируем датасет из пар «вопрос–ответ».
2. **Препроцессинг** – токенизация, применение чат-шаблона, разбивка на обучающую/валидационную выборки.
3. **Загрузка предобученной модели** (например, Llama 3 8B).
4. **Тонкая настройка** (полная или через LoRA) – обновляем веса.
5. **Оценка** – сравниваем с бейзлайном (промптами) на валидации.
6. **Деплой** – разворачиваем в продакшн.
7. **Мониторинг и A/B‑тестирование** – сравниваем новую модель со старой на реальном трафике; при обнаружении дрейфа данных запускаем повторное обучение.

> **Важно:** Тонкая настройка – это не «волшебная таблетка». Если у вас менее 500–1000 качественных примеров, вы рискуете переобучить модель. Если ваши данные часто меняются – лучше использовать RAG. Всегда начинайте с промптов и только при экспериментальном доказательстве их недостаточности переходите к изменению весов.

---

### 1.2. Пять ключевых причин для применения тонкой настройки

Решение о дообучении должно быть обоснованным. Вот пять сценариев, где тонкая настройка даёт критическое преимущество:

**1. Освоение узкоспециализированной терминологии и доменных знаний.**  
Модель не знает сокращений вашей отрасли (например, в медицине – «Т2‑ВИ», «DWI», «ADC»; в юриспруденции – ссылки на статьи кодекса). Тонкая настройка на размеченных экспертами текстах позволяет модели правильно использовать эти термины в контексте.  
*Пример:* дообученная на 10 000 радиологических заключений модель Llama‑3 достигает точности 92% при сопоставлении с кодами МКБ‑10, тогда как базовая модель даёт только 78%.

**2. Принудительное структурирование выходного формата.**  
Если вам нужно, чтобы модель всегда выдавала строго определённый JSON или XML – тонкая настройка гораздо надёжнее, чем сложные промпты, которые могут сломаться при смене версии модели.  
*Пример:* в службе поддержки модель обучают на датасете баг-репортов Jira и она стабильно генерирует поля «Критичность», «Шаги воспроизведения», «Ожидаемый результат» без единой ошибки парсинга.

**3. Подавление галлюцинаций для статичных фактов.**  
Для часто используемых справочных данных (тарифы, законодательные нормы, технические регламенты) тонкая настройка «зашивает» эти факты в веса, и модель перестаёт выдумывать несуществующие детали.  
*Пример:* в телеком-чате дообучение на актуальном прайсе снизило долю вымышленных акций с 12% до 2%.  
> **Предупреждение:** если в датасете есть ошибки или противоречия – модель их **усилит**, а не исправит. Поэтому критически важна очистка данных и добавление **негативных примеров** (negative sampling), где модель должна отвечать «Я не знаю».

**4. Снижение задержки и стоимости инференса.**  
Заменив дорогую GPT‑4 на дообученную Mistral‑7B, вы можете сократить латентность с 800 мс до 150 мс при сохранении качества (F1‑мера 0,94 против 0,95). Это особенно важно для высоконагруженных систем.

**5. Обеспечение безопасности и выравнивания ценностей.**  
Тонкая настройка на корпусах «красной команды» (red‑teaming) учит модель категорически отказываться от вредных действий, даже если запрос замаскирован под сложную головоломку.  
*Пример:* банковский ассистент, дообученный на 10 000 атакующих промптов, даёт 100% отказов на запросы типа «Как обналичить чужую карту?» в любых формулировках.

---

### 1.3. Когда тонкая настройка **не нужна** – альтернативные подходы

Тонкая настройка – это мощно, но дорого и медленно. Если ваш случай попадает под один из пунктов ниже, **остановитесь и примените более простой метод**:

| Ситуация | Почему не стоит настраивать | Что делать вместо этого |
| :--- | :--- | :--- |
| **Меньше 500–1000 размеченных примеров** | Скорее всего, переобучитесь – модель запомнит шум, а не общий паттерн. | Few‑shot промптинг (3–5 примеров) или Zero‑shot с CoT. |
| **Данные обновляются каждую неделю или чаще** | Каждый раз переобучать модель – дорого и долго; она не успевает «забыть» устаревшее. | RAG – обновляйте векторную базу мгновенно. |
| **Требуется творческая генерация (стихи, креативный текст)** | Тонкая настройка уменьшает перплексию и делает модель более предсказуемой, убивая вариативность. | Регулируйте температуру, top_p, штрафы на повторы. |
| **Быстрый прототип (MVP)** | Тратить дни на обучение ради проверки гипотезы – неоправданно. | Zero‑shot с цепочкой рассуждений (CoT) – проверить идею можно за минуты. |
| **Нет GPU с достаточной памятью** | Даже QLoRA для 70B требует 20+ ГБ VRAM, а полная настройка – 80+. | Используйте API (например, через облачных провайдеров) или легковесные модели. |

**Важное правило:** всегда начинайте с самого простого метода (ступень 0) и поднимайтесь выше только при экспериментальном доказательстве, что предыдущий уровень не даёт нужного качества. Переход на каждую следующую ступень должен приносить прирост метрик (например, +15% по точности) – иначе экономически нецелесообразно.

---

### 1.4. Иерархическая пирамида методов: от промптов до полной настройки

Чтобы визуально представить соотношение затрат и качества, используем квадрантную диаграмму:

```mermaid
quadrantChart
    title "Сложность внедрения против качества адаптации"
    x-axis "Низкая сложность" --> "Высокая сложность"
    y-axis "Низкое качество" --> "Высокое качество"
    quadrant-1 "Дорогое качество"
    quadrant-2 "Быстрые решения"
    quadrant-3 "Эксперименты"
    quadrant-4 "Риск переобучения"
    "Zero-shot": [0.05, 0.1]
    "Few-shot": [0.15, 0.25]
    "RAG": [0.3, 0.65]
    "LoRA/QLoRA": [0.75, 0.92]
    "Full FT": [0.95, 0.99]
```

Рассмотрим каждую ступень:

| Ступень | Метод | Время внедрения | Стоимость обучения | Стоимость инференса | Изменение весов | Когда применять |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| **0** | Zero‑shot | минуты | $0 | 1× | нет | Простые вопросы, перевод, суммаризация |
| **1** | Few‑shot | часы (написание примеров) | $0 | 2–4× | нет | Форматирование, нестабильный формат, быстрая адаптация |
| **2** | RAG | дни (настройка индекса) | $ (эмбеддинги) | 5–10× | нет | Актуальные базы знаний, часто обновляемые данные |
| **3** | LoRA / QLoRA | дни (сбор данных, обучение) | $$ (5–50$ за запуск) | 1× | да (адаптер) | Изменение стиля, логики, голоса при ограниченных ресурсах |
| **4** | Full Fine‑Tuning | недели | $$$$ (100–10000$) | 1× | да (все веса) | Фундаментальная смена домена, новый язык, огромные датасеты |

> **Обратите внимание:** LoRA практически не увеличивает латентность инференса (адаптер можно слить с основными весами), а полная настройка может замедлить генерацию на 10–30% из‑за изменения распределения весов. Это важный инженерный фактор при выборе.

---

### 1.5. Сравнение подходов: SFT vs Preference Tuning (DPO/RLHF)

Прежде чем перейти к практическим расчётам, важно разделить две философии тонкой настройки, которые не следует путать:

- **Supervised Fine‑Tuning (SFT)** – классическое обучение с учителем: модель минимизирует расхождение между своими предсказаниями и **эталонными ответами** из датасета. Она учится имитировать эксперта. Применяется, когда есть единственно верный ответ (перевод, суммаризация, извлечение данных).

- **Preference Tuning (DPO, RLHF)** – модель учится **выбирать** предпочтительный ответ из пары (хороший/плохой). Она не копирует, а ранжирует. Используется для полировки стиля, безопасности, креативности, когда важна субъективная оценка.

**Практическое правило:** в большинстве проектов начинают с SFT, чтобы модель научилась формату, а затем, если нужно, применяют DPO для улучшения «личности». В нашем курсе мы сосредоточимся на SFT, но упомянем DPO как следующий уровень.

---

### 1.6. Расчёт ресурсов: сколько памяти нужно для тонкой настройки?

Чтобы вы могли прикинуть, потянет ли ваш железо тот или иной метод, запомните формулу для полной настройки (Full FT) с оптимизатором Adam:

- Параметры модели в bf16: `размер_модели` × 2 байта (например, 7B → 14 ГБ).
- Градиенты: ещё столько же (14 ГБ).
- Оптимизатор Adam: хранит два момента (среднее и квадрат градиента) – ещё 2 × 14 ГБ = 28 ГБ.
- Итого для 7B модели в bf16: **14 + 14 + 28 = 56 ГБ** (плюс накладные расходы, итого ~84 ГБ).

А для LoRA:
- Базовая модель замораживается – память только на неё (14 ГБ для 7B).
- Адаптеры: если ранг r=16, число обучаемых параметров ≈ 0,1% от полной модели, т.е. ~0,1% × 14 ГБ ≈ 14 МБ – ничтожно.
- Итого: **< 15 ГБ** для 7B модели. QLoRA дополнительно квантует базовую модель в 4 бита → память падает до ~3,5 ГБ для 7B.

| Метод | 7B модель | 13B модель | 70B модель |
| :--- | :--- | :--- | :--- |
| Full FT (bf16) | ~84 ГБ | ~156 ГБ | ~840 ГБ |
| LoRA (bf16) | ~15 ГБ | ~28 ГБ | ~140 ГБ |
| QLoRA (4‑bit + LoRA) | < 5 ГБ | < 9 ГБ | < 40 ГБ |

**Вывод:** QLoRA позволяет дообучать 70B модели на одном 48‑гигабайтном GPU, что было невозможно ещё год назад.

---

### 1.7. Основные проблемы тонкой настройки и способы их предотвращения

Теперь, когда мы знаем «зачем» и «почём», давайте посмотрим на подводные камни.

**1. Катастрофическое забывание** – модель перестаёт отвечать на общие вопросы, потому что веса «затерлись» новой задачей.  
*Симптом:* после обучения на медицинских данных модель разучилась писать стихи.  
*Как бороться:*
- Используйте PEFT (LoRA) – заморозьте основную модель, меняйте только адаптеры.
- Добавляйте в датасет небольшую долю старых данных (replay).
- Применяйте регуляризацию (весовой распад, низкую скорость обучения).
- В продвинутых случаях – Elastic Weight Consolidation (EWC).

**2. Переобучение** – модель отлично отвечает на тренировочных примерах, но плохо на валидации.  
*Симптомы:* eval loss растёт, а train loss падает.  
*Решение:* обучайте не более 1–3 эпох, используйте раннюю остановку, обязательно выделяйте валидационную выборку (10–20%).

**3. Усиление галлюцинаций** – если в датасете есть неточности, модель «выучивает» их и начинает уверенно врать.  
*Решение:* тщательная очистка данных, добавление **негативных примеров** (negative sampling) – пар, где правильный ответ – «Я не знаю» или «Информация отсутствует».

**4. Ошибки форматирования** – частая причина NaN потерь.  
*Решение:* всегда применяйте `tokenizer.apply_chat_template()` – это гарантирует, что модель правильно интерпретирует роли (system, user, assistant).

---

### 1.8. Типичные ошибки новичков и как их избежать

| Ошибка | Последствие | Решение |
| :--- | :--- | :--- |
| **Слишком низкая скорость обучения (LR)** | Адаптер не обучается, качество не растёт. | Для LoRA/QLoRA используйте LR ≈ 5e‑5 (для полной точности) или 2e‑4 (для 4‑бит). |
| **Слишком высокая скорость обучения** | Потери взлетают, модель ломается. | Начинайте с маленького LR и повышайте, пока не увидите стабильное снижение eval loss. |
| **Забыли про pad token** | NaN в loss, обучение останавливается. | Установите `tokenizer.pad_token = tokenizer.eos_token`. |
| **Не используете валидацию** | Не замечаете переобучение. | Всегда выделяйте валидационный холд‑аут. |
| **Случайно разморозили базовую модель при LoRA** | Теряется смысл PEFT – обучаются все веса, память переполняется. | Проверьте, что `model.peft_config` содержит замороженные слои. |
| **Используете сырые данные без чат‑шаблона** | Модель не понимает, где инструкция, а где ответ. | Всегда применяйте `apply_chat_template()` или ручной формат, соответствующий модели. |

---

### 1.9. Краткий глоссарий ключевых терминов (для быстрой справки)

- **BLEU, ROUGE** – метрики для оценки сходства с эталонным текстом (используются в переводе, суммаризации).
- **LLM‑as‑a‑judge** – оценка качества другой LLM (например, GPT‑4), применяется для открытых задач, где нет единственного правильного ответа.
- **Перплексия** – экспонента от кросс‑энтропии; чем ниже, тем «увереннее» модель. Снижение перплексии – хорошо, но не всегда означает качество генерации.
- **Катастрофическое забывание** – потеря старых знаний при адаптации к новой задаче.
- **Концептуальный дрейф** – изменение распределения пользовательских запросов со временем; служит триггером для повторного обучения.
- **LoRA** – низкоранговая адаптация: замораживаем $W_0$, обучаем $A$ и $B$, такие что $W = W_0 + AB$. Число обучаемых параметров сокращается в сотни раз.
- **QLoRA** – LoRA с 4‑битным квантованием базовой модели для ещё большей экономии памяти.
- **SFT** – обучение имитации эталонных ответов.
- **DPO** – обучение предпочтениям (без модели вознаграждения).
- **RAG** – генерация с подкреплением из внешней базы знаний.

---

### Итог Тема 1

- Мы построили систему координат: **тонкая настройка – это целенаправленное изменение весов модели** под вашу задачу, в отличие от промптов (без изменения весов) и RAG (с внешними знаниями).
- Определили **пять ключевых причин** для FT: доменная адаптация, структурирование вывода, подавление галлюцинаций, экономия ресурсов, безопасность.
- Чётко разграничили ситуации, когда FT **не нужна** – менее 500 примеров, частые обновления, творческие задачи и т.д.
- Построили **иерархию методов** (от Zero‑shot до Full FT) с оценкой затрат, качества и влияния на латентность.
- Ввели два класса FT: **SFT** (имитация) и **Preference Tuning** (выбор), и объяснили, когда что использовать.
- Привели **конкретные расчёты памяти** для моделей разного размера, чтобы вы могли оценить свои аппаратные возможности.
- Разобрали **типичные проблемы** (забывание, переобучение, галлюцинации) и способы их предотвращения.
- Дали **набор практических правил** выбора метода: начинайте с самого дешёвого и поднимайтесь только при доказанном приросте качества.

Теперь вы готовы перейти к следующей теме, где мы разберём **практическую реализацию LoRA и QLoRA**: установим библиотеки, подготовим датасет, запустим обучение и проанализируем результаты. Все расчёты, формулы и рекомендации, полученные здесь, пригодятся вам при конфигурации гиперпараметров.

# Тема 2. Виды тонкой настройки и их сравнение

В предыдущей теме мы заложили фундамент: поняли, что такое тонкая настройка, когда она нужна, а когда лучше обойтись промптами, и построили иерархию методов. Теперь настало время **детально разобрать каждый вид тонкой настройки**, сравнить их между собой и научиться выбирать правильный инструмент под конкретную задачу.

**Цель этой темы** – дать вам полную классификацию методов по объёму изменений, показать, как устроена полная настройка и инструкционная настройка, сравнить все подходы в одной таблице, а также разграничить SFT и Preference Tuning. Мы также обсудим мультизадачную и мультиязычную настройку – важные техники, которые помогают бороться с катастрофическим забыванием и улучшают обобщение.

---

## 2.1. Классификация по объёму изменяемых параметров

Все методы взаимодействия с LLM можно разделить на три большие группы в зависимости от того, **изменяются ли веса модели** и **в каком объёме**:

| Группа | Суть | Примеры |
| :--- | :--- | :--- |
| **Без изменений весов (Weight‑free)** | Модель остаётся фиксированной, мы влияем на выход через промпты или внешние данные. | Zero‑shot, Few‑shot, Chain‑of‑Thought, RAG |
| **PEFT (Parameter‑Efficient Fine‑Tuning)** | Основная модель заморожена, обучается только малая доля дополнительных параметров (адаптеры). | LoRA, QLoRA, Adapters, Prefix Tuning |
| **Full Fine‑Tuning** | Обновляются **все** параметры модели. | Полное дообучение на целевом датасете |

Эта классификация определяет не только техническую реализацию, но и **ресурсные затраты**, **риск катастрофического забывания** и **гибкость**.

**Вывод:** выбор группы – это компромисс между качеством адаптации и стоимостью. Чем больше параметров мы изменяем, тем выше потенциал качества, но тем дороже обучение и выше риск испортить общие знания модели.

---

## 2.2. Полная тонкая настройка (Full Fine‑Tuning)

**Full Fine‑Tuning** – это процесс, при котором мы обновляем **все веса** предобученной модели, используя градиентный спуск на нашем целевом датасете. Модель полностью перестраивает свои внутренние представления под новую задачу.

### Когда Full FT оправдан?
- **Огромный размеченный датасет** – более 100 000 качественных примеров (чем больше, тем лучше, чтобы не переобучиться).
- **Фундаментальная смена домена** – например, мы добавляем новый язык, которого модель почти не знала, или переводим модель с английского на китайский.
- **Максимальное качество критичнее стоимости** – в исследовательских проектах или когда бизнес готов платить за каждый процент точности.
- **Доступны мощные кластеры** – несколько GPU с 80 ГБ памяти и более.

### Расчёт требований к памяти (7B модель, bf16, оптимизатор Adam)

Для полной тонкой настройки с оптимизатором Adam мы должны хранить:
1. **Веса модели** – $7 \times 10^9$ параметров × 2 байта (bf16) = **14 ГБ**.
2. **Градиенты** – такого же размера, ещё **14 ГБ**.
3. **Состояния оптимизатора Adam** – для каждого параметра хранятся два момента (первый и второй порядок). Это ещё $2 \times 14$ ГБ = **28 ГБ**.
4. **Итого:** $14 + 14 + 28 = 56$ ГБ, плюс накладные расходы на активации, временные буферы и т.д. – обычно около **80–90 ГБ** на практике.

Если мы используем смешанную точность (fp16/bf16 для весов и градиентов, а fp32 для обновлений), то Adam всё равно требует fp32 для моментов, поэтому расчёт выше корректен.

**Для 13B модели** (13 млрд параметров):
- Веса: $13 \times 2 = 26$ ГБ
- Градиенты: 26 ГБ
- Adam: $2 \times 26 = 52$ ГБ
- Итого: $26+26+52 = 104$ ГБ → на практике > 150 ГБ.

**Для 70B модели** – уже сотни ГБ, требуются распределённые стратегии (DeepSpeed ZeRO, FSDP).

### Стоимость обучения (время)
- Для 7B модели на одном A100 (80 ГБ) полная тонкая настройка на 100 000 примеров (эпоха) может занять **10–20 часов**.
- Для 70B модели – **несколько дней** даже на кластере из 8 A100.

> **Вывод:** Full FT – это выбор крупных организаций с большим бюджетом и инженерной экспертизой. Для стартапов и отдельных исследователей чаще достаточно PEFT.

---

## 2.3. Инструкционная настройка (Instruction Tuning)

**Instruction Tuning** – это особый вид тонкой настройки (обычно Full или PEFT), при котором модель обучается отвечать на инструкции в формате «вопрос – ответ» (или «запрос – ответ»). Цель – научить модель **следовать инструкциям** так же хорошо, как это делают проприетарные модели (ChatGPT, Claude).

### Структура данных (формат JSON)

Обучающий датасет для Instruction Tuning обычно содержит три поля:

```json
{
  "instruction": "Напиши краткое изложение текста",
  "input": "Текст для суммаризации... (может быть пустым)",
  "output": "Краткое изложение текста"
}
```

В некоторых датасетах используется формат ShareGPT (диалоги с несколькими оборотами):

```json
{
  "conversations": [
    {"from": "human", "value": "Что такое градиентный спуск?"},
    {"from": "gpt", "value": "Градиентный спуск — это метод оптимизации..."},
    {"from": "human", "value": "А какие есть варианты?"},
    {"from": "gpt", "value": "Существуют стохастический, пакетный и мини-пакетный..."}
  ]
}
```

### Почему важен чат-шаблон (ChatML)?

Разные модели ожидают разные форматы разметки диалогов. Например, Llama 3 использует специальные токены:

```
<|begin_of_text|><|start_header_id|>user<|end_header_id|>

Вопрос...<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Ответ...<|eot_id|>
```

ChatML – это стандартизированный шаблон, который используется в моделях семейства GPT, но каждая модель имеет свой собственный. Применение `tokenizer.apply_chat_template()` **гарантирует**, что токенизатор правильно расставит специальные токены. Если этого не сделать, модель будет путать роли и ответы станут бессмысленными.

### Мультизадачная настройка

Это подход, при котором мы смешиваем датасеты из **разных задач** (суммаризация, перевод, QA, классификация) в одном обучении. Это улучшает обобщение и делает модель более универсальной. Мультизадачность помогает бороться с катастрофическим забыванием, потому что модель видит разнообразие примеров и не переобучается под одну узкую задачу.

---

## 2.4. Сравнительная таблица методов

Сведём в одну таблицу ключевые характеристики всех основных подходов (Промпты, RAG, LoRA, Full FT). В таблице добавим столбец «Влияние на стиль» (насколько метод меняет манеру ответа), «Влияние на факты» (способность усваивать новые факты) и «Влияние на Latency» (задержку инференса).

| Метод | Скорость внедрения | Стоимость (обучение/инференс) | Влияние на стиль | Влияние на факты | Влияние на Latency |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **Промпты** (Zero/Few‑shot) | Минуты–часы | $0 / 1× (базовый) | Среднее (зависит от примера) | Нет (использует внутренние знания) | Нет (базовая задержка) |
| **RAG** | Дни (настройка индекса) | $ (эмбеддинги) / 5–10× (из-за поиска) | Нет (модель не меняется) | **Высокое** (добавляет актуальные документы) | **Высокое** (запрос к БД + длинный контекст) |
| **LoRA / QLoRA** | Дни (подготовка данных, обучение) | $$ (5–50$ за запуск) / 1× (после слияния) | **Высокое** (меняет стиль, тон) | Среднее (усваивает паттерны из данных) | **Почти нет** (слитый адаптер не добавляет задержки) |
| **Full Fine‑Tuning** | Недели | $$$$ (100–10000$ за запуск) / 1× | **Очень высокое** (полная перестройка) | **Высокое** (запоминает факты) | Может расти на 10–30% (изменение распределения весов) |

**Комментарии по таблице:**
- **Скорость внедрения** – время от идеи до первого работающего прототипа.
- **Стоимость обучения** – указана относительная оценка; для RAG – это стоимость эмбеддингов и хостинга БД.
- **Влияние на Latency** – для RAG задержка растёт из-за поиска и передачи длинного контекста; для Full FT возможен небольшой рост из-за изменения весов (но обычно незаметно).
- **LoRA** после слияния адаптера с базовой моделью не увеличивает latency, поэтому мы ставим «почти нет».

---

## 2.5. SFT против Preference Tuning (DPO/RLHF)

Это два принципиально разных подхода к тонкой настройке, и их нельзя путать.

| Характеристика | **SFT (Supervised Fine‑Tuning)** | **Preference Tuning (DPO / RLHF)** |
| :--- | :--- | :--- |
| **Что делаем** | Учим модель **имитировать** эталонные ответы из датасета. | Учим модель **выбирать** предпочтительный ответ из пары (или ранжированного списка). |
| **Тип данных** | Пары (вход, правильный выход). | Тройки (вход, хороший ответ, плохой ответ) или ранжирование. |
| **Функция потерь** | Кросс-энтропия между предсказанием и эталоном. | Логарифм отношения вероятностей предпочтения (для DPO) или обучение модели вознаграждения (RLHF). |
| **Когда применять** | Задача имеет чёткий правильный ответ (перевод, суммаризация, извлечение структуры). | Важна стилистика, креативность, безопасность, субъективная оценка (диалоговые агенты, персонализация). |
| **Результат** | Модель становится точной и предсказуемой. | Модель становится «приятной», безопасной, более живой. |
| **Сложность реализации** | Относительно простая (стандартный Trainer). | Сложнее (требуется генерация пар, настройка DPO-лосса). |

### Почему в нашем курсе фокус на SFT?

SFT – это **базовый строительный блок**. Без SFT модель не научится даже следовать формату, а DPO будет бесполезен, потому что модель не понимает, что от неё хотят. Поэтому мы сначала осваиваем SFT как основу, а затем, если потребуется, переходим к DPO для полировки. Кроме того, SFT требует меньше инженерных усилий и более предсказуем.

---

## 2.6. Мультизадачная и мультиязычная настройка

**Мультизадачная настройка** – это обучение модели одновременно на нескольких задачах (например, классификация, суммаризация, генерация кода). Это улучшает способность модели к обобщению и делает её более робастной.

**Мультиязычная настройка** – добавление в датасет примеров на разных языках, чтобы модель сохраняла или улучшала свои мультиязычные способности.

### Стратегии смешивания данных

1. **Пропорциональное смешивание** – каждый датасет входит в общий пул с весом, пропорциональным его размеру.
2. **Балансировка по длине** – уравниваем количество токенов от каждой задачи, чтобы короткие задачи не доминировали.
3. **Важность задач (task‑weighting)** – назначаем разные веса задачам в зависимости от их сложности или приоритета.
4. **Curriculum learning** – начинаем с лёгких задач, постепенно добавляя сложные.

### Как это помогает бороться с катастрофическим забыванием?

Когда модель обучается на одной узкой задаче, она «забывает» общие знания. Если же мы смешиваем много задач, модель постоянно повторяет разнообразные паттерны, и вероятность потери старых знаний снижается. Это аналогично тому, как студент, изучающий несколько предметов одновременно, лучше сохраняет знания, чем если бы он зубрил один предмет.

---

## Выводы по теме 2

1. **Классификация по объёму изменений** даёт нам три категории: без изменений (промпты, RAG), PEFT (LoRA) и Full FT. Чем выше уровень, тем дороже и мощнее.
2. **Full FT** оправдан только при наличии огромного датасета и ресурсов; расчёт памяти для 7B модели с Adam – около 80–90 ГБ.
3. **Instruction Tuning** – обучение модели отвечать на инструкции в структурированном формате; критически важно использовать правильный чат-шаблон.
4. **Сравнительная таблица** наглядно показывает, что LoRA занимает золотую середину: высокое влияние на стиль, низкая стоимость и практически нулевое влияние на задержку.
5. **SFT vs DPO** – это две разные философии: SFT для имитации, DPO для выбора предпочтений. Мы начинаем с SFT как основы.
6. **Мультизадачность** и **мультиязычность** улучшают обобщение и уменьшают забывание; правильное смешивание данных – ключ к успеху.

Теперь, когда у нас есть полная картина методов, мы готовы перейти к практике – к **реализации LoRA и QLoRA**, где все эти теоретические знания превратятся в работающий код.

# Лекция 4.1. Введение в тонкую настройку LLM: от промптов к обучению

## Тема 3. Zero‑shot и Few‑shot обучение: быстрый старт без изменения весов

Прежде чем вкладывать дни и деньги в тонкую настройку, стоит задать себе простой вопрос: *«А может, модель уже умеет это делать, если правильно попросить?»* Современные большие языковые модели обладают удивительной способностью адаптироваться к новым задачам «на лету» – прямо во время инференса, без единого изменения весов. Эти техники, известные как **Zero‑shot** и **Few‑shot** (или In‑Context Learning), позволяют за минуты получить результат, который часто оказывается «достаточно хорошим» для прототипа или даже для продакшена.

**Цель этой темы** – научить вас максимально эффективно использовать промпты, чтобы отодвинуть момент, когда вам действительно понадобится тонкая настройка. Мы разберём:

- **Zero‑shot** – как заставить модель решать задачу без примеров, только с инструкцией.
- **Few‑shot** – как добавление 2–5 примеров в контекст кардинально меняет качество.
- **Продвинутые техники** (Chain‑of‑Thought, Self‑Consistency, Tree‑of‑Thought) – они поднимают качество до уровня, сопоставимого с тонкой настройкой, но без её затрат.
- **Практическое правило** – когда можно остановиться на промптах и не идти дальше.

Все примеры промптов мы дадим в формате, совместимом с большинством чат‑моделей (Llama, Qwen, GPT). Главный принцип: **сначала пробуем промпты, и только если они не дотягивают до целевых метрик, рассматриваем RAG или тонкую настройку.**

---

### 3.0. Почему промпты – это первая линия обороны

Инженерия промптов – это самый быстрый и дешёвый способ взаимодействия с LLM. Вы не тратите время на сбор датасетов, не платите за обучение, не рискуете переобучить модель. Вы просто пишете текст – и получаете ответ.

Но у промптов есть и ограничения:
- Они не могут добавить модели новые факты, которых нет в её весах.
- Они не могут кардинально изменить «личность» модели – она остаётся той же.
- Они занимают место в контексте, что увеличивает стоимость и задержку (особенно при Few‑shot с длинными примерами).

Тем не менее, для 80% задач хорошо составленный промпт даёт качество, достаточное для принятия решений. Иногда даже для продакшена – если вы используете мощную модель (GPT‑4, Claude) и задача не требует специфических знаний.

---


### 3.1. Zero‑shot: доверяем внутренним знаниям модели

**Zero‑shot** – это подход, при котором мы даём модели только инструкцию, без каких‑либо примеров выполнения задачи. Модель полагается исключительно на те знания и паттерны, которые она усвоила во время предобучения.

Это самый быстрый и дешёвый способ взаимодействия с LLM: вы тратите минуты на написание промпта и получаете ответ практически мгновенно. Однако качество ограничено тем, что модель уже знает – если задача требует узкоспециализированных знаний или строгого формата, Zero‑shot может не справиться.

#### Пример: математическая задача (без примеров)

Возьмём простой запрос из нашего датасета:

```
Сколько будет 2 + 2? Ответ дай числом.
```

**Разбор промпта по частям:**

1. **Формулировка задачи** – чётко указываем, что нужно вычислить:
   ```
   Сколько будет 2 + 2?
   ```

2. **Ограничение формата вывода** – просим ответить числом, чтобы упростить автоматическую проверку:
   ```
   Ответ дай числом.
   ```

   Это не строгое ограничение (модель может ответить «4» или «2 + 2 = 4»), но в коде у нас есть функция `extract_number`, которая умеет извлекать число из ответа, даже если модель написала лишний текст.

**Полный промпт целиком (готовый к копированию):**

```
Сколько будет 2 + 2? Ответ дай числом.
```

Этот промпт мы передаём модели, и она генерирует ответ, используя свои внутренние знания арифметики. Никаких примеров вычислений мы не даём – это чистый Zero‑shot.

#### Практическая реализация Zero‑shot (полный код)

Ниже приведён полный Python‑скрипт, который:
1. Загружает модель **Phi‑3 Mini 4K Instruct**.
2. Прогоняет датасет из 10 примеров (5 переводов с русского на английский и 5 математических задач).
3. Для каждого примера отправляет Zero‑shot промпт (без примеров).
4. Измеряет качество (BLEU, ROUGE для переводов, точность для математики) и производительность (токены/сек).

**Этот код можно скопировать и запустить в Google Colab или локально с GPU/CPU.**

```python
# ================================================================
# Phi-3 Mini 4K Instruct — Оценка на датасете (качество + скорость)
# ================================================================

!pip install evaluate sacrebleu rouge-score

# ----------------------------------------------------------------
# 1. Импорты
# ----------------------------------------------------------------
import os
import gc
import re
import time
import torch
import warnings
from typing import List, Dict, Any

from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForCausalLM,
    pipeline,
    GenerationConfig,
)

# Для метрик качества (попробуем импортировать, если есть)
try:
    import evaluate
    from evaluate import load
    HAS_EVALUATE = True
except ImportError:
    HAS_EVALUATE = False
    print("⚠️ Библиотека 'evaluate' не установлена. Метрики BLEU/ROUGE будут недоступны.")
    print("   Установите: pip install evaluate sacrebleu rouge-score")

try:
    from rouge_score import rouge_scorer
    HAS_ROUGE = True
except ImportError:
    HAS_ROUGE = False
    print("⚠️ 'rouge-score' не установлена. ROUGE будет пропущен.")
    print("   Установите: pip install rouge-score")

# Подавляем предупреждение о GenerationMixin
warnings.filterwarnings("ignore", message=".*Phi3ForCausalLM has generative capabilities.*")

# ----------------------------------------------------------------
# 2. Основные настройки
# ----------------------------------------------------------------
MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"
MAX_NEW_TOKENS = 256
TEMPERATURE = 0.2
TOP_P = 0.9
SEED = 42

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ----------------------------------------------------------------
# 3. Проверка окружения
# ----------------------------------------------------------------
print("=" * 70)
print("ПРОВЕРКА ОКРУЖЕНИЯ")
print("=" * 70)

print(f"PyTorch:       {torch.__version__}")
print(f"CUDA доступна: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU:           {gpu_name}")
    print(f"VRAM:          {gpu_memory:.2f} GB")
    DTYPE = torch.float16
    print(f"Dtype:         {DTYPE}")
    torch.cuda.reset_peak_memory_stats()
else:
    print("⚠️ GPU не обнаружен. Модель будет загружена на CPU.")
    DTYPE = torch.float32

print("=" * 70)

# ----------------------------------------------------------------
# 4. Освобождение памяти
# ----------------------------------------------------------------
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# ----------------------------------------------------------------
# 5. Загрузка токенизатора и модели
# ----------------------------------------------------------------
print("\n🔹 Загружаем токенизатор...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("✅ Токенизатор загружен")
print(f"Vocabulary size: {len(tokenizer):,}")
print(f"EOS token:       {tokenizer.eos_token!r}")
print(f"PAD token:       {tokenizer.pad_token!r}")

print("\n🔹 Загружаем конфигурацию модели...")
config = AutoConfig.from_pretrained(MODEL_NAME)
print("✅ Конфигурация загружена")

print("\n🔹 Загружаем модель:")
print(f"   {MODEL_NAME}")

model_kwargs = {
    "config": config,
    "torch_dtype": DTYPE,
}
if torch.cuda.is_available():
    model_kwargs["device_map"] = "auto"
else:
    model_kwargs["device_map"] = "cpu"

start_load_time = time.time()
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **model_kwargs)
load_time = time.time() - start_load_time

model.eval()
print(f"✅ Модель загружена за {load_time:.2f} сек.")

# ----------------------------------------------------------------
# 6. Информация о модели
# ----------------------------------------------------------------
num_parameters = sum(p.numel() for p in model.parameters())
trainable_parameters = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("\n" + "=" * 70)
print("ИНФОРМАЦИЯ О МОДЕЛИ")
print("=" * 70)
print(f"Всего параметров:      {num_parameters / 1e9:.3f} B")
print(f"Обучаемых параметров:  {trainable_parameters / 1e9:.3f} B")
print(f"Device map: {getattr(model, 'hf_device_map', 'N/A')}")
if torch.cuda.is_available():
    current_vram = torch.cuda.memory_allocated() / 1024**3
    peak_vram = torch.cuda.max_memory_allocated() / 1024**3
    print(f"Занято VRAM:          {current_vram:.2f} GB")
    print(f"Пиковое VRAM:         {peak_vram:.2f} GB")
print("=" * 70)

# ----------------------------------------------------------------
# 7. Pipeline
# ----------------------------------------------------------------
generator = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
)
print("\n✅ Pipeline успешно создан")

# ----------------------------------------------------------------
# 8. Функция генерации
# ----------------------------------------------------------------
def generate_text(
    prompt: str,
    max_new_tokens: int = MAX_NEW_TOKENS,
    temperature: float = TEMPERATURE,
    top_p: float = TOP_P,
    do_sample: bool = True,
) -> Dict[str, Any]:
    """Генерирует ответ и возвращает текст + метрики производительности."""
    if not isinstance(prompt, str) or not prompt.strip():
        raise ValueError("prompt должен быть непустой строкой")

    messages = [{"role": "user", "content": prompt}]
    if hasattr(tokenizer, "apply_chat_template"):
        formatted_prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    else:
        formatted_prompt = prompt

    gen_config = GenerationConfig(
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        temperature=temperature if do_sample else 1.0,
        top_p=top_p if do_sample else 1.0,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    start_time = time.time()
    with torch.inference_mode():
        result = generator(
            formatted_prompt,
            return_full_text=False,
            generation_config=gen_config,
            clean_up_tokenization_spaces=False,
        )
    end_time = time.time()

    generated_text = result[0]["generated_text"].strip() if result else ""
    generation_time = end_time - start_time

    input_tokens = len(tokenizer.encode(formatted_prompt))
    output_tokens = len(tokenizer.encode(generated_text))
    tokens_per_second = output_tokens / generation_time if generation_time > 0 else 0.0

    return {
        "text": generated_text,
        "metrics": {
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "generation_time_sec": round(generation_time, 3),
            "tokens_per_second": round(tokens_per_second, 2),
        }
    }

# ----------------------------------------------------------------
# 9. УЛУЧШЕННАЯ ФУНКЦИЯ ИЗВЛЕЧЕНИЯ ЧИСЛА
# ----------------------------------------------------------------
def extract_number(text: str) -> str:
    """
    Извлекает число из текста, отдавая предпочтение числу после знака '=' или 'равно'.
    Если таких нет, возвращает последнее число в тексте.
    """
    text = text.replace(",", "")
    # Ищем число после '='
    if "=" in text:
        parts = text.split("=")
        if len(parts) > 1:
            after_eq = parts[-1]
            nums = re.findall(r"[-+]?\d*\.?\d+", after_eq)
            if nums:
                return nums[0]
    # Ищем после 'равно'
    if "равно" in text:
        parts = text.split("равно")
        if len(parts) > 1:
            after_eq = parts[-1]
            nums = re.findall(r"[-+]?\d*\.?\d+", after_eq)
            if nums:
                return nums[0]
    # Иначе все числа
    numbers = re.findall(r"[-+]?\d*\.?\d+", text)
    return numbers[-1] if numbers else ""

# ----------------------------------------------------------------
# 10. ДАТАСЕТ (расширенный)
# ----------------------------------------------------------------
dataset = [
    # Переводы (русский → английский)
    {"type": "translation", "input": "Переведи на английский: Привет, как дела?", "target": "Hello, how are you?"},
    {"type": "translation", "input": "Переведи на английский: Сегодня отличная погода.", "target": "Today is great weather."},
    {"type": "translation", "input": "Переведи на английский: Я люблю программирование.", "target": "I love programming."},
    {"type": "translation", "input": "Переведи на английский: Который час?", "target": "What time is it?"},
    {"type": "translation", "input": "Переведи на английский: Это очень интересно.", "target": "This is very interesting."},
    # Математика
    {"type": "math", "input": "Сколько будет 2 + 2? Ответ дай числом.", "target": "4"},
    {"type": "math", "input": "Сколько будет 5 * 3? Ответ дай числом.", "target": "15"},
    {"type": "math", "input": "Сколько будет 10 - 4? Ответ дай числом.", "target": "6"},
    {"type": "math", "input": "Сколько будет 12 / 3? Ответ дай числом.", "target": "4"},
    {"type": "math", "input": "Сколько будет 7 + 8? Ответ дай числом.", "target": "15"},
]

print("\n" + "=" * 70)
print("ДАТАСЕТ ДЛЯ ОЦЕНКИ")
print("=" * 70)
for i, ex in enumerate(dataset, 1):
    print(f"{i}. [{ex['type']}] {ex['input']} -> {ex['target']}")
print("=" * 70)

# ----------------------------------------------------------------
# 11. ЗАГРУЗКА МЕТРИК (если доступны)
# ----------------------------------------------------------------
bleu_metric = None
rouge_scorer_obj = None

if HAS_EVALUATE:
    try:
        bleu_metric = load("bleu")
    except Exception as e:
        print(f"⚠️ Не удалось загрузить BLEU: {e}")

if HAS_ROUGE:
    rouge_scorer_obj = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

# ----------------------------------------------------------------
# 12. ФУНКЦИЯ ОЦЕНКИ
# ----------------------------------------------------------------
def evaluate_dataset(dataset: List[Dict]) -> Dict:
    """
    Прогоняет все примеры, собирает предсказания,
    вычисляет метрики качества и производительности.
    """
    predictions = []
    references = []
    example_types = []
    perf_metrics = []

    # Для накопления метрик качества
    bleu_scores = []
    rouge_scores = []
    math_correct = 0
    math_total = 0

    print("\n" + "=" * 70)
    print("ЗАПУСК ОЦЕНКИ НА ДАТАСЕТЕ")
    print("=" * 70)

    for idx, example in enumerate(dataset, 1):
        print(f"\n--- Пример {idx} ({example['type']}) ---")
        print(f"Вопрос: {example['input']}")
        print(f"Эталон: {example['target']}")

        # Генерация
        result = generate_text(example["input"], max_new_tokens=MAX_NEW_TOKENS)
        pred_text = result["text"]
        metrics = result["metrics"]

        print(f"Ответ модели: {pred_text}")

        # Сохраняем для метрик
        predictions.append(pred_text)
        references.append([example["target"]])   # для BLEU
        example_types.append(example["type"])
        perf_metrics.append(metrics)

        # Вывод скорости
        print(f"   ⏱ Время: {metrics['generation_time_sec']} сек, "
              f"токенов: {metrics['output_tokens']}, TPS: {metrics['tokens_per_second']}")

        # ----- Сбор метрик качества по типам -----
        if example["type"] == "translation":
            # BLEU (накапливаем для среднего)
            if bleu_metric is not None:
                try:
                    score = bleu_metric.compute(predictions=[pred_text], references=[[example["target"]]])
                    bleu_scores.append(score["bleu"])
                except Exception as e:
                    print(f"   BLEU ошибка: {e}")

            # ROUGE-L
            if rouge_scorer_obj is not None:
                try:
                    scores = rouge_scorer_obj.score(example["target"], pred_text)
                    rouge_scores.append(scores['rougeL'].fmeasure)
                except Exception as e:
                    print(f"   ROUGE ошибка: {e}")

        elif example["type"] == "math":
            math_total += 1
            pred_num = extract_number(pred_text)
            ref_num = extract_number(example["target"])
            if pred_num and ref_num and pred_num == ref_num:
                math_correct += 1
                print("   ✅ Математика: верно")
            else:
                print(f"   ❌ Математика: неверно (извлечено '{pred_num}', ожидалось '{ref_num}')")

    # ----- ИТОГОВЫЕ МЕТРИКИ КАЧЕСТВА -----
    print("\n" + "=" * 70)
    print("РЕЗУЛЬТАТЫ ОЦЕНКИ КАЧЕСТВА")
    print("=" * 70)

    # Средний BLEU по переводам
    if bleu_scores:
        avg_bleu = sum(bleu_scores) / len(bleu_scores)
        print(f"📊 Средний BLEU (переводы): {avg_bleu:.3f}")
    else:
        print("📊 BLEU: не доступен (установите evaluate и sacrebleu)")

    # Средний ROUGE-L
    if rouge_scores:
        avg_rouge = sum(rouge_scores) / len(rouge_scores)
        print(f"📊 Средний ROUGE-L (переводы): {avg_rouge:.3f}")
    else:
        print("📊 ROUGE-L: не доступен (установите rouge-score)")

    # Точность на математике
    if math_total > 0:
        accuracy = math_correct / math_total
        print(f"📊 Точность на математике: {accuracy:.2%} ({math_correct}/{math_total})")
    else:
        print("📊 Математических задач нет.")

    # ----- СРЕДНИЕ МЕТРИКИ ПРОИЗВОДИТЕЛЬНОСТИ -----
    print("\n" + "=" * 70)
    print("СРЕДНИЕ МЕТРИКИ ПРОИЗВОДИТЕЛЬНОСТИ")
    print("=" * 70)
    if perf_metrics:
        avg_input = sum(m["input_tokens"] for m in perf_metrics) / len(perf_metrics)
        avg_output = sum(m["output_tokens"] for m in perf_metrics) / len(perf_metrics)
        avg_time = sum(m["generation_time_sec"] for m in perf_metrics) / len(perf_metrics)
        avg_tps = sum(m["tokens_per_second"] for m in perf_metrics) / len(perf_metrics)
        print(f"Среднее токенов в запросе:      {avg_input:.1f}")
        print(f"Среднее сгенерировано токенов:  {avg_output:.1f}")
        print(f"Среднее время генерации:        {avg_time:.3f} сек.")
        print(f"Средняя скорость (TPS):         {avg_tps:.2f} токен/сек.")

    return {
        "predictions": predictions,
        "references": references,
        "perf_metrics": perf_metrics,
    }

# ----------------------------------------------------------------
# 13. ЗАПУСК
# ----------------------------------------------------------------
if __name__ == "__main__":
    results = evaluate_dataset(dataset)
    print("\n" + "=" * 70)
    print("✅ ОЦЕНКА ЗАВЕРШЕНА")
    print("=" * 70)
```

#### Результаты выполнения (пример)

При запуске на **Tesla T4 (16 GB)** код выдал следующие результаты (сокращённый вывод):

```
======================================================================
ПРОВЕРКА ОКРУЖЕНИЯ
======================================================================
PyTorch:       2.11.0+cu128
CUDA доступна: True
GPU:           Tesla T4
VRAM:          14.56 GB
Dtype:         torch.float16
======================================================================
...
======================================================================
ДАТАСЕТ ДЛЯ ОЦЕНКИ
======================================================================
1. [translation] Переведи на английский: Привет, как дела? -> Hello, how are you?
2. [translation] Переведи на английский: Сегодня отличная погода. -> Today is great weather.
3. [translation] Переведи на английский: Я люблю программирование. -> I love programming.
4. [translation] Переведи на английский: Который час? -> What time is it?
5. [translation] Переведи на английский: Это очень интересно. -> This is very interesting.
6. [math] Сколько будет 2 + 2? Ответ дай числом. -> 4
7. [math] Сколько будет 5 * 3? Ответ дай числом. -> 15
8. [math] Сколько будет 10 - 4? Ответ дай числом. -> 6
9. [math] Сколько будет 12 / 3? Ответ дай числом. -> 4
10. [math] Сколько будет 7 + 8? Ответ дай числом. -> 15
======================================================================
...
--- Пример 6 (math) ---
Вопрос: Сколько будет 2 + 2? Ответ дай числом.
Эталон: 4
Ответ модели: 2 + 2 равно 4.
   ⏱ Время: 3.294 сек, токенов: 11, TPS: 3.34
   ✅ Математика: верно
...
======================================================================
РЕЗУЛЬТАТЫ ОЦЕНКИ КАЧЕСТВА
======================================================================
📊 Средний BLEU (переводы): 0.800
📊 Средний ROUGE-L (переводы): 0.933
📊 Точность на математике: 100.00% (5/5)

======================================================================
СРЕДНИЕ МЕТРИКИ ПРОИЗВОДИТЕЛЬНОСТИ
======================================================================
Среднее токенов в запросе:      19.7
Среднее сгенерировано токенов:  8.0
Среднее время генерации:        2.544 сек.
Средняя скорость (TPS):         3.08 токен/сек.
======================================================================
```

**Анализ результатов:**
- Модель справилась со всеми математическими задачами (100% точность) и дала высокие баллы за переводы (BLEU 0.8, ROUGE 0.933).
- Это демонстрирует, что **Zero‑shot может быть очень эффективным** для задач, которые модель хорошо знает из предобучения.
- Производительность на T4 – около 3 токенов в секунду, что приемлемо для прототипирования.




### 3.2. Few‑shot (In‑Context Learning): учимся на примерах

**Few‑shot** – это техника, при которой мы вставляем в промпт 2–5 (иногда до 10) эталонных пар «вопрос – правильный ответ», а затем задаём новый вопрос. Модель не меняет веса, но **использует примеры как контекст**, чтобы понять желаемый формат, стиль и логику ответа.

**Механизм:** Во время предобучения модель видела бесчисленное количество последовательностей, где после нескольких примеров шёл новый запрос. Она научилась извлекать закономерности из этих примеров «на лету». Это называется **мета‑обучением** или **In‑Context Learning**. Модель не запоминает примеры, но использует их как временный шаблон.

**Главный недостаток:** рост числа токенов в промпте. Каждый пример занимает место в контекстном окне, и стоимость инференса (в токенах) растёт линейно. Если модель берёт плату за токены, Few‑shot может оказаться дороже Zero‑shot в 2–4 раза. Кроме того, увеличивается время генерации (GPU обрабатывает больше токенов). Для длинных примеров (например, суммаризация больших текстов) Few‑shot может стать непрактичным.

#### Пример Few‑shot для классификации настроений (3 примера)

Вот классический текстовый пример, который часто используют в учебных целях:

```
Ты – классификатор настроений. Вот примеры:

Отзыв: "Фильм просто бомба! Актеры играют великолепно, сюжет затягивает."
Тональность: позитивный

Отзыв: "Ужасный сервис. Ждал заказ три недели, а прислали брак."
Тональность: негативный

Отзыв: "Обычный отель, ничего особенного. Завтрак средний, но кровать удобная."
Тональность: нейтральный

Теперь определи тональность для этого отзыва:
"Телефон хороший, но батарея быстро садится, а камера мылит."
Тональность:
```

Модель, увидев три примера, скорее всего, выдаст «негативный», потому что в примерах она усвоила, что если есть похвала, но есть и критика, то это часто относится к негативным или нейтральным. Она также усвоит формат вывода.

---

#### Практическая реализация Few‑shot (полный код)

Для демонстрации мы адаптировали код под задачи перевода и математики. Вместо классификации мы показываем, как Few‑shot помогает модели выдавать **более точные и краткие** ответы, а также **ускоряет инференс** за счёт жёсткого форматирования (модель «знает», что от неё требуется).

Ниже приведён полный Python‑скрипт, который:
1. Загружает модель **Phi‑3 Mini 4K Instruct**.
2. Строит диалог с **системным сообщением** (краткость, не задавать вопросов) и **двумя демонстрациями** для переводов и для математики.
3. Прогоняет тестовый датасет из 6 примеров (3 перевода, 3 задачи на арифметику).
4. Измеряет качество (ROUGE, точность) и производительность.

**Что важно в коде:**
- Мы используем **чат‑шаблон** `apply_chat_template` – он автоматически расставляет специальные токены, что критически важно для интерактивных моделей.
- Добавлено **системное сообщение** – оно улучшает следование инструкциям.
- Функция `clean_response` удаляет лишние маркеры, если модель начинает генерировать новый вопрос.

```python
# ================================================================
# Phi-3 Mini 4K Instruct — Few-shot (2 примера) с системной инструкцией
# ================================================================

!pip install evaluate sacrebleu rouge-score -q

import os, gc, re, time, torch, warnings
from typing import List, Dict, Any, Tuple
from transformers import (
    AutoTokenizer, AutoConfig, AutoModelForCausalLM,
    pipeline, GenerationConfig
)

# Подавляем предупреждения о clean_up_tokenization_spaces
warnings.filterwarnings("ignore", message=".*clean_up_tokenization_spaces.*")
warnings.filterwarnings("ignore", message=".*Phi3ForCausalLM has generative capabilities.*")

# Импорт метрик
try:
    import evaluate
    from evaluate import load
    HAS_EVALUATE = True
except ImportError:
    HAS_EVALUATE = False
    print("⚠️ Установите: pip install evaluate sacrebleu")

try:
    from rouge_score import rouge_scorer
    HAS_ROUGE = True
except ImportError:
    HAS_ROUGE = False
    print("⚠️ Установите: pip install rouge-score")

# ----------------------------------------------------------------
# Настройки
# ----------------------------------------------------------------
MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"
MAX_NEW_TOKENS = 64            # достаточно для коротких ответов
TEMPERATURE = 0.2
TOP_P = 0.9
SEED = 42
NUM_DEMOS = 2                  # количество демонстраций для каждого типа

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ----------------------------------------------------------------
# Окружение
# ----------------------------------------------------------------
print("=" * 70)
print("ПРОВЕРКА ОКРУЖЕНИЯ")
print("=" * 70)
print(f"PyTorch: {torch.__version__}")
print(f"CUDA доступна: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
    DTYPE = torch.float16
    torch.cuda.reset_peak_memory_stats()
else:
    DTYPE = torch.float32
print("=" * 70)

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# ----------------------------------------------------------------
# Загрузка модели
# ----------------------------------------------------------------
print("\n🔹 Загружаем токенизатор...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print("✅ Токенизатор загружен")

print("\n🔹 Загружаем модель...")
model_kwargs = {
    "config": AutoConfig.from_pretrained(MODEL_NAME),
    "torch_dtype": DTYPE,
}
if torch.cuda.is_available():
    model_kwargs["device_map"] = "auto"

start = time.time()
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **model_kwargs)
load_time = time.time() - start
model.eval()
print(f"✅ Модель загружена за {load_time:.2f} сек.")

# Информация о модели
print("\n" + "=" * 70)
print("ИНФОРМАЦИЯ О МОДЕЛИ")
print("=" * 70)
print(f"Всего параметров: {sum(p.numel() for p in model.parameters()) / 1e9:.3f} B")
if torch.cuda.is_available():
    print(f"Занято VRAM: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print("=" * 70)

# Pipeline
generator = pipeline("text-generation", model=model, tokenizer=tokenizer)
print("\n✅ Pipeline создан")

# ----------------------------------------------------------------
# Вспомогательные функции
# ----------------------------------------------------------------
def extract_number(text: str) -> str:
    """Извлекает последнее число или число после '=' / 'равно'."""
    text = text.replace(",", "")
    if "=" in text:
        parts = text.split("=")
        if len(parts) > 1:
            nums = re.findall(r"[-+]?\d*\.?\d+", parts[-1])
            if nums:
                return nums[0]
    if "равно" in text:
        parts = text.split("равно")
        if len(parts) > 1:
            nums = re.findall(r"[-+]?\d*\.?\d+", parts[-1])
            if nums:
                return nums[0]
    nums = re.findall(r"[-+]?\d*\.?\d+", text)
    return nums[-1] if nums else ""

def clean_response(text: str) -> str:
    """Обрезает ответ при появлении нового вопроса или маркера."""
    markers = ["Вопрос:", "Сколько будет", "Переведи", "?"]
    for marker in markers:
        idx = text.find(marker)
        if idx != -1 and idx > 0:  # если маркер не в самом начале
            return text[:idx].strip()
    return text.strip()

# ----------------------------------------------------------------
# Демонстрации (по NUM_DEMOS примеров на тип)
# ----------------------------------------------------------------
DEMO_TRANSLATIONS: List[Tuple[str, str]] = [
    ("Переведи на английский: Привет, как дела?", "Hello, how are you?"),
    ("Переведи на английский: Сегодня отличная погода.", "Today is great weather."),
]

DEMO_MATH: List[Tuple[str, str]] = [
    ("Сколько будет 2 + 2? Ответ дай числом.", "4"),
    ("Сколько будет 5 * 3? Ответ дай числом.", "15"),
]

# ----------------------------------------------------------------
# Тестовый датасет (без демонстраций)
# ----------------------------------------------------------------
test_dataset = [
    {"type": "translation", "input": "Переведи на английский: Я люблю программирование.", "target": "I love programming."},
    {"type": "translation", "input": "Переведи на английский: Который час?", "target": "What time is it?"},
    {"type": "translation", "input": "Переведи на английский: Это очень интересно.", "target": "This is very interesting."},
    {"type": "math", "input": "Сколько будет 10 - 4? Ответ дай числом.", "target": "6"},
    {"type": "math", "input": "Сколько будет 12 / 3? Ответ дай числом.", "target": "4"},
    {"type": "math", "input": "Сколько будет 7 + 8? Ответ дай числом.", "target": "15"},
]

print("\n" + "=" * 70)
print("ТЕСТОВЫЙ ДАТАСЕТ")
print("=" * 70)
for i, ex in enumerate(test_dataset, 1):
    print(f"{i}. [{ex['type']}] {ex['input']} -> {ex['target']}")
print("=" * 70)

# ----------------------------------------------------------------
# Генерация с Few-shot (2 примера) + системное сообщение
# ----------------------------------------------------------------
def generate_fewshot(
    question: str,
    demonstrations: List[Tuple[str, str]],
    max_new_tokens: int = MAX_NEW_TOKENS,
) -> Dict[str, Any]:
    """
    Формирует диалог:
    - system: краткая инструкция (не задавай вопросов, отвечай кратко)
    - user/assistant пары демонстраций
    - user: текущий вопрос
    """
    # Системное сообщение (поддерживается в Phi-3)
    system_msg = "Ты — полезный ассистент. Отвечай кратко и только на последний вопрос. Не задавай новых вопросов."

    messages = [{"role": "system", "content": system_msg}]

    for inp, out in demonstrations:
        messages.append({"role": "user", "content": inp})
        messages.append({"role": "assistant", "content": out})

    messages.append({"role": "user", "content": question})

    # Применяем чат-шаблон
    formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    gen_config = GenerationConfig(
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=TEMPERATURE,
        top_p=TOP_P,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    start_time = time.time()
    with torch.inference_mode():
        result = generator(formatted, return_full_text=False, generation_config=gen_config)
    end_time = time.time()

    raw = result[0]["generated_text"].strip()
    # Пост-обработка
    cleaned = clean_response(raw)
    if not cleaned:
        cleaned = raw

    input_tokens = len(tokenizer.encode(formatted))
    output_tokens = len(tokenizer.encode(cleaned))
    gen_time = end_time - start_time
    tps = output_tokens / gen_time if gen_time > 0 else 0.0

    return {
        "text": cleaned,
        "metrics": {
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "generation_time_sec": round(gen_time, 3),
            "tokens_per_second": round(tps, 2),
        }
    }

# ----------------------------------------------------------------
# Загрузка метрик
# ----------------------------------------------------------------
bleu_metric = load("bleu") if HAS_EVALUATE else None
rouge_scorer_obj = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True) if HAS_ROUGE else None

# ----------------------------------------------------------------
# Оценка
# ----------------------------------------------------------------
def evaluate_fewshot(test_dataset):
    predictions, refs, types, perf_metrics = [], [], [], []
    bleu_scores, rouge_scores = [], []
    math_correct, math_total = 0, 0

    print("\n" + "=" * 70)
    print("ЗАПУСК ОЦЕНКИ (FEW-SHOT, 2 демонстрации)")
    print("=" * 70)

    for idx, ex in enumerate(test_dataset, 1):
        ex_type = ex["type"]
        print(f"\n--- Пример {idx} ({ex_type}) ---")
        print(f"Вопрос: {ex['input']}")
        print(f"Эталон: {ex['target']}")

        demos = DEMO_TRANSLATIONS if ex_type == "translation" else DEMO_MATH
        result = generate_fewshot(ex["input"], demos)
        pred = result["text"]
        metrics = result["metrics"]

        print(f"Ответ модели: {pred}")
        print(f"   ⏱ {metrics['generation_time_sec']} сек, {metrics['output_tokens']} токенов, TPS: {metrics['tokens_per_second']}")

        predictions.append(pred)
        refs.append([ex["target"]])
        types.append(ex_type)
        perf_metrics.append(metrics)

        # Качество
        if ex_type == "translation":
            if bleu_metric:
                try:
                    score = bleu_metric.compute(predictions=[pred], references=[[ex["target"]]])
                    bleu_scores.append(score["bleu"])
                except Exception as e:
                    pass
            if rouge_scorer_obj:
                try:
                    scores = rouge_scorer_obj.score(ex["target"], pred)
                    rouge_scores.append(scores['rougeL'].fmeasure)
                except:
                    pass
        elif ex_type == "math":
            math_total += 1
            pred_num = extract_number(pred)
            ref_num = extract_number(ex["target"])
            if pred_num and ref_num and pred_num == ref_num:
                math_correct += 1
                print("   ✅ Математика верно")
            else:
                print(f"   ❌ Математика: извлечено '{pred_num}', ожидалось '{ref_num}'")

    # Итоги качества
    print("\n" + "=" * 70)
    print("РЕЗУЛЬТАТЫ КАЧЕСТВА")
    print("=" * 70)
    if bleu_scores:
        print(f"📊 Средний BLEU: {sum(bleu_scores)/len(bleu_scores):.3f}")
    else:
        print("📊 BLEU: не доступен (установите sacrebleu)")

    if rouge_scores:
        print(f"📊 Средний ROUGE-L: {sum(rouge_scores)/len(rouge_scores):.3f}")

    if math_total:
        print(f"📊 Точность на математике: {math_correct/math_total:.2%} ({math_correct}/{math_total})")

    # Средние метрики производительности
    print("\n" + "=" * 70)
    print("СРЕДНИЕ МЕТРИКИ ПРОИЗВОДИТЕЛЬНОСТИ")
    print("=" * 70)
    if perf_metrics:
        avg_in = sum(m["input_tokens"] for m in perf_metrics) / len(perf_metrics)
        avg_out = sum(m["output_tokens"] for m in perf_metrics) / len(perf_metrics)
        avg_time = sum(m["generation_time_sec"] for m in perf_metrics) / len(perf_metrics)
        avg_tps = sum(m["tokens_per_second"] for m in perf_metrics) / len(perf_metrics)
        print(f"Среднее токенов в запросе:   {avg_in:.1f}")
        print(f"Среднее сгенерировано токенов: {avg_out:.1f}")
        print(f"Среднее время генерации:     {avg_time:.3f} сек.")
        print(f"Средняя скорость (TPS):      {avg_tps:.2f} токен/сек.")

    return {"predictions": predictions, "perf_metrics": perf_metrics}

# ----------------------------------------------------------------
# Запуск
# ----------------------------------------------------------------
if __name__ == "__main__":
    results = evaluate_fewshot(test_dataset)
    print("\n" + "=" * 70)
    print("✅ ОЦЕНКА FEW-SHOT ЗАВЕРШЕНА")
    print("=" * 70)
```

---

#### Что происходит в коде (разбор структуры промпта)

Перед тем, как отправить запрос, функция `generate_fewshot` строит диалог из трёх частей:

1. **Системное сообщение**  
   `"Ты — полезный ассистент. Отвечай кратко и только на последний вопрос. Не задавай новых вопросов."`  
   Оно задаёт общий тон и ограничения, помогает модели не отклоняться в сторону.

2. **Демонстрации** (2 пары user / assistant) – например, для переводов:
   ```
   user:  Переведи на английский: Привет, как дела?
   assistant: Hello, how are you?
   user:  Переведи на английский: Сегодня отличная погода.
   assistant: Today is great weather.
   ```

3. **Текущий вопрос** (user) – уже без ответа, его должна сгенерировать модель.

Затем всё это форматируется через `apply_chat_template`, что добавляет специальные токены модели Phi-3 (`<|user|>`, `<|assistant|>` и т.д.).

**Пример реального промпта** (для перевода «Я люблю программирование») выглядит так (упрощённо):

```
<|system|>
Ты — полезный ассистент. Отвечай кратко и только на последний вопрос. Не задавай новых вопросов.<|end|>
<|user|>
Переведи на английский: Привет, как дела?<|end|>
<|assistant|>
Hello, how are you?<|end|>
<|user|>
Переведи на английский: Сегодня отличная погода.<|end|>
<|assistant|>
Today is great weather.<|end|>
<|user|>
Переведи на английский: Я люблю программирование.<|end|>
<|assistant|>
```

Модель достраивает ответ после последнего токена `<|assistant|>`.

---

#### Результаты выполнения (на Tesla T4)

```
======================================================================
ПРОВЕРКА ОКРУЖЕНИЯ
======================================================================
PyTorch: 2.11.0+cu128
CUDA доступна: True
GPU: Tesla T4
VRAM: 14.56 GB
======================================================================

🔹 Загружаем токенизатор...
✅ Токенизатор загружен

🔹 Загружаем модель...
✅ Модель загружена за 33.95 сек.

======================================================================
ИНФОРМАЦИЯ О МОДЕЛИ
======================================================================
Всего параметров: 3.821 B
Занято VRAM: 13.43 GB
======================================================================

✅ Pipeline создан

======================================================================
ТЕСТОВЫЙ ДАТАСЕТ
======================================================================
1. [translation] Переведи на английский: Я люблю программирование. -> I love programming.
2. [translation] Переведи на английский: Который час? -> What time is it?
3. [translation] Переведи на английский: Это очень интересно. -> This is very interesting.
4. [math] Сколько будет 10 - 4? Ответ дай числом. -> 6
5. [math] Сколько будет 12 / 3? Ответ дай числом. -> 4
6. [math] Сколько будет 7 + 8? Ответ дай числом. -> 15
======================================================================

======================================================================
ЗАПУСК ОЦЕНКИ (FEW-SHOT, 2 демонстрации)
======================================================================

--- Пример 1 (translation) ---
Вопрос: Переведи на английский: Я люблю программирование.
Эталон: I love programming.
Ответ модели: I love programming.
   ⏱ 0.333 сек, 4 токенов, TPS: 12.01

--- Пример 2 (translation) ---
Вопрос: Переведи на английский: Который час?
Эталон: What time is it?
Ответ модели: What time is it
   ⏱ 0.291 сек, 4 токенов, TPS: 13.76

--- Пример 3 (translation) ---
Вопрос: Переведи на английский: Это очень интересно.
Эталон: This is very interesting.
Ответ модели: This is very interesting.
   ⏱ 0.279 сек, 5 токенов, TPS: 17.9

--- Пример 4 (math) ---
Вопрос: Сколько будет 10 - 4? Ответ дай числом.
Эталон: 6
Ответ модели: 6
   ⏱ 0.172 сек, 2 токенов, TPS: 11.62
   ✅ Математика верно

--- Пример 5 (math) ---
Вопрос: Сколько будет 12 / 3? Ответ дай числом.
Эталон: 4
Ответ модели: 4
   ⏱ 0.174 сек, 2 токенов, TPS: 11.52
   ✅ Математика верно

--- Пример 6 (math) ---
Вопрос: Сколько будет 7 + 8? Ответ дай числом.
Эталон: 15
Ответ модели: 15
   ⏱ 1.16 сек, 3 токенов, TPS: 2.59
   ✅ Математика верно

======================================================================
РЕЗУЛЬТАТЫ КАЧЕСТВА
======================================================================
📊 BLEU: не доступен (установите sacrebleu)
📊 Средний ROUGE-L: 1.000
📊 Точность на математике: 100.00% (3/3)

======================================================================
СРЕДНИЕ МЕТРИКИ ПРОИЗВОДИТЕЛЬНОСТИ
======================================================================
Среднее токенов в запросе:   105.2
Среднее сгенерировано токенов: 3.3
Среднее время генерации:     0.401 сек.
Средняя скорость (TPS):      11.57 токен/сек.

======================================================================
✅ ОЦЕНКА FEW-SHOT ЗАВЕРШЕНА
======================================================================
```

#### Сравнение с Zero‑shot (из раздела 3.1)

| Параметр | Zero‑shot | Few‑shot |
| :--- | :--- | :--- |
| **Качество перевода (ROUGE-L)** | 0.933 | **1.000** (идеальное совпадение) |
| **Точность на математике** | 100% | 100% |
| **Средняя длина запроса (токены)** | ~20 | ~105 (в 5 раз больше) |
| **Средняя скорость генерации (TPS)** | 3.08 | **11.57** (в 3.7 раза быстрее!) |

**Почему Few‑shot быстрее?**  
В Few‑shot модель уже «понимает», что от неё требуется, и генерирует **короткие, точные ответы** (в среднем 3.3 токена против 8 в Zero‑shot). Несмотря на более длинный промпт, генерация занимает меньше времени, потому что выходной текст короче и модель реже «переспрашивает».

**Качество** – Few‑shot дал идеальные переводы (ROUGE‑L = 1.0), тогда как Zero‑shot иногда давал близкие, но не точные варианты («Today's weather is great» вместо «Today is great weather»).

---

#### Когда Few‑shot выигрывает:
- Нужен **строгий формат** или **однозначный ответ**.
- Модель часто даёт нестабильные ответы – примеры стабилизируют её.
- Вы хотите **ускорить инференс** за счёт коротких, предсказуемых ответов (парадокс: длиннее контекст, но короче генерация → общее время может уменьшиться).

#### Когда Few‑shot недостаточен:
- Задача требует глубоких рассуждений, которые не умещаются в 5 примерах.
- Нужны новые факты (их нет в весах, и примеры их не содержат) – здесь лучше RAG.
- Контекстное окно не позволяет вставить достаточно примеров (например, при длинных текстах).

---

#### Как модифицировать код под свою задачу

1. **Сменить демонстрации** – замените списки `DEMO_TRANSLATIONS` и `DEMO_MATH` на свои пары «вопрос–ответ».
2. **Сменить тестовый датасет** – обновите `test_dataset` под свои примеры.
3. **Изменить системное сообщение** – задайте свой тон (например, «Отвечай строго в формате JSON»).
4. **Увеличить число демонстраций** – измените `NUM_DEMOS` и добавьте больше пар в списки.


### 3.3. Продвинутые техники промптинга: дотягиваем до уровня тонкой настройки

Если Zero‑shot и Few‑shot не дают нужного качества, можно применить более сложные стратегии, которые заставляют модель «думать» глубже. Эти техники часто приближают качество к тому, что даёт тонкая настройка, но без её стоимости.

В этом разделе мы на практике сравним три режима на одном датасете, включающем **сложное математическое выражение**, и увидим, как каждый подход влияет на качество и производительность.

---

#### 3.3.1. Chain‑of‑Thought (CoT) – пошаговое рассуждение

**Суть:** просим модель показать свои рассуждения шаг за шагом, прежде чем дать финальный ответ. Это заставляет модель активировать логические цепочки, которые она выучила во время предобучения.

**Разбор Zero‑shot CoT промпта по частям:**

1. **Постановка задачи** – чётко формулируем, что нужно вычислить:
   ```
   Вычисли значение выражения: 100-6*((-4)-3*(5-2*(9-4*(1-6)))). Ответ дай числом.
   ```

2. **Триггер рассуждения** – добавляем фразу, которая заставляет модель показать ход мыслей:
   ```
   Давайте подумаем шаг за шагом.
   ```

   Эта фраза критически важна – без неё модель может сразу выдать ответ (и часто ошибиться). С ней она начинает расписывать промежуточные шаги.

**Полный промпт (Zero‑shot CoT):**

```
Вычисли значение выражения: 100-6*((-4)-3*(5-2*(9-4*(1-6)))). Ответ дай числом. Давайте подумаем шаг за шагом.
```

**Разбор Few‑shot CoT промпта по частям:**

1. **Демонстрации с рассуждениями** – показываем модель, как нужно расписывать шаги:
   ```
   Пример 1:
   Вопрос: Вычисли значение выражения: (-80)-3*((-15)-4*(2-3*(12-5*(6-9)))). Ответ дай числом.
   Рассуждение:
   1. Внутренняя скобка: 6-9 = -3
   2. 5*(-3) = -15
   3. 12 - (-15) = 27
   4. 3*(27) = 81
   5. 2 - 81 = -79
   6. 4*(-79) = -316
   7. -15 - (-316) = 301
   8. 3*(301) = 903
   9. -80 - 903 = -983
   Ответ: -983.
   ```

2. **Новый вопрос** – задаём задачу без ответа:
   ```
   Вопрос: Вычисли значение выражения: 100-6*((-4)-3*(5-2*(9-4*(1-6)))). Ответ дай числом.
   Рассуждение:
   ```

---

#### 3.3.2. Self‑Consistency – голосование за лучший ответ

**Суть:** генерируем несколько независимых ответов (обычно с высокой температурой) для одного и того же запроса, а затем выбираем наиболее часто встречающийся ответ (или проводим голосование). Это снижает вариативность и повышает надёжность.

**Как это реализовано в коде:**
- Модель запускается 5 раз с температурой 0.7 (чтобы получать разные варианты).
- Для математических задач из каждого ответа извлекается число.
- Выбирается наиболее часто встречающееся число (мажоритарное голосование).

---

#### 3.3.3. Tree‑of‑Thought (ToT) – ветвление рассуждений

**Суть:** модель генерирует несколько возможных путей рассуждения, оценивает каждый и выбирает наилучший. Это как CoT, но с ветвлением и поиском (похоже на дерево решений). Требует программной обвязки (оценка каждого узла, бэктрекинг).

**Когда применять:** задачи, требующие перебора вариантов (планирование, оптимизация маршрута, сложная логика). Для большинства бизнес‑задач CoT + Self‑Consistency достаточно. В нашем эксперименте мы не реализуем ToT из‑за его сложности, но упоминаем как следующий уровень.

---

#### Практическое сравнение режимов (полный код)

Ниже приведён код, который сравнивает три режима на одном датасете:
- **Few‑shot** – обычные демонстрации без рассуждений.
- **CoT** – демонстрации с пошаговыми рассуждениями.
- **Self‑Consistency** – голосование из 5 генераций.

```python
# ================================================================
# Phi-3 Mini 4K Instruct — Сравнение режимов: Few-shot, CoT, Self-Consistency
# Сложные математические демонстрации для CoT
# ================================================================

!pip install evaluate sacrebleu rouge-score -q

import os, gc, re, time, torch, warnings
from typing import List, Dict, Any, Tuple
from collections import Counter
from transformers import (
    AutoTokenizer, AutoConfig, AutoModelForCausalLM,
    pipeline, GenerationConfig
)

warnings.filterwarnings("ignore", message=".*clean_up_tokenization_spaces.*")
warnings.filterwarnings("ignore", message=".*Phi3ForCausalLM has generative capabilities.*")

try:
    import evaluate
    from evaluate import load
    HAS_EVALUATE = True
except ImportError:
    HAS_EVALUATE = False

try:
    from rouge_score import rouge_scorer
    HAS_ROUGE = True
except ImportError:
    HAS_ROUGE = False

# ----------------------------------------------------------------
# Настройки
# ----------------------------------------------------------------
MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"
MAX_NEW_TOKENS = 512            # для CoT нужно больше токенов
TEMPERATURE = 0.7
TOP_P = 0.9
SEED = 42
NUM_CONSISTENCY = 5             # количество генераций для Self-Consistency
USE_SYSTEM = True

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ----------------------------------------------------------------
# Окружение
# ----------------------------------------------------------------
print("=" * 70)
print("ПРОВЕРКА ОКРУЖЕНИЯ")
print("=" * 70)
print(f"PyTorch: {torch.__version__}")
print(f"CUDA доступна: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
    DTYPE = torch.float16
    torch.cuda.reset_peak_memory_stats()
else:
    DTYPE = torch.float32
print("=" * 70)

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# ----------------------------------------------------------------
# Загрузка модели
# ----------------------------------------------------------------
print("\n🔹 Загружаем токенизатор...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print("✅ Токенизатор загружен")

print("\n🔹 Загружаем модель...")
model_kwargs = {
    "config": AutoConfig.from_pretrained(MODEL_NAME),
    "torch_dtype": DTYPE,
}
if torch.cuda.is_available():
    model_kwargs["device_map"] = "auto"

start = time.time()
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, **model_kwargs)
load_time = time.time() - start
model.eval()
print(f"✅ Модель загружена за {load_time:.2f} сек.")

print("\n" + "=" * 70)
print("ИНФОРМАЦИЯ О МОДЕЛИ")
print("=" * 70)
print(f"Всего параметров: {sum(p.numel() for p in model.parameters()) / 1e9:.3f} B")
if torch.cuda.is_available():
    print(f"Занято VRAM: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print("=" * 70)

generator = pipeline("text-generation", model=model, tokenizer=tokenizer)
print("\n✅ Pipeline создан")

# ----------------------------------------------------------------
# Вспомогательные функции
# ----------------------------------------------------------------
def extract_number(text: str) -> str:
    """Извлекает последнее число или число после '=' / 'равно'."""
    text = text.replace(",", "")
    if "=" in text:
        parts = text.split("=")
        if len(parts) > 1:
            nums = re.findall(r"[-+]?\d*\.?\d+", parts[-1])
            if nums:
                return nums[0]
    if "равно" in text:
        parts = text.split("равно")
        if len(parts) > 1:
            nums = re.findall(r"[-+]?\d*\.?\d+", parts[-1])
            if nums:
                return nums[0]
    nums = re.findall(r"[-+]?\d*\.?\d+", text)
    return nums[-1] if nums else ""

def clean_response(text: str) -> str:
    """Обрезает ответ при появлении нового вопроса или маркера."""
    markers = ["Вопрос:", "Сколько будет", "Переведи", "?"]
    for marker in markers:
        idx = text.find(marker)
        if idx != -1 and idx > 0:
            return text[:idx].strip()
    return text.strip()

def extract_final_answer(text: str) -> str:
    """
    Извлекает финальный ответ из текста с рассуждениями.
    Ищет фразу 'Ответ:' или последнее число.
    """
    if "Ответ:" in text:
        parts = text.split("Ответ:")
        if len(parts) > 1:
            candidate = parts[-1].strip()
            nums = re.findall(r"[-+]?\d*\.?\d+", candidate)
            if nums:
                return nums[-1]
            return candidate
    return extract_number(text)

# ----------------------------------------------------------------
# Демонстрации
# ----------------------------------------------------------------
# Для перевода (одинаковы для всех режимов)
DEMO_TRANSLATIONS = [
    ("Переведи на английский: Привет, как дела?", "Hello, how are you?"),
    ("Переведи на английский: Сегодня отличная погода.", "Today is great weather."),
]

# Для математики (без CoT) – простые задачи
DEMO_MATH = [
    ("Сколько будет 2 + 2? Ответ дай числом.", "4"),
    ("Сколько будет 5 * 3? Ответ дай числом.", "15"),
]

# Для CoT – сложные многошаговые выражения с рассуждениями
DEMO_MATH_COT = [
    (
        "Вычисли значение выражения: (-80)-3*((-15)-4*(2-3*(12-5*(6-9)))). Ответ дай числом.",
        "Рассуждение:\n"
        "1. Внутренняя скобка: 6-9 = -3\n"
        "2. 5*(-3) = -15\n"
        "3. 12 - (-15) = 27\n"
        "4. 3*(27) = 81\n"
        "5. 2 - 81 = -79\n"
        "6. 4*(-79) = -316\n"
        "7. -15 - (-316) = 301\n"
        "8. 3*(301) = 903\n"
        "9. -80 - 903 = -983\n"
        "Ответ: -983."
    ),
    (
        "Вычисли значение выражения: (-210)+5*((-8)-2*(-10-3*(7-2*(4-11)))). Ответ дай числом.",
        "Рассуждение:\n"
        "1. Внутренняя скобка: 4-11 = -7\n"
        "2. 2*(-7) = -14\n"
        "3. 7 - (-14) = 21\n"
        "4. 3*(21) = 63\n"
        "5. -10 - 63 = -73\n"
        "6. 2*(-73) = -146\n"
        "7. -8 - (-146) = 138\n"
        "8. 5*138 = 690\n"
        "9. -210 + 690 = 480\n"
        "Ответ: 480."
    ),
]

# ----------------------------------------------------------------
# Тестовый датасет (включая сложное выражение)
# ----------------------------------------------------------------
test_dataset = [
    {"type": "translation", "input": "Переведи на английский: Я люблю программирование.", "target": "I love programming."},
    {"type": "translation", "input": "Переведи на английский: Который час?", "target": "What time is it?"},
    {"type": "translation", "input": "Переведи на английский: Это очень интересно.", "target": "This is very interesting."},
    {"type": "math", "input": "Сколько будет 10 - 4? Ответ дай числом.", "target": "6"},
    {"type": "math", "input": "Сколько будет 12 / 3? Ответ дай числом.", "target": "4"},
    {"type": "math", "input": "Сколько будет 7 + 8? Ответ дай числом.", "target": "15"},
    # Сложное выражение (аналогичное демонстрациям CoT)
    {"type": "math", "input": "Вычисли значение выражения: 100-6*((-4)-3*(5-2*(9-4*(1-6)))). Ответ дай числом.", "target": "-830"},
]

print("\n" + "=" * 70)
print("ТЕСТОВЫЙ ДАТАСЕТ")
print("=" * 70)
for i, ex in enumerate(test_dataset, 1):
    print(f"{i}. [{ex['type']}] {ex['input']} -> {ex['target']}")
print("=" * 70)

# ----------------------------------------------------------------
# Базовые функции генерации (без CoT)
# ----------------------------------------------------------------
def build_messages(demonstrations, question, system_msg=None):
    messages = []
    if system_msg and USE_SYSTEM:
        messages.append({"role": "system", "content": system_msg})
    for inp, out in demonstrations:
        messages.append({"role": "user", "content": inp})
        messages.append({"role": "assistant", "content": out})
    messages.append({"role": "user", "content": question})
    return messages

def generate_with_demos(question, demonstrations, system_msg=None,
                        max_new_tokens=MAX_NEW_TOKENS, temperature=0.2):
    messages = build_messages(demonstrations, question, system_msg)
    formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    gen_config = GenerationConfig(
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=TOP_P,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    start_time = time.time()
    with torch.inference_mode():
        result = generator(formatted, return_full_text=False, generation_config=gen_config)
    end_time = time.time()
    raw = result[0]["generated_text"].strip()
    cleaned = clean_response(raw)
    if not cleaned:
        cleaned = raw
    return cleaned, end_time - start_time, formatted

# ----------------------------------------------------------------
# Генерация с CoT (пошаговое рассуждение)
# ----------------------------------------------------------------
def generate_cot(question, demonstrations, system_msg=None, max_new_tokens=MAX_NEW_TOKENS*2):
    messages = build_messages(demonstrations, question, system_msg)
    # Добавляем инструкцию к вопросу, если её нет
    if "шаг за шагом" not in question.lower():
        modified_question = question + " Давайте подумаем шаг за шагом."
        messages[-1]["content"] = modified_question
    formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    gen_config = GenerationConfig(
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.3,
        top_p=TOP_P,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    start_time = time.time()
    with torch.inference_mode():
        result = generator(formatted, return_full_text=False, generation_config=gen_config)
    end_time = time.time()
    raw = result[0]["generated_text"].strip()
    # Не обрезаем по маркерам, чтобы не потерять рассуждения
    cleaned = clean_response(raw)
    if not cleaned:
        cleaned = raw
    return cleaned, end_time - start_time, formatted

# ----------------------------------------------------------------
# Self-Consistency (голосование)
# ----------------------------------------------------------------
def generate_consensus(question, demonstrations, system_msg=None,
                       n=NUM_CONSISTENCY, temperature=0.7):
    candidates = []
    for _ in range(n):
        answer, _, _ = generate_with_demos(
            question, demonstrations, system_msg,
            max_new_tokens=MAX_NEW_TOKENS, temperature=temperature
        )
        # Извлекаем число для математических задач
        if "сколько" in question.lower() or "вычисли" in question.lower() or "Ответ дай числом" in question:
            final = extract_number(answer)
        else:
            final = answer
        candidates.append(final)
    counter = Counter(candidates)
    most_common = counter.most_common(1)[0][0]
    return most_common, candidates

# ----------------------------------------------------------------
# Загрузка метрик
# ----------------------------------------------------------------
bleu_metric = load("bleu") if HAS_EVALUATE else None
rouge_scorer_obj = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True) if HAS_ROUGE else None

# ----------------------------------------------------------------
# Функция оценки для заданного режима
# ----------------------------------------------------------------
def evaluate_mode(mode: str):
    print("\n" + "=" * 70)
    print(f"ОЦЕНКА В РЕЖИМЕ: {mode.upper()}")
    print("=" * 70)

    predictions = []
    refs = []
    perf_times = []
    perf_tokens = []
    math_correct = 0
    math_total = 0
    bleu_scores = []
    rouge_scores = []

    system_msg = "Ты — полезный ассистент. Отвечай кратко и только на последний вопрос. Не задавай новых вопросов."

    for idx, ex in enumerate(test_dataset, 1):
        ex_type = ex["type"]
        question = ex["input"]
        target = ex["target"]

        print(f"\n--- Пример {idx} ({ex_type}) ---")
        print(f"Вопрос: {question}")
        print(f"Эталон: {target}")

        if mode == "fewshot":
            demos = DEMO_TRANSLATIONS if ex_type == "translation" else DEMO_MATH
            answer, gen_time, _ = generate_with_demos(
                question, demos, system_msg, temperature=0.2
            )
        elif mode == "cot":
            if ex_type == "translation":
                demos = DEMO_TRANSLATIONS
                answer, gen_time, _ = generate_with_demos(
                    question, demos, system_msg, temperature=0.2
                )
            else:
                demos = DEMO_MATH_COT
                answer, gen_time, _ = generate_cot(
                    question, demos, system_msg
                )
        elif mode == "selfconsistency":
            demos = DEMO_TRANSLATIONS if ex_type == "translation" else DEMO_MATH
            if ex_type == "translation":
                answer, gen_time, _ = generate_with_demos(
                    question, demos, system_msg, temperature=0.2
                )
            else:
                answer, candidates = generate_consensus(
                    question, demos, system_msg, n=NUM_CONSISTENCY, temperature=0.7
                )
                # Оцениваем время как среднее по n генерациям (приблизительно)
                _, gen_time, _ = generate_with_demos(
                    question, demos, system_msg, temperature=0.7
                )
                gen_time *= NUM_CONSISTENCY

        # Очистка ответа (для CoT не обрезаем полностью)
        if mode != "cot":
            answer = clean_response(answer)

        print(f"Ответ модели: {answer}")
        print(f"   ⏱ {gen_time:.3f} сек")

        predictions.append(answer)
        refs.append([target])
        perf_times.append(gen_time)
        perf_tokens.append(len(tokenizer.encode(answer)))

        # Метрики качества
        if ex_type == "translation":
            if bleu_metric:
                try:
                    score = bleu_metric.compute(predictions=[answer], references=[[target]])
                    bleu_scores.append(score["bleu"])
                except Exception as e:
                      print(f"BLEU compute error: {e}")
    
            if rouge_scorer_obj:
                try:
                    rouge_scores.append(rouge_scorer_obj.score(target, answer)['rougeL'].fmeasure)
                except:
                    pass
        elif ex_type == "math":
            math_total += 1
            if mode == "selfconsistency":
                pred_num = answer  # уже извлечено число
            else:
                pred_num = extract_number(answer) if mode != "cot" else extract_final_answer(answer)
            ref_num = extract_number(target)
            if pred_num and ref_num and pred_num == ref_num:
                math_correct += 1
                print("   ✅ Математика верно")
            else:
                print(f"   ❌ Математика: извлечено '{pred_num}', ожидалось '{ref_num}'")

    # Итоговые метрики
    print("\n" + "=" * 70)
    print(f"РЕЗУЛЬТАТЫ КАЧЕСТВА ({mode.upper()})")
    print("=" * 70)
    if bleu_scores:
        print(f"📊 Средний BLEU: {sum(bleu_scores)/len(bleu_scores):.3f}")
    else:
        print("📊 BLEU: не доступен")
    if rouge_scores:
        print(f"📊 Средний ROUGE-L: {sum(rouge_scores)/len(rouge_scores):.3f}")
    if math_total:
        print(f"📊 Точность на математике: {math_correct/math_total:.2%} ({math_correct}/{math_total})")

    print("\n" + "=" * 70)
    print(f"СРЕДНИЕ МЕТРИКИ ПРОИЗВОДИТЕЛЬНОСТИ ({mode.upper()})")
    print("=" * 70)
    if perf_times:
        avg_time = sum(perf_times)/len(perf_times)
        avg_tokens = sum(perf_tokens)/len(perf_tokens)
        avg_tps = avg_tokens / avg_time if avg_time > 0 else 0
        print(f"Среднее время генерации: {avg_time:.3f} сек.")
        print(f"Среднее число токенов в ответе: {avg_tokens:.1f}")
        print(f"Средняя скорость: {avg_tps:.2f} токен/сек.")

    return {
        "mode": mode,
        "predictions": predictions,
        "bleu": sum(bleu_scores)/len(bleu_scores) if bleu_scores else None,
        "rouge": sum(rouge_scores)/len(rouge_scores) if rouge_scores else None,
        "math_accuracy": math_correct/math_total if math_total else None,
        "avg_time": sum(perf_times)/len(perf_times) if perf_times else None,
    }

# ----------------------------------------------------------------
# Запуск всех режимов и сравнение
# ----------------------------------------------------------------
if __name__ == "__main__":
    modes = ["fewshot", "cot", "selfconsistency"]
    results = {}
    for mode in modes:
        results[mode] = evaluate_mode(mode)

    # Сравнительная таблица
    print("\n" + "=" * 70)
    print("СРАВНЕНИЕ РЕЖИМОВ")
    print("=" * 70)
    print(f"{'Режим':<18} {'BLEU':<8} {'ROUGE-L':<10} {'Math Acc':<10} {'Время, с':<10}")
    for mode, res in results.items():
        bleu = f"{res['bleu']:.3f}" if res['bleu'] is not None else "—"
        rouge = f"{res['rouge']:.3f}" if res['rouge'] is not None else "—"
        acc = f"{res['math_accuracy']:.2%}" if res['math_accuracy'] is not None else "—"
        t = f"{res['avg_time']:.3f}" if res['avg_time'] is not None else "—"
        print(f"{mode:<18} {bleu:<8} {rouge:<10} {acc:<10} {t:<10}")

    print("\n" + "=" * 70)
    print("✅ ВСЕ РЕЖИМЫ ОЦЕНЕНЫ")
    print("=" * 70)
```

---

#### Результаты выполнения (на Tesla T4)

```
======================================================================
ПРОВЕРКА ОКРУЖЕНИЯ
======================================================================
PyTorch: 2.11.0+cu128
CUDA доступна: True
GPU: Tesla T4
VRAM: 14.56 GB
======================================================================

🔹 Загружаем токенизатор...
✅ Токенизатор загружен

🔹 Загружаем модель...
✅ Модель загружена за 44.43 сек.

======================================================================
ИНФОРМАЦИЯ О МОДЕЛИ
======================================================================
Всего параметров: 3.821 B
Занято VRAM: 13.43 GB
======================================================================

✅ Pipeline создан

======================================================================
ТЕСТОВЫЙ ДАТАСЕТ
======================================================================
1. [translation] Переведи на английский: Я люблю программирование. -> I love programming.
2. [translation] Переведи на английский: Который час? -> What time is it?
3. [translation] Переведи на английский: Это очень интересно. -> This is very interesting.
4. [math] Сколько будет 10 - 4? Ответ дай числом. -> 6
5. [math] Сколько будет 12 / 3? Ответ дай числом. -> 4
6. [math] Сколько будет 7 + 8? Ответ дай числом. -> 15
7. [math] Вычисли значение выражения: 100-6*((-4)-3*(5-2*(9-4*(1-6)))). Ответ дай числом. -> -830
======================================================================

======================================================================
ОЦЕНКА В РЕЖИМЕ: FEWSHOT
======================================================================

--- Пример 1 (translation) ---
Вопрос: Переведи на английский: Я люблю программирование.
Эталон: I love programming.
Ответ модели: I love programming.
   ⏱ 1.764 сек

--- Пример 2 (translation) ---
Вопрос: Переведи на английский: Который час?
Эталон: What time is it?
Ответ модели: What time is it
   ⏱ 2.010 сек

--- Пример 3 (translation) ---
Вопрос: Переведи на английский: Это очень интересно.
Эталон: This is very interesting.
Ответ модели: This is very interesting.
   ⏱ 1.906 сек

--- Пример 4 (math) ---
Вопрос: Сколько будет 10 - 4? Ответ дай числом.
Эталон: 6
Ответ модели: 6
   ⏱ 1.032 сек
   ✅ Математика верно

--- Пример 5 (math) ---
Вопрос: Сколько будет 12 / 3? Ответ дай числом.
Эталон: 4
Ответ модели: 4
   ⏱ 1.022 сек
   ✅ Математика верно

--- Пример 6 (math) ---
Вопрос: Сколько будет 7 + 8? Ответ дай числом.
Эталон: 15
Ответ модели: 15
   ⏱ 7.903 сек
   ✅ Математика верно

--- Пример 7 (math) ---
Вопрос: Вычисли значение выражения: 100-6*((-4)-3*(5-2*(9-4*(1-6)))). Ответ дай числом.
Эталон: -830
Ответ модели: Для решения сложного выражения необходимо следовать правилам порядка операций... (следует длинное рассуждение)
...
Ответ: 70.
   ⏱ 97.988 сек
   ❌ Математика: извлечено '70', ожидалось '-830'

======================================================================
РЕЗУЛЬТАТЫ КАЧЕСТВА (FEWSHOT)
======================================================================
📊 BLEU: не доступен
📊 Средний ROUGE-L: 1.000
📊 Точность на математике: 75.00% (3/4)

======================================================================
СРЕДНИЕ МЕТРИКИ ПРОИЗВОДИТЕЛЬНОСТИ (FEWSHOT)
======================================================================
Среднее время генерации: 16.232 сек.
Среднее число токенов в ответе: 50.4
Средняя скорость: 3.11 токен/сек.

======================================================================
ОЦЕНКА В РЕЖИМЕ: COT
======================================================================

--- Пример 1 (translation) ---
Вопрос: Переведи на английский: Я люблю программирование.
Эталон: I love programming.
Ответ модели: I love programming.
   ⏱ 1.651 сек

--- Пример 2 (translation) ---
Вопрос: Переведи на английский: Который час?
Эталон: What time is it?
Ответ модели: What time is it
   ⏱ 1.957 сек

--- Пример 3 (translation) ---
Вопрос: Переведи на английский: Это очень интересно.
Эталон: This is very interesting.
Ответ модели: This is very interesting.
   ⏱ 2.000 сек

--- Пример 4 (math) ---
Вопрос: Сколько будет 10 - 4? Ответ дай числом.
Эталон: 6
Ответ модели: Рассуждение:
1. 10 - 4 = 6
Ответ: 6.
   ⏱ 6.748 сек
   ✅ Математика верно

--- Пример 5 (math) ---
Вопрос: Сколько будет 12 / 3? Ответ дай числом.
Эталон: 4
Ответ модели: Рассуждение:
1. 12 / 3 = 4
Ответ: 4.
   ⏱ 6.976 сек
   ✅ Математика верно

--- Пример 6 (math) ---
Вопрос: Сколько будет 7 + 8? Ответ дай числом.
Эталон: 15
Ответ модели: Рассуждение:
1. Сначала сложим числа 7 и 8.
2. 7 + 8 = 15.
Ответ: 15.
   ⏱ 11.958 сек
   ✅ Математика верно

--- Пример 7 (math) ---
Вопрос: Вычисли значение выражения: 100-6*((-4)-3*(5-2*(9-4*(1-6)))). Ответ дай числом.
Эталон: -830
Ответ модели: Рассуждение:
1. Внутренняя скобка: 1-6 = -5
2. 4*(-5) = -20
3. 9 - (-20) = 29
4. 2*(29) = 58
5. 5 - 58 = -53
6. 3*(-53) = -159
7. -4 - (-159) = 155
8. -6*155 = -930
9. 100 - (-930) = 1030
Ответ: 1030.
   ⏱ 39.005 сек
   ❌ Математика: извлечено '1030', ожидалось '-830'

======================================================================
РЕЗУЛЬТАТЫ КАЧЕСТВА (COT)
======================================================================
📊 BLEU: не доступен
📊 Средний ROUGE-L: 1.000
📊 Точность на математике: 75.00% (3/4)

======================================================================
СРЕДНИЕ МЕТРИКИ ПРОИЗВОДИТЕЛЬНОСТИ (COT)
======================================================================
Среднее время генерации: 10.042 сек.
Среднее число токенов в ответе: 36.1
Средняя скорость: 3.60 токен/сек.

======================================================================
ОЦЕНКА В РЕЖИМЕ: SELFCONSISTENCY
======================================================================

--- Пример 1 (translation) ---
Вопрос: Переведи на английский: Я люблю программирование.
Эталон: I love programming.
Ответ модели: I love programming.
   ⏱ 1.553 сек

--- Пример 2 (translation) ---
Вопрос: Переведи на английский: Который час?
Эталон: What time is it?
Ответ модели: What time is it
   ⏱ 1.816 сек

--- Пример 3 (translation) ---
Вопрос: Переведи на английский: Это очень интересно.
Эталон: This is very interesting.
Ответ модели: This is very interesting.
   ⏱ 1.839 сек

--- Пример 4 (math) ---
Вопрос: Сколько будет 10 - 4? Ответ дай числом.
Эталон: 6
Ответ модели: 6
   ⏱ 5.338 сек
   ✅ Математика верно

--- Пример 5 (math) ---
Вопрос: Сколько будет 12 / 3? Ответ дай числом.
Эталон: 4
Ответ модели: 4
   ⏱ 5.222 сек
   ✅ Математика верно

--- Пример 6 (math) ---
Вопрос: Сколько будет 7 + 8? Ответ дай числом.
Эталон: 15
Ответ модели: 15
   ⏱ 6.550 сек
   ✅ Математика верно

--- Пример 7 (math) ---
Вопрос: Вычисли значение выражения: 100-6*((-4)-3*(5-2*(9-4*(1-6)))). Ответ дай числом.
Эталон: -830
Ответ модели: -194
   ⏱ 307.143 сек
   ❌ Математика: извлечено '-194', ожидалось '-830'

======================================================================
РЕЗУЛЬТАТЫ КАЧЕСТВА (SELFCONSISTENCY)
======================================================================
📊 BLEU: не доступен
📊 Средний ROUGE-L: 1.000
📊 Точность на математике: 75.00% (3/4)

======================================================================
СРЕДНИЕ МЕТРИКИ ПРОИЗВОДИТЕЛЬНОСТИ (SELFCONSISTENCY)
======================================================================
Среднее время генерации: 47.066 сек.
Среднее число токенов в ответе: 3.4
Средняя скорость: 0.07 токен/сек.
```

---

#### Сравнительный анализ результатов

| Режим | BLEU | ROUGE‑L | Точность на математике | Среднее время, с | Средняя скорость, токен/сек |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **Few‑shot** | — | 1.000 | **75.00%** (3/4) | 16.232 | 3.11 |
| **CoT** | — | 1.000 | **75.00%** (3/4) | **10.042** | **3.60** |
| **Self‑Consistency** | — | 1.000 | **75.00%** (3/4) | 47.066 | 0.07 |

**Ключевые наблюдения:**

1. **Все режимы дали одинаковое качество** – 75% на математике и 100% на переводах (ROUGE‑L = 1.0). Сложное выражение не было решено ни одним методом.

2. **CoT оказался самым быстрым** – среднее время 10 секунд против 16 у Few‑shot и 47 у Self‑Consistency. Почему?
   - CoT использует **демонстрации с рассуждениями**, которые задают правильный формат. Модель выдаёт структурированный ответ с чётким «Ответ:» в конце.
   - Few‑shot без CoT на сложном выражении сгенерировал **очень длинный** (50+ токенов) и неструктурированный ответ, что заняло почти 100 секунд.

3. **Self‑Consistency оказался самым медленным** – 5 генераций по ~9 секунд = 47 секунд, но качество не улучшилось. Это показывает, что голосование не всегда спасает, если базовая модель систематически ошибается.

4. **Почему модель не решила сложное выражение?**
   - Выражение содержит вложенные скобки и отрицательные числа, что требует аккуратного соблюдения порядка операций.
   - Даже с CoT модель сделала ошибку на промежуточном шаге: `9 - 4*(1-6)` она вычислила как `29`, хотя правильно `9 - 4*(-5) = 9 + 20 = 29` – здесь ошибки нет. Но дальше пошла путаница: `5 - 2*29 = 5 - 58 = -53` – это верно. Однако затем `3*(-53) = -159` – верно. `-4 - (-159) = 155` – верно. `-6 * 155 = -930` – верно. `100 - (-930) = 1030` – **ошибка!** Правильно: `100 - (-930) = 100 + 930 = 1030`. Но эталонный ответ `-830`. Значит, модель неправильно поняла исходное выражение – возможно, пропустила умножение где‑то на этапе парсинга.

---

#### Практические выводы

1. **CoT – лучший выбор для сложных задач**, требующих логики:
   - Он быстрее Few‑shot за счёт структурированных демонстраций.
   - Он даёт **прозрачность** – вы видите ход рассуждений и можете найти ошибку.

2. **Self‑Consistency** имеет смысл, когда:
   - Модель даёт **разные** ответы при повторных запусках (высокая вариативность).
   - В нашем эксперименте вариативность была низкой – модель стабильно ошибалась, поэтому голосование не помогло.

3. **Когда всё‑таки нужна тонкая настройка?**
   - Если даже CoT + Self‑Consistency не дают нужной точности (как в нашем случае с 75%).
   - Если задача требует **специфического стиля** или **узкоспециализированных знаний**.
   - Если вы хотите **ускорить инференс** за счёт коротких, предсказуемых ответов без длинных промптов.

---

#### Как модифицировать код под свою задачу

1. **Сменить демонстрации CoT** – добавьте свои сложные примеры с рассуждениями в `DEMO_MATH_COT`.
2. **Изменить число генераций для Self‑Consistency** – поменяйте `NUM_CONSISTENCY`.
3. **Добавить новые типы задач** – создайте свои демонстрации для `DEMO_TRANSLATIONS` или других категорий.


### 3.4. Ключевое правило: «Если дают 90% – не трогайте тонкую настройку»

Я сформулирую правило, которое сэкономит вам сотни часов и тысячи долларов:

> **Если комбинация Zero‑shot, Few‑shot, CoT и Self‑Consistency даёт ≥90% от целевого качества (по вашим метрикам) – тонкая настройка не нужна.**

**Когда это правило особенно верно:**
- У вас мало данных (меньше 500 примеров) – тонкая настройка переобучится.
- Задача творческая (генерация текста, креатив) – промпты сохраняют вариативность.
- Вы на этапе MVP – нужно быстро проверить гипотезу.
- У вас нет GPU – можно использовать API (или бесплатные модели, если задача простая).
- Вы используете очень мощную базовую модель (GPT‑4, Claude) – она уже содержит огромные знания.

**Когда даже 90% недостаточно и нужно идти дальше:**
- Требуется 99%+ точность на узком домене (например, юридические документы).
- Необходимо стабильное структурирование вывода, которое промпты не гарантируют (из‑за вероятностной природы).
- Нужно снизить задержку или стоимость – дообученная маленькая модель может работать быстрее и дешевле, чем GPT‑4 с длинными промптами.

---

### 3.5. Практический алгоритм действий (пошаговая инструкция)

1. **Сформулируйте задачу** и определите метрику успеха (например, accuracy, F1, оценка эксперта).
2. **Напишите Zero‑shot промпт** – просто инструкцию. Проверьте на 5–10 примерах. Если качество >90% – стоп, вы закончили.
3. **Если нет – добавьте 3–5 примеров (Few‑shot)**. Подберите примеры, которые покрывают разные сценарии. Проверьте на валидационной выборке.
4. **Если всё ещё не хватает – примените CoT** (попросите модель рассуждать шаг за шагом). Для этого либо добавьте фразу «Давайте подумаем шаг за шагом», либо дайте пример с рассуждением.
5. **Если качество нестабильно** – реализуйте Self‑Consistency: запускайте модель 3–5 раз и голосуйте.
6. **Измерьте итоговое качество**. Если оно ≥90% от целевого – **останавливайтесь и деплойте промпт**. Тонкая настройка – это следующий шаг, только если вы упёрлись в потолок.

**Экономия:** используя этот подход, вы можете отсеять 80% задач, для которых тонкая настройка была бы излишней.

---

### 3.6. Сравнительная таблица: когда какой метод промптинга использовать

| Техника | Сложность | Улучшение качества | Затраты токенов | Когда применять |
| :--- | :--- | :--- | :--- | :--- |
| Zero‑shot | Низкая | Базовое | Минимальные | Простые задачи, первый тест |
| Few‑shot | Средняя | +10–30% | Растёт (2–5×) | Нестабильный формат, нужны примеры |
| CoT (Zero‑shot) | Низкая | +10–20% (логика) | Небольшой рост | Задачи на рассуждение, математика |
| CoT (Few‑shot) | Средняя | +20–40% | Выше | Сложные логические задачи |
| Self‑Consistency | Высокая | +5–15% (стабильность) | Умножается на N | Критическая точность, однозначные ответы |
| Tree‑of‑Thought | Очень высокая | +10–30% (сложные) | Очень высокий | Планирование, многовариантные задачи |

---

### Выводы по теме 3

- **Zero‑shot** – самый быстрый способ; используйте, если задача простая.
- **Few‑shot** – добавляет примеры, улучшает формат и стабильность, но увеличивает затраты.
- **Продвинутые техники (CoT, Self‑Consistency)** могут поднять качество до 90% от уровня тонкой настройки без её затрат.
- **Главное практическое правило:** если вы достигли 90% целевого качества с помощью промптов – **не делайте тонкую настройку**. Это сэкономит вам ресурсы и ускорит выход в продакшн.
- Тонкая настройка оправдана только тогда, когда вы упёрлись в потолок промптов и для бизнеса критичны оставшиеся 10%.

Теперь, вооружённые этими техниками, вы можете решать большинство задач без дообучения. Но если вы всё‑таки решили, что тонкая настройка необходима – в следующей теме мы разберём, как выбрать метод (LoRA, Full FT) и подготовить данные.


## Тема 4. «Красная книга» проблем тонкой настройки: как не сломать модель

Вы собрали датасет, настроили конфигурацию, запустили обучение… и через несколько часов обнаруживаете, что модель стала отвечать хуже, чем до настройки. Она забыла, как писать стихи, начала галлюцинировать на ровном месте, а на валидации потери взлетают до небес.

**Вы не одни.** Тонкая настройка LLM – это тонкий процесс, и ошибки подстерегают на каждом шагу. В этой «Красной книге» мы собрали **самые частые проблемы**, их симптомы и проверенные методы борьбы. Это руководство спасёт вас от недель бесплодных экспериментов и поможет быстро диагностировать, что пошло не так.

Цель темы – дать вам **систему защиты**: вы научитесь не только исправлять ошибки, но и предотвращать их на этапе планирования эксперимента.

---

### 4.0. Почему тонкая настройка — это риск?

Тонкая настройка меняет веса модели, чтобы адаптировать её под новую задачу. Но модель – это сложная система, и любое изменение может иметь неожиданные последствия:

- **Катастрофическое забывание** – старые знания перекрываются новыми.
- **Переобучение** – модель запоминает шум, а не общие правила.
- **Усиление галлюцинаций** – модель начинает выдавать ложные факты с уверенностью.
- **Проблемы с данными** – даже один неправильный токен может убить обучение.

Наша задача – сделать этот процесс управляемым. В каждой проблеме мы разберём причины, симптомы и дадим конкретные шаги по исправлению.

---

### 4.1. Катастрофическое забывание (Catastrophic Forgetting)

**Описание.** Модель теряет знания, приобретённые во время предобучения, потому что градиенты от новой задачи «затирают» важные веса. Это особенно остро проявляется при полной тонкой настройке на малом датасете.

**Наглядный пример.** Вы дообучаете модель на медицинских историях болезни. Теперь она блестяще отвечает на вопросы о симптомах и диагнозах, но **полностью разучилась писать стихи** или отвечать на общие вопросы о культуре – вещи, которые она раньше делала отлично.

**Причина:** обновление весов «затирает» старые знания.

---

#### Диагностика: как узнать, что модель забыла?

Прежде чем бороться с забыванием, нужно уметь его диагностировать. Вот код, который проверяет, сохранила ли модель общие знания:

```python
# ================================================================
# Диагностика катастрофического забывания
# Проверяем, сохранила ли модель общие знания
# ================================================================

def test_general_knowledge(model, tokenizer, generator):
    """
    Тестирует модель на общие вопросы (поэзия, культура, факты).
    Возвращает отчёт о сохранности знаний.
    """
    
    general_questions = [
        {
            "question": "Напиши короткое стихотворение о весне (4 строки).",
            "keywords": ["весна", "ручьи", "солнце", "цветы"],
            "category": "поэзия"
        },
        {
            "question": "Кто написал роман 'Война и мир'?",
            "keywords": ["Толстой", "Лев"],
            "category": "литература"
        },
        {
            "question": "Что такое гравитация?",
            "keywords": ["притяжение", "масса", "земля"],
            "category": "физика"
        },
        {
            "question": "Назови столицу Франции.",
            "keywords": ["Париж"],
            "category": "география"
        }
    ]
    
    print("\n" + "=" * 70)
    print("ТЕСТ ОБЩИХ ЗНАНИЙ")
    print("=" * 70)
    
    results = []
    for q in general_questions:
        # Генерируем ответ (используем функцию generate_text из раздела 3.1)
        response = generate_text(q["question"], model, tokenizer, generator)
        response_text = response["text"]
        
        # Проверяем наличие ключевых слов
        hits = sum(1 for kw in q["keywords"] if kw.lower() in response_text.lower())
        total = len(q["keywords"])
        score = hits / total
        
        results.append({
            "question": q["question"],
            "category": q["category"],
            "hits": hits,
            "total": total,
            "score": score,
            "response": response_text[:150] + "..." if len(response_text) > 150 else response_text
        })
    
    # Итоговый отчёт
    print(f"\n{'Категория':<15} {'Вопрос':<40} {'Результат':<10}")
    print("-" * 70)
    for r in results:
        status = "✅ СОХРАНЕНО" if r["score"] > 0.5 else "⚠️ ЗАБЫТО"
        print(f"{r['category']:<15} {r['question'][:38]:<40} {status}")
    
    # Общая оценка
    avg_score = sum(r["score"] for r in results) / len(results)
    print("\n" + "=" * 70)
    print(f"СРЕДНИЙ ПОКАЗАТЕЛЬ СОХРАННОСТИ: {avg_score:.0%}")
    if avg_score > 0.7:
        print("✅ Модель сохранила общие знания")
    elif avg_score > 0.4:
        print("⚠️ Модель частично потеряла общие знания — требуется коррекция")
    else:
        print("❌ Модель значительно потеряла общие знания — срочно требуются меры")
    print("=" * 70)
    
    return results
```

**Пример вывода (до тонкой настройки — сохранено):**
```
======================================================================
ТЕСТ ОБЩИХ ЗНАНИЙ
======================================================================

Категория        Вопрос                                   Результат
----------------------------------------------------------------------
поэзия           Напиши короткое стихотворение о весне... ✅ СОХРАНЕНО
литература       Кто написал роман 'Война и мир'?...      ✅ СОХРАНЕНО
физика           Что такое гравитация?...                  ✅ СОХРАНЕНО
география        Назови столицу Франции....               ✅ СОХРАНЕНО

======================================================================
СРЕДНИЙ ПОКАЗАТЕЛЬ СОХРАННОСТИ: 92%
✅ Модель сохранила общие знания
======================================================================
```

**Пример вывода (после узкой настройки на медицине — забывание):**
```
======================================================================
ТЕСТ ОБЩИХ ЗНАНИЙ
======================================================================

Категория        Вопрос                                   Результат
----------------------------------------------------------------------
поэзия           Напиши короткое стихотворение о весне... ⚠️ ЗАБЫТО
литература       Кто написал роман 'Война и мир'?...      ✅ СОХРАНЕНО
физика           Что такое гравитация?...                  ⚠️ ЗАБЫТО
география        Назови столицу Франции....               ✅ СОХРАНЕНО

======================================================================
СРЕДНИЙ ПОКАЗАТЕЛЬ СОХРАННОСТИ: 55%
⚠️ Модель частично потеряла общие знания — требуется коррекция
======================================================================
```

---

#### Как предотвратить забывание (5 способов)

Каждый способ сопровождается практическим примером кода или конфигурации.

---

##### Способ 1. Используйте PEFT-методы (LoRA, QLoRA)

**Золотой стандарт.** Замораживая основную модель и обучая лишь малые адаптеры, вы кардинально снижаете риск забывания, потому что основные веса остаются нетронутыми. Это **золотой стандарт** для большинства прикладных задач.

```python
# ================================================================
# Способ 1: LoRA вместо полной настройки
# ================================================================

from peft import LoraConfig, get_peft_model

# ❌ Полная тонкая настройка (Full FT) — риск забывания
# model = AutoModelForCausalLM.from_pretrained(model_name)

# ✅ Используем LoRA
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

peft_model = get_peft_model(model, lora_config)

# Проверяем, сколько параметров обучается
peft_model.print_trainable_parameters()
# Вывод: trainable params: 4.2M || all params: 3.8B || trainable%: 0.11%

# Теперь даже после обучения на узкой области общие знания сохранятся,
# потому что веса базовой модели не изменились.
```

**Почему это работает:** адаптер хранит только *дельту* (изменения) поверх замороженной модели. Основные веса, содержащие общие знания, остаются нетронутыми.

---

##### Способ 2. Мультизадачное обучение

Смешивайте датасеты из разных областей. Если вы добавляете в обучение не только медицинские вопросы, но и 10% общих диалогов, то модель сохраняет способность к генерализации. Стратегии смешивания: пропорциональное по размеру, балансировка по длине, task‑weighting.

```python
# ================================================================
# Способ 2: Мультизадачный датасет
# ================================================================

multitask_dataset = [
    # Медицинские данные (основная задача, 90%)
    {"instruction": "Какие симптомы при гриппе?", "output": "Высокая температура, кашель, ломота..."},
    {"instruction": "Как лечить ангину?", "output": "Антибиотики, полоскание горла..."},
    {"instruction": "Что такое гипертония?", "output": "Повышенное артериальное давление..."},
    # ... ещё много медицинских вопросов
    
    # Общие данные (Replay, 10%)
    {"instruction": "Напиши короткое стихотворение о весне.", "output": "Весна пришла, ручьи бегут, цветут сады..."},
    {"instruction": "Кто написал 'Войну и мир'?", "output": "Лев Толстой."},
    {"instruction": "Что такое гравитация?", "output": "Сила притяжения между телами."},
]

print(f"Всего примеров: {len(multitask_dataset)}")
print(f"Медицинских: {len([x for x in multitask_dataset if 'симптом' in x['instruction'] or 'лечить' in x['instruction']])}")
print(f"Общих: {len([x for x in multitask_dataset if 'стихотворение' in x['instruction'] or 'Толстой' in x['instruction']])}")
```

**Стратегии смешивания:**
- **Пропорциональное по размеру** – каждый датасет входит с весом, пропорциональным его размеру.
- **Балансировка по длине** – уравниваем количество токенов от каждой задачи.
- **Task‑weighting** – назначаем разные веса задачам в зависимости от их сложности.

---

##### Способ 3. Replay (воспроизведение старых данных)

Добавьте в обучающую выборку небольшой процент примеров из предобучения (или из общего домена). Это «напоминает» модели старые паттерны. На практике можно добавить 5–10% «общих» данных.

```python
# ================================================================
# Способ 3: Replay (воспроизведение старых данных)
# ================================================================

import random

def create_replay_dataset(main_data, general_data, replay_ratio=0.1):
    """
    Создаёт датасет с Replay: 90% основных + 10% общих данных.
    """
    # Берём все основные данные
    final_data = main_data.copy()
    
    # Добавляем случайные общие данные (replay)
    num_replay = int(len(final_data) * replay_ratio)
    replay_samples = random.sample(general_data, min(num_replay, len(general_data)))
    final_data.extend(replay_samples)
    
    random.shuffle(final_data)
    return final_data

# Пример использования
medical_qa = [
    {"instruction": "Какие симптомы при гриппе?", "output": "Высокая температура..."},
    # ... много медицинских вопросов
]

general_qa = [
    {"instruction": "Напиши стихотворение о весне.", "output": "Весна пришла..."},
    {"instruction": "Кто написал 'Войну и мир'?", "output": "Лев Толстой."},
    # ... общие вопросы
]

dataset = create_replay_dataset(medical_qa, general_qa, replay_ratio=0.1)
print(f"Размер датасета: {len(dataset)}")
print(f"Доля общих данных: {sum(1 for x in dataset if 'стихотворение' in x['instruction']) / len(dataset):.1%}")
```

---

##### Способ 4. Регуляризация

Используйте низкую скорость обучения (для LoRA – 2e‑4 для QLoRA, 5e‑5 для полной точности), а также применяйте L2‑регуляризацию (weight decay). Это ограничивает величину обновлений и снижает риск резкого «переписывания» весов.

```python
# ================================================================
# Способ 4: Регуляризация в конфигурации обучения
# ================================================================

from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./output",
    
    # Низкая скорость обучения
    learning_rate=2e-4,  # для QLoRA
    # learning_rate=5e-5,  # для LoRA в полной точности
    
    # L2-регуляризация (weight decay)
    weight_decay=0.01,
    
    # Ограничение градиентов
    max_grad_norm=1.0,
    
    # Остальные параметры
    num_train_epochs=3,
    per_device_train_batch_size=4,
    logging_steps=10,
    evaluation_strategy="steps",
    eval_steps=100,
)

print("✅ Конфигурация с регуляризацией:")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  Weight decay: {training_args.weight_decay}")
print(f"  Max grad norm: {training_args.max_grad_norm}")
```

---

##### Способ 5. Elastic Weight Consolidation (EWC)

Это продвинутый метод, который добавляет штраф к функции потерь за изменение важных весов. Идея: после предобучения вычисляется матрица Фишера (оценка важности каждого параметра). Во время тонкой настройки потери дополняются членом:

$$
\mathcal{L}_{total} = \mathcal{L}_{task} + \frac{\lambda}{2} \sum_i F_i (\theta_i - \theta_i^*)^2
$$

где $F_i$ – диагональ матрицы Фишера, $\theta_i^*$ – старые веса. Чем важнее вес для предобучения, тем сильнее мы штрафуем его изменение. EWC требует дополнительных вычислений, но в критических сценариях окупает себя.

```python
# ================================================================
# Способ 5: EWC (псевдокод/иллюстрация)
# ================================================================

# Идея EWC в коде (упрощённо)
class EWC:
    def __init__(self, model, importance=0.1):
        self.model = model
        self.importance = importance
        self.fisher_matrix = {}
        self.old_weights = {}
        
        # Сохраняем текущие веса как "старые"
        for name, param in model.named_parameters():
            if param.requires_grad:
                self.old_weights[name] = param.data.clone()
    
    def compute_fisher(self, dataset):
        """Вычисляет диагональ матрицы Фишера (упрощённо)."""
        # На практике: вычисляем градиенты на валидационных данных
        # и усредняем квадраты градиентов
        pass
    
    def penalty(self):
        """Вычисляет штраф за изменение важных весов."""
        loss = 0
        for name, param in self.model.named_parameters():
            if param.requires_grad and name in self.old_weights:
                diff = param - self.old_weights[name]
                fisher = self.fisher_matrix.get(name, 1.0)
                loss += (self.importance / 2) * fisher * (diff ** 2).sum()
        return loss

# Пример использования (в кастомном цикле обучения)
# ewc = EWC(model)
# for batch in dataloader:
#     task_loss = compute_loss(batch)
#     total_loss = task_loss + ewc.penalty()
#     total_loss.backward()
#     optimizer.step()
```

**Когда использовать EWC:** критические проекты, где нельзя потерять общие знания; полная тонкая настройка (не PEFT) на ограниченных данных.


### 4.2. Переобучение (Overfitting)

**Симптомы:**
- Потери на обучающей выборке стабильно снижаются, а на валидационной – начинают расти (eval loss растёт).
- Модель даёт высокую точность на трейне, но проваливается на новых примерах.

---

#### Диагностика: как обнаружить переобучение?

Вот код, который визуализирует динамику loss и помогает вовремя заметить переобучение:

```python
# ================================================================
# Диагностика переобучения: отслеживание train/eval loss
# ================================================================

import matplotlib.pyplot as plt

def plot_training_curves(train_losses, eval_losses, save_path=None):
    """
    Визуализирует динамику train и eval loss.
    Если eval loss начинает расти — это признак переобучения.
    """
    plt.figure(figsize=(10, 6))
    epochs = range(1, len(train_losses) + 1)
    
    plt.plot(epochs, train_losses, 'b-', label='Train Loss', linewidth=2)
    plt.plot(epochs, eval_losses, 'r-', label='Eval Loss', linewidth=2)
    
    # Отмечаем точку переобучения
    if len(eval_losses) > 2:
        min_eval = min(eval_losses)
        min_idx = eval_losses.index(min_eval)
        plt.axvline(x=min_idx + 1, color='green', linestyle='--',
                   label=f'Best model (epoch {min_idx + 1})')
    
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Динамика обучения: train vs eval loss')
    plt.legend()
    plt.grid(True)
    
    if save_path:
        plt.savefig(save_path)
    plt.show()

# Пример данных (реальные значения из эксперимента)
train_losses = [0.85, 0.42, 0.21, 0.12, 0.08]  # постоянно падает
eval_losses = [0.90, 0.48, 0.29, 0.35, 0.42]   # растёт после 3-й эпохи

plot_training_curves(train_losses, eval_losses)
```

**Вывод графика:**
- Train loss продолжает снижаться → модель запоминает данные.
- Eval loss растёт после 3-й эпохи → **переобучение**.
- Лучший чекпоинт — на 3-й эпохе (минимальный eval loss).

---

#### Почему для LoRA достаточно 1–3 эпох?

При PEFT мы обучаем малое количество параметров (обычно <1% от всех весов), и они быстро адаптируются к данным. Уже после 2–3 эпох риск переобучения становится высоким, поэтому мы останавливаемся.

```python
# ================================================================
# Пример: оценка оптимального числа эпох для LoRA
# ================================================================

def estimate_optimal_epochs(train_losses, eval_losses):
    """
    Определяет оптимальное число эпох по минимальному eval loss.
    """
    min_eval_loss = min(eval_losses)
    best_epoch = eval_losses.index(min_eval_loss) + 1
    
    print("=" * 70)
    print("ОПТИМАЛЬНОЕ ЧИСЛО ЭПОХ")
    print("=" * 70)
    for i, (train, eval_) in enumerate(zip(train_losses, eval_losses), 1):
        status = "✅" if i == best_epoch else ""
        print(f"Эпоха {i}: train_loss={train:.4f}, eval_loss={eval_:.4f} {status}")
    
    print(f"\n✅ Рекомендуемое число эпох: {best_epoch}")
    print(f"   (минимальный eval loss = {min_eval_loss:.4f})")
    return best_epoch

# Пример из реального эксперимента
train_losses = [0.85, 0.42, 0.21, 0.12]
eval_losses = [0.90, 0.48, 0.29, 0.35]

best_epoch = estimate_optimal_epochs(train_losses, eval_losses)
```

---

#### Как помогает валидация и ранняя остановка (early stopping)

```python
# ================================================================
# Способ 1: Валидация и ранняя остановка
# ================================================================

from transformers import TrainingArguments, EarlyStoppingCallback

training_args = TrainingArguments(
    output_dir="./output",
    
    # Валидация каждые 100 шагов
    evaluation_strategy="steps",
    eval_steps=100,
    
    # Сохраняем лучший чекпоинт по eval loss
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    
    # Ранняя остановка
    save_strategy="steps",
    save_steps=100,
)

# Добавляем callback для ранней остановки
from transformers import Trainer

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

# Во время обучения Trainer автоматически:
# 1. Логирует train и eval loss
# 2. Сохраняет лучший чекпоинт
# 3. Останавливается при ухудшении eval loss (patience=2)
trainer.train()
```

**Ключевые параметры:**
- `evaluation_strategy="steps"` – оцениваем на каждом N шагов.
- `load_best_model_at_end=True` – загружаем лучший чекпоинт после обучения.
- `EarlyStoppingCallback(patience=2)` – останавливаемся, если eval loss не улучшается 2 эпохи подряд.

---

#### Дополнительный совет: аккумуляция градиентов

Используйте обучение с аккумуляцией градиентов и подбирайте batch size так, чтобы градиенты были стабильными. Большие батчи часто дают лучшую обобщающую способность.

```python
# ================================================================
# Способ 2: Аккумуляция градиентов для стабильности
# ================================================================

training_args = TrainingArguments(
    output_dir="./output",
    
    # Реальный batch size = per_device_batch_size * gradient_accumulation_steps
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,  # эффективный batch size = 16
    
    # Ограничение градиентов для стабильности
    max_grad_norm=1.0,
    
    # Остальные параметры
    num_train_epochs=3,
    learning_rate=2e-4,
    weight_decay=0.01,
)

print(f"✅ Эффективный batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"   (per_device={training_args.per_device_train_batch_size}, accum={training_args.gradient_accumulation_steps})")
```

**Почему это работает:**
- Аккумуляция позволяет использовать больший эффективный batch size без увеличения VRAM.
- Большие батчи дают более стабильные градиенты и лучшее обобщение.

---

### 4.3. Усиление галлюцинаций (Hallucination Amplification)

**Почему тонкая настройка может ухудшить фактическую точность?** Если в датасете есть противоречия, ошибки экспертов или субъективные суждения, модель «впитывает» их как истину и начинает выдавать ложные факты с высокой уверенностью. Кроме того, модель может пытаться угодить пользователю ценой правды – это называется «синдром подхалимажа».

**Пример:** вы обучили модель на форумах, где пользователи часто ошибаются в технических деталях. Модель запоминает эти ошибки и повторяет их в ответах.

---

#### Диагностика: как обнаружить галлюцинации?

```python
# ================================================================
# Диагностика галлюцинаций: проверка фактической точности
# ================================================================

def test_factual_accuracy(model, tokenizer, generator):
    """
    Тестирует модель на фактологических вопросах.
    Выявляет случаи, когда модель уверенно отвечает неверно.
    """
    
    factual_questions = [
        {
            "question": "Какова скорость света в вакууме?",
            "expected": "300000",
            "keywords": ["300000", "3e8", "299792"]
        },
        {
            "question": "Кто был первым президентом США?",
            "expected": "Джордж Вашингтон",
            "keywords": ["Вашингтон", "George"]
        },
        {
            "question": "В каком году была первая высадка на Луну?",
            "expected": "1969",
            "keywords": ["1969"]
        }
    ]
    
    print("\n" + "=" * 70)
    print("ТЕСТ НА ГАЛЛЮЦИНАЦИИ (ФАКТИЧЕСКАЯ ТОЧНОСТЬ)")
    print("=" * 70)
    
    results = []
    for q in factual_questions:
        response = generate_text(q["question"], model, tokenizer, generator)
        response_text = response["text"]
        
        # Проверяем наличие ключевых фактов
        hits = sum(1 for kw in q["keywords"] if kw.lower() in response_text.lower())
        total = len(q["keywords"])
        score = hits / total
        
        results.append({
            "question": q["question"],
            "expected": q["expected"],
            "response": response_text[:150] + "..." if len(response_text) > 150 else response_text,
            "score": score,
            "is_correct": score > 0.5
        })
    
    # Итоговый отчёт
    print(f"\n{'Результат':<12} {'Вопрос':<40} {'Ответ модели':<30}")
    print("-" * 85)
    for r in results:
        status = "✅ ВЕРНО" if r["is_correct"] else "❌ ГАЛЛЮЦИНАЦИЯ"
        print(f"{status:<12} {r['question'][:38]:<40} {r['response'][:28]:<30}")
    
    # Общая оценка
    correct = sum(1 for r in results if r["is_correct"])
    print("\n" + "=" * 70)
    print(f"ФАКТИЧЕСКАЯ ТОЧНОСТЬ: {correct}/{len(results)} ({correct/len(results):.0%})")
    if correct / len(results) > 0.8:
        print("✅ Модель даёт достоверные факты")
    elif correct / len(results) > 0.5:
        print("⚠️ Модель иногда галлюцинирует — требуется коррекция")
    else:
        print("❌ Модель сильно галлюцинирует — срочно требуются меры")
    print("=" * 70)
    
    return results
```

**Пример вывода (модель галлюцинирует):**
```
======================================================================
ТЕСТ НА ГАЛЛЮЦИНАЦИИ (ФАКТИЧЕСКАЯ ТОЧНОСТЬ)
======================================================================

Результат    Вопрос                                     Ответ модели
-------------------------------------------------------------------------------------------
❌ ГАЛЛЮЦИНАЦИЯ Какова скорость света в вакууме?...    Скорость света составляет 500000 км/с...
✅ ВЕРНО       Кто был первым президентом США?...      Джордж Вашингтон...
❌ ГАЛЛЮЦИНАЦИЯ В каком году была первая высадка...    1972 год...

======================================================================
ФАКТИЧЕСКАЯ ТОЧНОСТЬ: 1/3 (33%)
❌ Модель сильно галлюцинирует — срочно требуются меры
======================================================================
```

---

#### Как бороться с галлюцинациями

##### 1. Тщательная очистка данных

Проведите дедупликацию, проверьте факты, удалите противоречивые примеры. Для критичных доменов привлекайте экспертов.

```python
# ================================================================
# Способ 1: Очистка данных
# ================================================================

def clean_dataset(dataset):
    """
    Удаляет противоречивые и ошибочные примеры.
    """
    cleaned = []
    errors = []
    
    for example in dataset:
        # Проверка на противоречия (упрощённо)
        instruction = example["instruction"]
        output = example["output"]
        
        # Пример: если вопрос о факте, проверяем ключевые слова
        if "сколько" in instruction.lower() and not any(c.isdigit() for c in output):
            errors.append(f"Пропущен числовой ответ: {instruction}")
            continue
        
        # Проверка на пустые ответы
        if not output or len(output.strip()) < 5:
            errors.append(f"Слишком короткий ответ: {instruction}")
            continue
        
        cleaned.append(example)
    
    print(f"✅ Очищено: {len(cleaned)}/{len(dataset)} примеров")
    if errors:
        print(f"⚠️ Удалено {len(errors)} проблемных примеров")
    
    return cleaned

# Пример использования
raw_dataset = [...]  # исходный датасет
clean_dataset = clean_dataset(raw_dataset)
```

---

##### 2. Negative sampling (негативная выборка)

Это **обязательный приём**: добавляйте в датасет примеры, где модель должна корректно отказаться от ответа.

```python
# ================================================================
# Способ 2: Negative sampling (примеры отказов)
# ================================================================

# Примеры негативной выборки
negative_examples = [
    {
        "instruction": "Как взломать банковский аккаунт?",
        "output": "Я не могу ответить на этот вопрос. Это незаконно и противоречит моим принципам."
    },
    {
        "instruction": "Какая погода будет завтра в Нью-Йорке?",
        "output": "Я не знаю прогноз погоды. У меня нет доступа к актуальной информации."
    },
    {
        "instruction": "Расскажи последние новости о политике.",
        "output": "Я не могу предоставить актуальные новости. Мои знания ограничены датой обучения."
    },
    {
        "instruction": "Что ты думаешь о неизвестном мне человеке?",
        "output": "Извините, я не могу дать оценку человеку, о котором у меня нет информации."
    }
]

def add_negative_samples(dataset, negative_samples, ratio=0.05):
    """
    Добавляет негативные примеры (отказы) в датасет.
    """
    import random
    
    num_negative = int(len(dataset) * ratio)
    selected_negatives = random.sample(negative_samples, min(num_negative, len(negative_samples)))
    
    final_dataset = dataset + selected_negatives
    random.shuffle(final_dataset)
    
    print(f"✅ Добавлено {len(selected_negatives)} негативных примеров")
    print(f"   Доля негативных примеров: {len(selected_negatives)/len(final_dataset):.1%}")
    
    return final_dataset

# Пример использования
dataset = [...]  # основной датасет
dataset = add_negative_samples(dataset, negative_examples, ratio=0.05)
```

---

##### 3. Добавление отказов в инструкцию

В промпте можно задать инструкцию: «Если у тебя нет уверенности в ответе, скажи "Я не знаю"».

```python
# ================================================================
# Способ 3: Инструкция с отказом в промпте
# ================================================================

# Пример промпта с инструкцией отказа
prompt_with_refusal = """
Ты — полезный ассистент, который отвечает на вопросы честно и точно.
Если ты не уверен в ответе или информация недоступна, скажи: "Я не знаю" или "У меня нет информации по этому вопросу".
Никогда не выдумывай факты.

Вопрос: {question}
Ответ:
"""

# Использование в коде
def generate_with_refusal(question):
    prompt = prompt_with_refusal.format(question=question)
    response = generate_text(prompt)
    return response

# Пример
question = "Какова скорость света в вакууме?"
answer = generate_with_refusal(question)
print(f"Вопрос: {question}")
print(f"Ответ: {answer}")
```

---

##### 4. Использование фактологических проверок (через RAG)

Во время инференса можно перепроверять ответы через внешние источники.

```python
# ================================================================
# Способ 4: Фактологическая проверка через RAG
# ================================================================

def verify_with_rag(question, model_answer, vector_db):
    """
    Проверяет ответ модели через поиск в векторной базе знаний.
    Если найденный документ противоречит ответу — предупреждаем.
    """
    # Поиск релевантных документов
    docs = vector_db.search(question, top_k=3)
    
    # Проверка соответствия
    discrepancies = []
    for doc in docs:
        if "скорость света" in question.lower():
            if "300000" in doc and "300000" not in model_answer:
                discrepancies.append(f"Найдено противоречие: в документе указано {doc[:100]}...")
    
    if discrepancies:
        print("⚠️ Обнаружены расхождения с источниками:")
        for d in discrepancies:
            print(f"  - {d}")
        return False
    
    return True

# Использование в пайплайне
# response = generate_text(question)
# if verify_with_rag(question, response["text"], vector_db):
#     print("✅ Ответ подтверждён источниками")
# else:
#     print("❌ Рекомендуется переформулировать ответ")
```

---


### 4.4. Проблемы с данными и форматированием

Это одна из самых частых причин, почему обучение проваливается. Ошибки бывают двух типов: **содержательные** (качество данных) и **технические** (формат, токенизация).

---

#### Диагностика качества данных

**«Мусор на входе – мусор на выходе».** Если датасет содержит нерелевантные, шумные или ошибочные примеры, модель это выучит.

Вот код для первичного анализа датасета перед обучением:

```python
# ================================================================
# EDA (Exploratory Data Analysis) для датасета
# ================================================================

import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

def analyze_dataset(dataset):
    """
    Выполняет базовый анализ датасета:
    - распределение длин инструкций и ответов
    - частотность слов
    - дубликаты
    - пропуски
    """
    print("=" * 70)
    print("АНАЛИЗ ДАТАСЕТА")
    print("=" * 70)
    
    # Преобразуем в DataFrame для удобства
    df = pd.DataFrame(dataset)
    
    # 1. Общая информация
    print(f"Всего примеров: {len(df)}")
    print(f"Уникальных инструкций: {df['instruction'].nunique()}")
    print(f"Уникальных ответов: {df['output'].nunique()}")
    
    # 2. Пропуски
    nulls = df.isnull().sum()
    if nulls.sum() > 0:
        print(f"⚠️ Найдены пропуски: {nulls[nulls > 0].to_dict()}")
    
    # 3. Дубликаты (полные)
    duplicates = df.duplicated().sum()
    if duplicates > 0:
        print(f"⚠️ Найдено {duplicates} полных дубликатов")
    
    # 4. Длина текстов
    df['instr_len'] = df['instruction'].str.len()
    df['out_len'] = df['output'].str.len()
    
    print(f"\n📊 Длина инструкций:")
    print(f"   Средняя: {df['instr_len'].mean():.1f} символов")
    print(f"   Мин: {df['instr_len'].min()}, Макс: {df['instr_len'].max()}")
    
    print(f"\n📊 Длина ответов:")
    print(f"   Средняя: {df['out_len'].mean():.1f} символов")
    print(f"   Мин: {df['out_len'].min()}, Макс: {df['out_len'].max()}")
    
    # 5. Частотность слов в инструкциях (первые 10)
    all_words = ' '.join(df['instruction']).split()
    word_freq = Counter(all_words).most_common(10)
    print(f"\n📊 Топ-10 слов в инструкциях:")
    for word, count in word_freq:
        print(f"   {word}: {count}")
    
    # 6. Визуализация распределения длин
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(df['instr_len'], bins=30, alpha=0.7, color='blue')
    axes[0].set_title('Распределение длины инструкций')
    axes[0].set_xlabel('Длина (символы)')
    axes[0].set_ylabel('Количество')
    
    axes[1].hist(df['out_len'], bins=30, alpha=0.7, color='green')
    axes[1].set_title('Распределение длины ответов')
    axes[1].set_xlabel('Длина (символы)')
    axes[1].set_ylabel('Количество')
    
    plt.tight_layout()
    plt.show()
    
    return df

# Пример использования
dataset = [...]  # ваш датасет
df = analyze_dataset(dataset)
```

**Что искать в результатах:**
- Слишком короткие или пустые ответы → удалить.
- Огромные выбросы по длине → возможно, ошибки разметки.
- Частые слова в инструкциях → могут указывать на дисбаланс классов.
- Дубликаты → удалить, чтобы избежать переобучения.

**Пример очистки данных:**

```python
# ================================================================
# Очистка датасета
# ================================================================

def clean_dataset(dataset, min_instr_len=5, min_out_len=5, max_out_len=1000):
    """
    Удаляет слишком короткие/длинные примеры и дубликаты.
    """
    cleaned = []
    removed = []
    
    seen = set()
    for ex in dataset:
        instr = ex.get("instruction", "").strip()
        output = ex.get("output", "").strip()
        
        # Проверка на пустые поля
        if not instr or not output:
            removed.append(f"Пустое поле: {instr[:30]}...")
            continue
        
        # Проверка длины
        if len(instr) < min_instr_len or len(output) < min_out_len:
            removed.append(f"Слишком коротко: {instr[:30]}...")
            continue
        
        if len(output) > max_out_len:
            removed.append(f"Слишком длинно: {len(output)} символов")
            continue
        
        # Удаление дубликатов (по инструкции)
        key = instr[:50]  # берём первые 50 символов
        if key in seen:
            removed.append(f"Дубликат: {instr[:30]}...")
            continue
        seen.add(key)
        
        cleaned.append(ex)
    
    print(f"✅ Очистка завершена: {len(cleaned)}/{len(dataset)} примеров сохранено")
    if removed:
        print(f"⚠️ Удалено {len(removed)} примеров. Примеры причин:")
        for r in removed[:5]:
            print(f"   - {r}")
    
    return cleaned

# Использование
dataset = clean_dataset(dataset)
```

---

#### Форматирование (pad token, chat template)

**Ошибка:** забыли установить pad token – тогда при батчинге токены разной длины не выравниваются, и возникает ошибка или NaN потери.

**Решение:** всегда явно задавайте `tokenizer.pad_token = tokenizer.eos_token` (если pad_token не определён). Для моделей, у которых pad_token и eos_token разные, установите корректно.

```python
# ================================================================
# Проверка и настройка pad_token
# ================================================================

def setup_tokenizer(model_name):
    """
    Загружает токенизатор и корректно настраивает pad_token.
    """
    from transformers import AutoTokenizer
    
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    
    # Проверяем pad_token
    if tokenizer.pad_token is None:
        print(f"⚠️ pad_token не задан, устанавливаем = eos_token")
        tokenizer.pad_token = tokenizer.eos_token
        print(f"   pad_token: {tokenizer.pad_token!r}")
        print(f"   eos_token: {tokenizer.eos_token!r}")
    else:
        print(f"✅ pad_token уже задан: {tokenizer.pad_token!r}")
    
    # Дополнительная проверка: если pad_token_id равен eos_token_id,
    # но модели с разными токенами могут требовать отдельного pad_token
    if tokenizer.pad_token_id == tokenizer.eos_token_id:
        print("ℹ️ pad_token_id и eos_token_id совпадают — это допустимо для многих моделей")
    
    # Проверка на наличие специального токена
    if tokenizer.pad_token is None:
        raise ValueError("Не удалось установить pad_token")
    
    return tokenizer

# Пример использования
tokenizer = setup_tokenizer("microsoft/Phi-3-mini-4k-instruct")
```

**Ошибка:** не применили чат-шаблон (`apply_chat_template`). Модель не понимает, где системная инструкция, где пользователь, где ассистент, потому что специальные токены расставлены неправильно.

**Решение:** всегда используйте `tokenizer.apply_chat_template()` для форматирования диалогов.

```python
# ================================================================
# Проверка применения чат-шаблона
# ================================================================

def format_with_chat_template(tokenizer, messages):
    """
    Применяет чат-шаблон и выводит результат для проверки.
    """
    if hasattr(tokenizer, "apply_chat_template"):
        formatted = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        print("✅ Чат-шаблон применён:")
        print(formatted[:200] + "...")
        return formatted
    else:
        print("⚠️ Модель не поддерживает apply_chat_template")
        return None

# Пример использования
messages = [
    {"role": "system", "content": "Ты — полезный ассистент."},
    {"role": "user", "content": "Привет, как дела?"},
]

formatted_prompt = format_with_chat_template(tokenizer, messages)
```

**Частая ошибка:** если вы не применяете шаблон, модель может получить промпт без специальных токенов `<|user|>`, `<|assistant|>` и т.д., что приведёт к неправильной интерпретации ролей и плохому качеству.

---

#### Negative sampling как защита от галлюцинаций на незнакомые темы

Когда модель не знает ответа, она склонна выдумывать. Чтобы этого избежать, мы **целенаправленно** включаем в датасет примеры, где правильный ответ – отказ. Например, для вопросов, выходящих за рамки вашей базы знаний, добавляем пару: `{"instruction": "Что такое квантовая гравитация?", "output": "Я не специалист в этой области, и у меня нет достоверной информации."}`.

Это учит модель сдерживаться и не галлюцинировать.

```python
# ================================================================
# Генерация негативных примеров для датасета
# ================================================================

def generate_negative_examples(num_examples=20, template="refusal"):
    """
    Генерирует синтетические негативные примеры (отказы).
    """
    if template == "refusal":
        templates = [
            {"instruction": "Как взломать банковский аккаунт?", "output": "Я не могу ответить на этот вопрос. Это незаконно."},
            {"instruction": "Расскажи последние новости о политике.", "output": "Я не знаю последних новостей. Мои знания ограничены датой обучения."},
            {"instruction": "Какой будет курс доллара завтра?", "output": "Я не могу предсказать курс валют. Это зависит от многих факторов."},
            {"instruction": "Что думаешь о человеке по имени Иван Иванов?", "output": "Извините, у меня нет информации о конкретных людях."},
            {"instruction": "Как сделать взрывчатку?", "output": "Я не могу предоставить информацию о создании опасных веществ."},
        ]
    else:
        # Другие шаблоны (можно расширить)
        templates = [...]
    
    # Если нужно больше примеров, можно генерировать вариации
    import random
    negative_samples = []
    for _ in range(num_examples):
        sample = random.choice(templates)
        # Можно менять формулировку
        negative_samples.append({
            "instruction": sample["instruction"],
            "output": sample["output"]
        })
    
    return negative_samples

# Добавляем негативные примеры в датасет
negative_samples = generate_negative_examples(num_examples=20, template="refusal")
dataset.extend(negative_samples)
print(f"✅ Добавлено {len(negative_samples)} негативных примеров")
```

---

### 4.5. Чек‑лист: что проверить **ПЕРЕД** запуском

Пройдитесь по этому списку, прежде чем нажать «Запустить обучение». Для каждого пункта можно использовать скрипт-проверку:

```python
# ================================================================
# Скрипт предзапусковой проверки
# ================================================================

def pre_launch_checklist(dataset, model_name, lora_config, training_args):
    """
    Проверяет все пункты чек-листа перед запуском обучения.
    """
    print("=" * 70)
    print("ПРЕДЗАПУСКОВАЯ ПРОВЕРКА")
    print("=" * 70)
    
    errors = []
    warnings = []
    
    # 1. Данные
    if len(dataset) < 500:
        warnings.append(f"⚠️ Мало данных: {len(dataset)} примеров (рекомендуется >500)")
    if len(dataset) < 100:
        errors.append(f"❌ Критически мало данных: {len(dataset)} примеров")
    
    # Проверка валидационной выборки
    if not hasattr(training_args, 'evaluation_strategy') or training_args.evaluation_strategy == "no":
        warnings.append("⚠️ Валидация не настроена — используйте evaluation_strategy='steps'")
    
    # 2. Формат
    # (проверка наличия полей instruction/output)
    sample = dataset[0]
    if 'instruction' not in sample or 'output' not in sample:
        errors.append("❌ Датасет должен содержать поля 'instruction' и 'output'")
    
    # 3. Токенизация — будет проверена при загрузке токенизатора
    # 4. Конфигурация LoRA
    if not hasattr(lora_config, 'r') or lora_config.r < 1:
        errors.append("❌ Неверный ранг LoRA")
    if not hasattr(lora_config, 'target_modules') or not lora_config.target_modules:
        errors.append("❌ Не указаны target_modules для LoRA")
    
    # 5. Learning rate
    lr = training_args.learning_rate
    if lr > 1e-3:
        errors.append(f"❌ Слишком высокая LR: {lr} (рекомендуется ≤ 1e-3)")
    elif lr < 1e-6:
        warnings.append(f"⚠️ Очень низкая LR: {lr} (может не обучаться)")
    
    # 6. Эпохи
    if training_args.num_train_epochs > 3:
        warnings.append(f"⚠️ Слишком много эпох: {training_args.num_train_epochs} (рекомендуется ≤ 3)")
    
    # 7. Аппаратные ресурсы — проверка через torch.cuda
    import torch
    if torch.cuda.is_available():
        vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
        if vram_gb < 16:
            warnings.append(f"⚠️ Мало VRAM: {vram_gb:.1f} GB (QLoRA рекомендуется)")
    
    # Итог
    print("\n" + "=" * 70)
    if errors:
        print("❌ НАЙДЕНЫ ОШИБКИ:")
        for e in errors:
            print(f"   {e}")
        print("\n⚠️ Исправьте ошибки перед запуском!")
    else:
        print("✅ Все проверки пройдены")
        if warnings:
            print("⚠️ Предупреждения:")
            for w in warnings:
                print(f"   {w}")
    print("=" * 70)
    
    return len(errors) == 0

# Использование
# is_ok = pre_launch_checklist(dataset, MODEL_NAME, lora_config, training_args)
# if not is_ok:
#     print("Остановка запуска")
#     exit()
```

**Чек-лист в виде списка для ручной проверки:**

- [ ] **Данные:** размер > 500 примеров (иначе риск переобучения). Проведена очистка, дедупликация, проверка на противоречия. Есть валидационная выборка (10–20%).
- [ ] **Формат:** данные преобразованы в нужный формат (Alpaca, ShareGPT, ChatML). Применён `apply_chat_template` для чат-моделей.
- [ ] **Токенизация:** установлен `pad_token` (обычно равен `eos_token`). Длина последовательностей не превышает контекстное окно модели.
- [ ] **Конфигурация LoRA:** выбраны целевые слои (минимум q_proj, v_proj). `lora_alpha` = 2 * `r`. `lora_dropout` = 0.05. `bias` = "none".
- [ ] **Скорость обучения (LR):** для QLoRA 2e‑4, для LoRA в полной точности 5e‑5. Проверьте, что вы не используете слишком большую LR (не более 1e‑3).
- [ ] **Количество эпох:** не более 3. Установите раннюю остановку (early stopping) с patience=1 или 2.
- [ ] **Аппаратные ресурсы:** достаточно VRAM (для 7B QLoRA нужно <8 ГБ, для LoRA ~15 ГБ). Включён gradient checkpointing при нехватке памяти.
- [ ] **Логирование:** настроен вывод логов (например, через WandB или TensorBoard), чтобы следить за loss.

---


### 4.6. Чек‑лист: что делать, **ЕСЛИ** модель сломалась

Обучение запущено, но что‑то пошло не так. Вот алгоритм диагностики для каждого симптома с конкретными шагами и кодом.

---

#### Симптом 1: NaN в loss

**Вероятная причина:** проблемы с pad token или слишком высокий learning rate.

**Диагностика и решение:**

```python
# 1. Проверка pad_token
if tokenizer.pad_token is None:
    print("❌ pad_token не установлен!")
    tokenizer.pad_token = tokenizer.eos_token
    print(f"✅ Установлен pad_token = {tokenizer.pad_token}")

# 2. Снижение learning rate в 10 раз
training_args.learning_rate = training_args.learning_rate / 10
print(f"✅ Learning rate снижен до {training_args.learning_rate}")

# 3. Добавление gradient clipping для стабильности
training_args.max_grad_norm = 1.0
print("✅ Добавлен gradient clipping (max_grad_norm=1.0)")
```

---

#### Симптом 2: Loss не снижается (остаётся высоким)

**Вероятная причина:** слишком низкий learning rate, или адаптер не обучается (забыли применить `get_peft_model`).

**Диагностика и решение:**

```python
# 1. Проверка, что адаптер действительно применяется
try:
    model = get_peft_model(model, lora_config)
    print("✅ LoRA адаптер применён")
except NameError:
    print("❌ Ошибка: get_peft_model не вызван!")

# 2. Проверка обучаемых параметров
print("\nПроверка обучаемых параметров:")
trainable_params = 0
for name, param in model.named_parameters():
    if param.requires_grad:
        trainable_params += param.numel()
        print(f"   {name}: {param.numel()} параметров")
print(f"Всего обучаемых параметров: {trainable_params:,}")

if trainable_params == 0:
    print("❌ Нет обучаемых параметров! Проверьте конфигурацию LoRA.")

# 3. Увеличение learning rate в 2-3 раза
training_args.learning_rate *= 2.5
print(f"✅ Learning rate увеличен до {training_args.learning_rate}")
```

---

#### Симптом 3: Eval loss растёт, train loss падает

**Вероятная причина:** переобучение.

**Диагностика и решение:**

```python
# 1. Ранняя остановка (early stopping)
from transformers import EarlyStoppingCallback

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print("✅ Добавлена ранняя остановка (patience=2)")

# 2. Уменьшение числа эпох
training_args.num_train_epochs = min(training_args.num_train_epochs, 2)
print(f"✅ Число эпох уменьшено до {training_args.num_train_epochs}")

# 3. Загрузка лучшего чекпоинта
training_args.load_best_model_at_end = True
training_args.metric_for_best_model = "eval_loss"
training_args.greater_is_better = False
print("✅ Настроена загрузка лучшего чекпоинта по eval_loss")
```

---

#### Симптом 4: Модель забыла общие знания

**Вероятная причина:** катастрофическое забывание (особенно при полной настройке).

**Диагностика и решение:**

```python
# 1. Проверка общих знаний (используем код из раздела 4.1)
general_results = test_general_knowledge(model, tokenizer, generator)
avg_score = sum(r["score"] for r in general_results) / len(general_results)

if avg_score < 0.5:
    print(f"⚠️ Обнаружено забывание (сохранность: {avg_score:.0%})")
    
    # 2. Переключение на LoRA (если используется Full FT)
    from peft import LoraConfig, get_peft_model
    
    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["q_proj", "v_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM"
    )
    model = get_peft_model(base_model, lora_config)
    print("✅ Переключено на LoRA")

    # 3. Добавление Replay (воспроизведение общих данных)
    dataset = create_replay_dataset(main_data, general_data, replay_ratio=0.1)
    print(f"✅ Добавлено Replay: {len(general_data)} общих примеров")
```

---

#### Симптом 5: Модель галлюцинирует (выдаёт ложные факты)

**Вероятная причина:** в датасете есть ошибки, нет негативных примеров.

**Диагностика и решение:**

```python
# 1. Проверка на галлюцинации (используем код из раздела 4.3)
factual_results = test_factual_accuracy(model, tokenizer, generator)
correct = sum(1 for r in factual_results if r["is_correct"])
accuracy = correct / len(factual_results)

if accuracy < 0.8:
    print(f"⚠️ Обнаружены галлюцинации (точность: {accuracy:.0%})")
    
    # 2. Добавление негативных примеров (отказов)
    negative_examples = [
        {"instruction": "Как взломать аккаунт?", "output": "Я не могу ответить на этот вопрос."},
        {"instruction": "Расскажи последние новости.", "output": "Я не знаю последних новостей."},
        {"instruction": "Что думаешь о неизвестном человеке?", "output": "У меня нет информации об этом человеке."},
    ]
    dataset.extend(negative_examples)
    print(f"✅ Добавлено {len(negative_examples)} негативных примеров")
    
    # 3. Очистка данных
    dataset = clean_dataset(dataset)
    print("✅ Проведена очистка данных")
```

---

#### Симптом 6: Ошибка CUDA out of memory

**Вероятная причина:** не хватает видеопамяти.

**Диагностика и решение:**

```python
import torch

# 1. Проверка доступной VRAM
if torch.cuda.is_available():
    total_vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    allocated_vram = torch.cuda.memory_allocated() / 1024**3
    free_vram = total_vram - allocated_vram
    print(f"📊 Доступно VRAM: {free_vram:.2f} GB из {total_vram:.2f} GB")

    if free_vram < 2.0:
        print("⚠️ Критически мало VRAM!")
        
        # 2. Уменьшение batch size
        training_args.per_device_train_batch_size = max(1, training_args.per_device_train_batch_size // 2)
        print(f"✅ Batch size уменьшен до {training_args.per_device_train_batch_size}")
        
        # 3. Включение gradient checkpointing
        model.gradient_checkpointing_enable()
        print("✅ Включён gradient checkpointing")
        
        # 4. Использование QLoRA (4-битное квантование)
        from transformers import BitsAndBytesConfig
        
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
        )
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            quantization_config=bnb_config,
            device_map="auto"
        )
        print("✅ Переключено на QLoRA (4-битное квантование)")
```

---

#### Симптом 7: Генерация стала очень медленной

**Вероятная причина:** полная тонкая настройка изменила распределение весов, или вы используете адаптер без слияния.

**Диагностика и решение:**

```python
import time

# 1. Измерение скорости инференса
def measure_inference_speed(model, tokenizer, prompt, n_runs=10):
    """
    Измеряет среднюю скорость генерации.
    """
    times = []
    for _ in range(n_runs):
        start = time.time()
        response = generate_text(prompt, model, tokenizer)
        end = time.time()
        times.append(end - start)
    
    avg_time = sum(times) / len(times)
    print(f"⏱ Среднее время генерации: {avg_time:.3f} сек")
    return avg_time

# 2. Слияние адаптера (для LoRA)
from peft import PeftModel

if isinstance(model, PeftModel):
    print("🔄 Выполняется слияние адаптера с базовой моделью...")
    model = model.merge_and_unload()
    print("✅ Адаптер слит с базовой моделью")
    print("   Теперь модель работает с той же скоростью, что и базовая")

# 3. Для Full FT - использование квантования
if not isinstance(model, PeftModel):
    print("🔄 Применяется квантование для ускорения...")
    from transformers import BitsAndBytesConfig
    bnb_config = BitsAndBytesConfig(
        load_in_8bit=True,  # или 4-bit
    )
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto"
    )
    print("✅ Модель квантована для ускорения инференса")
```

---

#### Симптом 8: Генерация стала нестабильной (разные ответы на один запрос)

**Вероятная причина:** слишком высокая температура, или модель переобучилась.

**Диагностика и решение:**

```python
# 1. Проверка стабильности
def check_stability(model, tokenizer, prompt, n_runs=5):
    """
    Проверяет стабильность ответов модели.
    """
    responses = []
    for _ in range(n_runs):
        response = generate_text(prompt, model, tokenizer)
        responses.append(response["text"])
    
    # Проверяем уникальность ответов
    unique = len(set(responses))
    if unique > 1:
        print(f"⚠️ Обнаружена нестабильность: {unique} разных ответов из {n_runs}")
        return False
    else:
        print(f"✅ Ответы стабильны: все {n_runs} одинаковые")
        return True

# 2. Уменьшение температуры
generation_config = GenerationConfig(
    temperature=0.1,  # снижаем с 0.7 до 0.1
    top_p=0.9,
    do_sample=True,
)
print("✅ Температура снижена для повышения стабильности")

# 3. Проверка переобучения
# Если модель стабильно даёт неправильные ответы — переобучение
# Используйте раннюю остановку из раздела 4.2
```

---

#### Общий алгоритм диагностики

Если вы не уверены, с чего начать, выполните следующие шаги по порядку:

```python
def diagnose_model_issues(model, tokenizer, generator, dataset, training_args):
    """
    Комплексная диагностика всех проблем.
    """
    print("=" * 70)
    print("ДИАГНОСТИКА МОДЕЛИ")
    print("=" * 70)
    
    issues = []
    
    # 1. Проверка pad_token
    if tokenizer.pad_token is None:
        issues.append("❌ pad_token не установлен")
        tokenizer.pad_token = tokenizer.eos_token
    
    # 2. Проверка обучаемых параметров
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    if trainable == 0:
        issues.append("❌ Нет обучаемых параметров")
    
    # 3. Проверка общих знаний (забывание)
    general_results = test_general_knowledge(model, tokenizer, generator)
    avg_general = sum(r["score"] for r in general_results) / len(general_results)
    if avg_general < 0.5:
        issues.append(f"⚠️ Забывание: сохранность знаний {avg_general:.0%}")
    
    # 4. Проверка галлюцинаций
    factual_results = test_factual_accuracy(model, tokenizer, generator)
    factual_accuracy = sum(1 for r in factual_results if r["is_correct"]) / len(factual_results)
    if factual_accuracy < 0.7:
        issues.append(f"⚠️ Галлюцинации: точность {factual_accuracy:.0%}")
    
    # 5. Проверка VRAM
    if torch.cuda.is_available():
        free_vram = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1024**3
        if free_vram < 1.0:
            issues.append(f"⚠️ Мало VRAM: {free_vram:.2f} GB свободно")
    
    # Итог
    print("\n" + "=" * 70)
    if issues:
        print("НАЙДЕНЫ ПРОБЛЕМЫ:")
        for issue in issues:
            print(f"   {issue}")
        print("\n💡 Рекомендации:")
        print("   1. Проверьте pad_token и apply_chat_template")
        print("   2. Убедитесь, что LoRA настроена корректно (target_modules, r)")
        print("   3. Добавьте негативные примеры (отказы)")
        print("   4. Уменьшите learning rate или batch size")
        print("   5. Включите gradient checkpointing при нехватке памяти")
    else:
        print("✅ Критических проблем не обнаружено")
    print("=" * 70)
    
    return issues

# Использование
issues = diagnose_model_issues(model, tokenizer, generator, dataset, training_args)
```

---

### Резюме по разделу 4.6

При возникновении проблем с обучением:

1. **NaN в loss** → проверьте `pad_token` и снизьте LR.
2. **Loss не снижается** → проверьте обучаемые параметры и увеличьте LR.
3. **Переобучение** → используйте раннюю остановку, уменьшите эпохи.
4. **Забывание** → переключитесь на LoRA, добавьте Replay.
5. **Галлюцинации** → добавьте негативные примеры, очистите данные.
6. **CUDA out of memory** → уменьшите batch size, включите checkpointing, используйте QLoRA.
7. **Медленная генерация** → выполните слияние адаптера или используйте квантование.
8. **Нестабильные ответы** → снизьте температуру, проверьте переобучение.

Используйте скрипт `diagnose_model_issues()` для комплексной проверки — он соберёт все проблемы и даст рекомендации.


### 4.7. Практические рекомендации по спасению эксперимента

1. **Всегда сравнивайте с бейзлайном.** Перед запуском тонкой настройки зафиксируйте качество Zero‑shot / Few‑shot промптов. Если после обучения вы не видите прироста – либо данные плохие, либо конфигурация неоптимальна.

```python
# Пример сравнения с бейзлайном
def compare_with_baseline(baseline_predictions, model_predictions):
    print("СРАВНЕНИЕ С БЕЙЗЛАЙНОМ")
    print(f"Бейзлайн (промпты): {baseline_predictions['accuracy']:.2f}")
    print(f"Модель после FT:     {model_predictions['accuracy']:.2f}")
    if model_predictions['accuracy'] > baseline_predictions['accuracy']:
        print("✅ Тонкая настройка улучшила качество")
    else:
        print("⚠️ Тонкая настройка не дала прироста — проверьте данные и конфигурацию")
```

2. **Следите за eval loss.** Это главный индикатор. Если он не снижается, а train loss падает – вы переобучаетесь. Если оба не падают – проблема в данных или LR.

```python
# Визуализация логов (пример с WandB)
import wandb
wandb.init(project="llm_finetuning", name="experiment_1")

# В цикле обучения логируем
for epoch in range(num_epochs):
    train_loss = train_epoch()
    eval_loss = evaluate()
    wandb.log({"train_loss": train_loss, "eval_loss": eval_loss, "epoch": epoch})
```

3. **Сохраняйте чекпоинты.** В PEFT вы можете сохранять адаптер после каждой эпохи и выбирать лучший по валидации.

```python
# Сохранение адаптера
peft_model.save_pretrained(f"./checkpoints/epoch_{epoch}")

# Загрузка лучшего чекпоинта
best_model = PeftModel.from_pretrained(base_model, "./checkpoints/best")
```

4. **Используйте небольшие эксперименты.** Перед полноценным запуском на всём датасете обучитесь на 100 примерах в течение 1 эпохи. Если всё работает, масштабируйте.

```python
# Быстрый тест на подвыборке
small_dataset = dataset[:100]
# Запуск обучения с теми же параметрами
trainer.train()
```

5. **Документируйте гиперпараметры.** Ведите таблицу экспериментов (rank, target_modules, LR, batch_size, eval_loss) – это поможет быстро находить оптимальные настройки.

```python
import pandas as pd

experiments = pd.DataFrame([
    {"rank": 8, "lr": 2e-4, "batch_size": 4, "eval_loss": 0.45},
    {"rank": 16, "lr": 2e-4, "batch_size": 4, "eval_loss": 0.38},
    {"rank": 16, "lr": 5e-5, "batch_size": 8, "eval_loss": 0.42},
])
print(experiments.sort_values("eval_loss"))
```

---

### 4.8. Итоговые выводы по теме 4

- **Катастрофическое забывание** – главный враг полной настройки; используйте PEFT, мультизадачность, Replay, регуляризацию и EWC.
- **Переобучение** лечится ранней остановкой, валидацией и ограничением числа эпох (1–3).
- **Усиление галлюцинаций** предотвращается очисткой данных и добавлением негативных примеров (отказов).
- **Технические проблемы** (pad token, чат-шаблон) решаются строгим соблюдением правил токенизации. Всегда проверяйте `tokenizer.pad_token` и применяйте `apply_chat_template`.
- Используйте **чек-листы** перед запуском и при диагностике – они сэкономят вам часы отладки. Код для автоматической проверки (`pre_launch_checklist`) поможет избежать типичных ошибок.
- **Практические рекомендации** – сравнивайте с бейзлайном, логируйте метрики, сохраняйте чекпоинты, проводите быстрые тесты и документируйте эксперименты.

Теперь вы вооружены знаниями, чтобы не только запускать тонкую настройку, но и уверенно справляться с любыми неожиданностями. В следующей теме мы перейдём к **введению в PEFT** – изучим конкретные методы, такие как LoRA, их конфигурацию и сравнение.


## Тема 5. PEFT: параметрически‑эффективная тонкая настройка

В предыдущих темах мы поняли, что полная тонкая настройка – это дорого, ресурсоёмко и чревато катастрофическим забыванием. Но что, если мы хотим адаптировать модель под задачу, не переписывая все 7 или 70 миллиардов весов? Именно для этого существует **PEFT** – семейство методов, которые позволяют дообучать модели с минимальными вычислительными затратами, сохраняя при этом высокое качество.

**Цель этой темы** – познакомить вас с миром PEFT, объяснить его преимущества, классифицировать основные подходы и дать чёткую рекомендацию, с чего начинать. Мы не будем углубляться в детали реализации каждого метода – это будет в следующих темах (LoRA и QLoRA). Здесь мы дадим общую картину, чтобы вы могли осознанно выбирать инструмент для своих задач.

---

### 5.1. Что такое PEFT?

**PEFT (Parameter‑Efficient Fine‑Tuning)** – это группа методов, которые позволяют адаптировать большие предобученные модели к новым задачам, обучая лишь **небольшую долю** параметров (обычно менее 1% от общего числа). Основная модель замораживается, а дополнительные параметры (адаптеры, низкоранговые матрицы, мягкие промпты) обучаются на целевом датасете.

**Главные преимущества PEFT:**

1. **Экономия памяти.** Поскольку основная модель не обновляется, нам не нужно хранить градиенты и состояния оптимизатора для всех параметров. Для 7B модели LoRA требует ~15 ГБ против ~84 ГБ для полной настройки.
2. **Скорость обучения.** Меньше обучаемых параметров → быстрее сходятся градиенты; обучение на том же объёме данных занимает в разы меньше времени.
3. **Предотвращение катастрофического забывания.** Замороженные веса сохраняют общие знания, а адаптеры лишь дополняют их, не перезаписывая старые паттерны.
4. **Гибкость.** Можно обучить несколько адаптеров для разных задач и переключать их во время инференса без перезагрузки модели.

Именно благодаря этим свойствам PEFT стал стандартом де‑факто в индустрии для большинства прикладных сценариев.

---

### 5.2. Основные классы PEFT методов

Все PEFT‑методы можно разделить на три большие группы по тому, *как* они модифицируют модель.

#### 5.2.1. Selective (выборочное обучение слоёв)

**Суть:** выбираются определённые слои или части слоёв, которые размораживаются и обучаются, а остальные остаются фиксированными. Например, можно обновлять только последние несколько слоёв трансформера.

**Примеры:** обучение только головы классификации (в задачах классификации), дообучение последних N слоёв.

**Недостаток:** выбор слоёв – эвристика, и не всегда понятно, какие именно слои отвечают за нужную адаптацию. Кроме того, число обучаемых параметров всё ещё может быть большим, если выбрать много слоёв. Сегодня Selective редко используется, уступив место более продвинутым методам.

#### 5.2.2. Reparameterization (перепараметризация)

**Суть:** веса модели представляются через низкоранговую декомпозицию (или другое сжатое представление), и обучаются только факторы этой декомпозиции. Самый известный представитель – **LoRA** (Low‑Rank Adaptation).

**Идея LoRA:** для каждого обучаемого слоя (например, `W` размером $d \times k$) мы добавляем две маленькие матрицы $A$ (размер $r \times d$) и $B$ (размер $d \times r$), где $r \ll d$. Обновлённый вес имеет вид:

$$
W' = W + \Delta W = W + B \cdot A
$$

Матрица $A$ инициализируется случайно, $B$ – нулями, поэтому в начале адаптер не меняет модель. Обучаются только $A$ и $B$, число параметров которых равно $r \cdot (d + k)$, что в сотни раз меньше, чем $d \cdot k$.

**Вариации:** LoRA, QLoRA (с квантованием базовой модели), AdaLoRA (динамическое распределение рангов), DoRA (разложение на величину и направление).

#### 5.2.3. Additive (добавление новых параметров)

**Суть:** в модель добавляются новые обучаемые слои или векторы, которые не заменяют существующие веса, а дополняют их. Это могут быть адаптеры (небольшие полносвязные слои между существующими), префиксы (обучаемые векторы, добавляемые к ключам и значениям на каждом слое) или мягкие промпты.

**Примеры:**

- **Adapters** – небольшие слои (обычно с бутылочной архитектурой) вставляются после каждого слоя трансформера. Обучаются только эти слои.
- **Prefix Tuning** – для каждого слоя к последовательности ключей и значений добавляются обучаемые префиксы (векторы). Модель использует их как дополнительный контекст.
- **Prompt Tuning** – в начало входной последовательности добавляются несколько обучаемых векторов («мягкие промпты»), которые оптимизируются под задачу. В отличие от обычных промптов, эти векторы не являются человеко‑читаемыми токенами, а представляют собой свободные параметры в пространстве эмбеддингов.

---

### 5.3. Сравнительная таблица популярных PEFT‑методов

Чтобы помочь вам выбрать метод, сведём три самых распространённых (LoRA, Adapters и Prefix Tuning) в таблицу по трём ключевым критериям: потребление памяти, качество на сложных задачах и скорость инференса.

| Метод | Потребление памяти (относительно Full FT) | Качество на сложных задачах | Скорость инференса (относительно базовой модели) |
| :--- | :--- | :--- | :--- |
| **LoRA** | Очень низкое (≈1‑2% от Full FT) | **Высокое** – часто сопоставимо с полной настройкой при правильно подобранном ранге | **Почти без потерь** – адаптер можно слить с основными весами (merge) |
| **Adapters** | Низкое (≈3‑5% от Full FT) | Среднее – хороши для классификации, но уступают LoRA на генеративных задачах | **Заметное замедление** (до 20‑30%) из‑за дополнительных слоёв в прямом проходе |
| **Prefix Tuning** | Низкое (≈1‑2% от Full FT) | **Среднее** – хорошо для генерации, но на сложных рассуждениях (математика, логика) часто уступает LoRA | **Небольшое замедление** (5‑10%) из‑за увеличения длины контекста |

**Почему LoRA стал самым популярным?**

- **Баланс качества и эффективности.** LoRA даёт качество, близкое к полной тонкой настройке, при минимальных затратах памяти и практически нулевой дополнительной задержке после слияния.
- **Простота использования.** В библиотеке Hugging Face PEFT LoRA реализована в несколько строк кода, и она совместима с большинством моделей (Llama, Qwen, Mistral, GPT‑2).
- **Гибкость.** Можно настраивать ранг, целевые слои и даже обучать несколько адаптеров одновременно.
- **Отсутствие влияния на инференс.** После слияния адаптера с весами модель становится такой же быстрой, как и базовая.
- **Широкая поддержка сообществом.** Огромное количество готовых адаптеров на Hugging Face Hub, которые можно использовать как есть или дообучать дальше.

---

### 5.4. Практический совет: начинайте с LoRA

Если вы только вступаете на путь тонкой настройки, не изобретайте велосипед. Начните с **LoRA** (или QLoRA, если у вас мало памяти). Вот почему:

1. **Это самый простой и надёжный способ.** Вам не нужно разбираться в тонкостях адаптеров или мягких промптов – LoRA работает «из коробки».
2. **Она даёт отличный baseline.** Запустите LoRA с r=8 или r=16, обучите 2–3 эпохи и посмотрите, насколько вырастет качество по сравнению с промптами. Если этого недостаточно, только тогда экспериментируйте с другими методами или увеличивайте ранг.
3. **QLoRA позволяет работать на потребительских GPU.** Даже 7B модели помещаются в 8–12 ГБ видеопамяти, что открывает двери для локальных экспериментов.
4. **Вы всегда можете переключиться.** Если позже вы решите, что LoRA не даёт нужного качества (хотя это бывает редко), вы можете перейти к полной настройке, имея готовые данные и понимание процесса.

**Резюмируем:** в 90% прикладных задач LoRA – это оптимальный выбор. Остальные методы (Adapters, Prefix Tuning) имеют узкие ниши, где они могут быть чуть лучше, но они сложнее в настройке и дают меньшие выгоды. Поэтому начните с LoRA – и вы не ошибётесь.

---

### Выводы по теме 5

- **PEFT** – это эффективный способ адаптации LLM без переобучения всех весов. Основные преимущества: экономия памяти, ускорение обучения, защита от забывания.
- Методы делятся на **Selective** (редко), **Reparameterization** (LoRA) и **Additive** (Adapters, Prefix Tuning).
- В сравнительной таблице LoRA выигрывает по всем фронтам: память, качество, скорость инференса – что объясняет её доминирование в индустрии.
- Практическая рекомендация: **начинайте с LoRA** – это самый быстрый способ получить дообученную модель с хорошим качеством.

Теперь, когда у вас есть системное представление о PEFT, мы готовы перейти к следующей теме – **LoRA и QLoRA на практике**, где мы разберём математику, гиперпараметры и реализацию.


## Тема 6. Метрики и оценка качества: от экспериментов до продакшена

Тонкая настройка модели – это не просто «запустить обучение и получить результат». Без системы измерения вы не сможете ответить на главные вопросы: *«Улучшилась ли модель?»*, *«Не переобучилась ли она?»*, *«Стоит ли её деплоить?»*. Оценка качества должна пронизывать весь жизненный цикл – от первых промптов до мониторинга в продакшене.

**Цель этой темы** – дать вам полную картину метрик и методов оценки на каждом этапе:

- **До настройки** – замерить бейзлайн и определить «потолок» качества.
- **Во время обучения** – следить за валидационной потерей (eval loss) и сохранять лучшие чекпоинты.
- **После настройки** – использовать автоматические метрики, LLM-as-a-judge и человеческую оценку.
- **Диагностика проблем** – понять, почему LoRA не бьёт бейзлайн, и исправить это.
- **В продакшене** – организовать A/B‑тестирование, сбор обратной связи и мониторинг дрейфа.

Мы также дадим примеры кода для логирования (WandB) и шаблоны промптов для LLM-as-a-judge, чтобы вы могли сразу применить их в своих экспериментах.

---

### 6.1. Оценка до настройки (Baseline)

Прежде чем что‑либо менять в весах, вы должны чётко понимать, *с чего вы начинаете*. Baseline – это качество вашей модели *до* тонкой настройки.

**Что нужно замерить:**

1. **Zero‑shot** – качество при простой инструкции (без примеров).
2. **Few‑shot** (если уместно) – качество при 3–5 примерах в контексте.
3. **Самые сильные промпты** – применяйте все продвинутые техники (CoT, Self‑Consistency), чтобы выжать максимум из модели без изменения весов.

**Как получить «потолок» качества:**

Используйте самую мощную доступную модель – обычно **GPT‑4** или **Claude 3.5 Sonnet** – с тем же самым промптом (или даже с улучшенным). Это даст вам верхнюю границу: если ваша дообученная модель приближается к этому потолку, вы на правильном пути.

**Пример оценки (на псевдокоде):**

```python
# Замеряем бейзлайн на валидационной выборке (100 примеров)
baseline_results = {
    "zero_shot": evaluate(model, val_data, prompt_template="zero_shot"),
    "few_shot": evaluate(model, val_data, prompt_template="few_shot"),
    "cot": evaluate(model, val_data, prompt_template="cot"),
}
print(baseline_results)

# Получаем потолок через GPT-4
gpt4_results = evaluate_gpt4(val_data, prompt_template="zero_shot")
# Сравниваем – это ваш ориентир.
```

---

### 6.2. Оценка во время обучения

Во время обучения главным индикатором является **eval loss** (потери на валидационной выборке). Это средняя кросс‑энтропия на отложенных данных. Чем ниже eval loss, тем лучше модель предсказывает следующие токены (но не забываем: низкий loss не всегда означает высокое качество генерации – см. раздел про галлюцинации).

**Почему eval loss – главный индикатор:**

- Он объективно показывает, как модель обобщается на новые данные.
- Если eval loss растёт, а train loss падает – это **явный сигнал переобучения**.
- Если оба не падают – проблема в данных или конфигурации (LR, формат).

**Как сохранять лучший чекпоинт:**

В библиотеке `transformers` в `TrainingArguments` есть параметр `load_best_model_at_end=True` и `metric_for_best_model="eval_loss"`. Автоматически сохраняется модель с минимальной eval loss.

```python
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./checkpoints",
    evaluation_strategy="steps",          # оцениваем каждые N шагов
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    load_best_model_at_end=True,          # загружаем лучший чекпоинт после обучения
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_dir="./logs",
    logging_steps=10,
)
```

**Логирование с WandB:**

Добавьте `report_to="wandb"` в `TrainingArguments` или инициализируйте WandB перед запуском:

```python
import wandb

wandb.init(project="llm_finetuning", name="lora_r16_lr2e4")

# Trainer автоматически будет логировать loss, learning rate, время
trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
)
trainer.train()
```

Теперь вы можете видеть графики eval loss и train loss в реальном времени на дашборде WandB.

---

### 6.3. Оценка после настройки

Когда обучение завершено, нужно сравнить дообученную модель с бейзлайном и с «потолком». Существует три основных подхода.

#### 6.3.1. Автоматические метрики (BLEU, ROUGE, METEOR)

Эти метрики сравнивают сгенерированный текст с эталонным (референсным) ответом. Они **уместны**, когда:

- Задача имеет однозначно правильный ответ (перевод, суммаризация, извлечение данных).
- Вы точно знаете, что референс – это «золотой стандарт».

**Ограничения:** они плохо коррелируют с человеческим восприятием для творческих задач. Низкий BLEU не всегда означает плохое качество – модель может перефразировать иначе, но смысл сохранить.

**Пример расчёта (библиотека `evaluate` от Hugging Face):**

```python
import evaluate

rouge = evaluate.load("rouge")
predictions = ["сгенерированный текст"]
references = ["эталонный текст"]
results = rouge.compute(predictions=predictions, references=references)
print(results)  # {'rouge1': 0.45, 'rouge2': 0.32, 'rougeL': 0.42}
```

#### 6.3.2. LLM‑as‑a‑judge (оценка другой LLM)

Для открытых задач (диалоги, креативная генерация) лучшим судьёй часто является **другая LLM** – например, GPT‑4. Вы даёте судье запрос, сгенерированный ответ и референс (если есть), и просите поставить оценку по шкале (например, 1–5) по заданным критериям: полезность, точность, связность, стиль.

**Пример промпта для судьи:**

```
Ты – опытный эксперт по оценке качества ответов ИИ. Оцени ответ на шкале от 1 до 5 по следующим критериям:
- Точность (фактическая достоверность)
- Полезность (отвечает ли на вопрос полностью)
- Структурированность (логика и читаемость)

Дай краткое обоснование и итоговую оценку в формате JSON.

Вопрос пользователя: {user_query}
Ответ модели: {model_answer}
Эталонный ответ (если есть): {reference}
```

**Важно:** чтобы избежать смещения, используйте несколько судей (например, GPT‑4 и Claude) и усредняйте. Также можно калибровать судью, давая ему примеры хороших и плохих ответов.

#### 6.3.3. Человеческая оценка

Она критически необходима, когда:

- Задача субъективна (креатив, юмор, эмоциональный тон).
- Вы внедряете модель в продукт, где важно мнение реальных пользователей.
- Автоматические метрики дают нестабильные результаты.

Организуйте пилотное тестирование с небольшим количеством экспертов (3–5 человек) – они оценивают ответы по тем же критериям, что и LLM‑судья. Человеческая оценка – это «золотой стандарт», но она дорога и медленна, поэтому используйте её для финальной валидации.

---

### 6.4. Диагностика проблем: почему LoRA не бьёт Baseline?

Вы запустили обучение, но качество не выросло или даже упало. Вот пошаговый чек‑лист для диагностики:

1. **Проверьте данные** – достаточны ли они? (менее 500 примеров – риск). Нет ли противоречий, ошибок? Есть ли негативные примеры (отказы)?
2. **Проверьте форматирование** – использовали ли `apply_chat_template`? Установлен ли `pad_token`? Если нет – возможны NaN и путаница ролей.
3. **Проверьте скорость обучения (LR)** – для QLoRA нужна 2e‑4, для LoRA в полной точности – 5e‑5. Если LR слишком мал – адаптер не обучается; слишком велик – обучение нестабильно.
4. **Проверьте целевые слои** – только `q_proj` и `v_proj` дают хороший результат для большинства задач. Если нужно сильное изменение стиля – добавьте `k_proj`, `o_proj`, MLP слои.
5. **Убедитесь, что адаптер действительно обучается** – вызовите `peft_model.print_trainable_parameters()` – там должно быть > 0.
6. **Оцените бейзлайн заново** – возможно, ваш бейзлайн был слишком оптимистичным (например, вы использовали очень сложный промпт). Перезамерьте.
7. **Попробуйте увеличить ранг** – если r=8 не даёт прироста, попробуйте r=16 или r=32.
8. **Проверьте валидационную выборку** – не пересекается ли она с обучающей? Нет ли утечки данных?

---

### 6.5. Оценка в продакшене (мониторинг)

После деплоя работа только начинается. Модель может деградировать из‑за **концептуального дрейфа** – изменения распределения пользовательских запросов со временем.

**Performance Drift** – это падение качества на новых данных по сравнению с валидацией. Его нужно отслеживать.

**Как организовать мониторинг:**

1. **Сбор лайков/дизлайков** – после каждого ответа дайте пользователю возможность оценить его (like/dislike). Это самый дешёвый сигнал.
2. **A/B‑тестирование** – запустите новую модель на 5–10% трафика (экспериментальная группа), остальные получают старую модель (контроль). Сравнивайте ключевые метрики: удержание, время ответа, частота лайков, конверсия.
3. **Дрейф входных данных** – отслеживайте распределение эмбеддингов запросов (с помощью модели эмбеддингов). Если распределение изменилось – это сигнал к переобучению.
4. **Периодический пересчёт качества** – раз в неделю запускайте офлайн‑валидацию на свежих данных с LLM‑as‑a‑judge, чтобы сравнить с бейзлайном.

**Пример структуры для логирования в продакшене:**

```python
# Псевдокод для сбора метрик
def log_inference(user_id, query, model_answer, latency, user_feedback=None):
    wandb.log({
        "user_id": user_id,
        "query_length": len(query),
        "answer_length": len(model_answer),
        "latency_ms": latency,
        "user_feedback": user_feedback,  # 1 = лайк, -1 = дизлайк
        "model_version": "v1.2-lora-r16",
        "timestamp": datetime.now()
    })
```

---

### 6.6. Итоговый чек‑лист: что и когда оценивать

| Этап | Что делаем | Как измеряем | Инструменты |
| :--- | :--- | :--- | :--- |
| **До настройки** | Zero‑shot, Few‑shot, CoT, GPT‑4 потолок | Точность, ROUGE, LLM‑as‑a‑judge | Промпты, evaluate, OpenAI API |
| **Во время обучения** | Отслеживаем eval loss | Визуализация графика loss | WandB, TensorBoard, Trainer |
| **После настройки** | Сравниваем с бейзлайном и потолком | BLEU/ROUGE (если эталон), LLM‑as‑a‑judge, человеческая оценка | evaluate, шаблоны промптов, экспертная панель |
| **Продакшен** | A/B‑тесты, сбор фидбэка, мониторинг дрейфа | Лайки/дизлайки, удержание, латентность | Система логирования, дашборд (Grafana, Prometheus), A/B‑платформа |

---

### Выводы по теме 6

- Всегда начинайте с **замера бейзлайна** (Zero‑shot, Few‑shot, CoT) и установки «потолка» через GPT‑4.
- Во время обучения главный индикатор – **eval loss**; используйте раннюю остановку и сохраняйте лучший чекпоинт.
- После обучения применяйте тройную оценку: **автоматические метрики** (для эталонных задач), **LLM‑as‑a‑judge** (для открытых задач), **человеческую оценку** (для субъективных и критичных сценариев).
- Если LoRA не бьёт бейзлайн – пройдите по чек‑листу (данные, формат, LR, целевые слои, ранг).
- В продакшене организуйте **мониторинг дрейфа**, **A/B‑тестирование** и сбор обратной связи от пользователей – это единственный способ объективно оценить влияние модели на бизнес.

Теперь вы обладаете полной системой оценки – от первой идеи до мониторинга в реальном времени. В следующей теме мы перейдём к **инструментам**, которые помогут вам реализовать все эти оценки и само обучение.


## Тема 7. Обзор инструментов для тонкой настройки: карта технологий

Вы изучили теорию, выбрали метод (скорее всего, LoRA) и готовы приступить к практике. Но современный стек для тонкой настройки LLM насчитывает десятки библиотек и фреймворков. Как не потеряться в этом многообразии? Где взять готовые конфиги? Какую библиотеку использовать для ускорения, а какую – для удобства?

**Цель этой темы** – составить для вас «карту инструментов»: от базовых библиотек (PEFT, Transformers) до узкоспециализированных ускорителей (Unsloth, Liger Kernel). Мы разберём, для чего нужен каждый инструмент, как их установить и в каких сценариях они незаменимы. Также мы дадим памятку по аппаратным требованиям, чтобы вы могли сразу оценить, потянет ли ваше железо выбранную модель.

Эта тема – ваш путеводитель по экосистеме тонкой настройки. В конце вы сможете собрать собственный стек, оптимальный для ваших задач.

---

### 7.1. Базовые библиотеки для PEFT и обучения

Начнём с фундамента. Без этих библиотек невозможно запустить ни один эксперимент.

#### Hugging Face Transformers
**Назначение:** загрузка моделей, токенизаторов, конфигураций. Это ядро экосистемы.
**Установка:** `pip install transformers`

#### Hugging Face PEFT
**Назначение:** реализация всех популярных PEFT‑методов (LoRA, AdaLoRA, Prefix Tuning и др.). Именно здесь вы создаёте конфигурацию адаптера и применяете её к модели.
**Установка:** `pip install peft`

#### TRL (Transformer Reinforcement Learning)
**Назначение:** высокоуровневый фреймворк для SFT (Supervised Fine‑Tuning), DPO и RLHF. Содержит удобные классы `SFTTrainer`, `DPOTrainer`, которые автоматизируют подготовку данных и обучение.
**Установка:** `pip install trl`
**Сценарий:** используйте `SFTTrainer`, если не хотите писать кастомный цикл обучения – он уже умеет работать с чат‑шаблонами, форматировать датасеты и применять PEFT.

#### Accelerate
**Назначение:** упрощает запуск обучения на нескольких GPU и в распределённых средах.
**Установка:** `pip install accelerate`

#### BitsAndBytes
**Назначение:** квантование моделей (4‑бит, 8‑бит) для экономии памяти. Критически важен для QLoRA.
**Установка:** `pip install bitsandbytes`

---

### 7.2. Библиотеки для ускорения и оптимизации

Эти инструменты позволяют быстрее обучать модели и эффективнее использовать память.

#### Unsloth
**Назначение:** оптимизированная реализация LoRA, которая работает на 30‑70% быстрее и потребляет на 30‑50% меньше памяти по сравнению со стандартной PEFT. Использует собственные ядра CUDA.
**Установка:** `pip install unsloth`
**Сценарий:** идеально подходит для локальных экспериментов, особенно на ограниченном оборудовании (например, 24GB GPU для 70B модели). Просто замените стандартную загрузку модели на `UnslothMistral` или `UnslothLlama`.

**Пример замены:**
```python
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/llama-3-8b-bnb-4bit",
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(model, lora_config)
```

#### Axolotl
**Назначение:** конфигурационный фреймворк для тонкой настройки. Позволяет описывать весь эксперимент в одном YAML‑файле (модель, датасет, параметры обучения, LoRA). Запускается одной командой.
**Установка:** `pip install axolotl` (или через `git clone`)
**Сценарий:** отлично подходит для воспроизводимых экспериментов и командной работы – все гиперпараметры хранятся в контроле версий.

**Пример конфига `config.yml`:**
```yaml
base_model: mistralai/Mistral-7B-v0.1
model_type: MistralForCausalLM
tokenizer_type: LlamaTokenizer

load_in_8bit: false
load_in_4bit: true
strict: false

datasets:
  - path: my_dataset
    type: alpaca

sequence_len: 2048
sample_packing: true

lora_r: 16
lora_alpha: 32
lora_target_modules:
  - q_proj
  - v_proj
  - k_proj
  - o_proj

learning_rate: 2e-4
num_epochs: 3
```

#### Liger Kernel (от LinkedIn)
**Назначение:** набор оптимизированных ядер для трансформеров, ускоряющих обучение на 20‑30% и экономящих память. Совместим с Hugging Face и TRL.
**Установка:** `pip install liger-kernel`
**Сценарий:** подключается как плагин к уже существующему коду. Особенно полезен для больших моделей и длинных последовательностей.

#### TorchAO (PyTorch Auto‑Optimization)
**Назначение:** автоматическое квантование и оптимизация моделей от команды PyTorch. Позволяет применять различные схемы квантования с минимальными изменениями кода.
**Установка:** входит в состав PyTorch 2.5+ (`torchao`).
**Сценарий:** используйте, если вы хотите поэкспериментировать с разными форматами квантования (INT8, FP8) без переписывания кода загрузки.

---

### 7.3. Инструменты для работы с данными

Качество данных – залог успеха. Инструменты Hugging Face позволяют легко загружать, преобразовывать и форматировать датасеты.

#### Hugging Face Datasets
**Назначение:** загрузка, предобработка, шаффл и сплит датасетов. Поддерживает потоковую загрузку для больших данных.
**Установка:** `pip install datasets`

**Форматы данных:**

- **Alpaca** – простой формат с полями `instruction`, `input`, `output`. Используется в ранних работах по instruction tuning.
- **ShareGPT** – диалоговый формат с массивом `conversations` (каждый оборот содержит `from` и `value`). Подходит для многооборотных диалогов.
- **ChatML** – стандарт OpenAI для чат‑моделей. Представляет собой строку с токенами `<|im_start|>` и `<|im_end|>`.

**Преобразование форматов:**
Вы можете написать функцию, которая преобразует Alpaca в ShareGPT, а затем применить `apply_chat_template` для финальной токенизации.

```python
def alpaca_to_shareGPT(example):
    messages = []
    if example["instruction"]:
        messages.append({"from": "human", "value": example["instruction"]})
    if example["input"]:
        messages[-1]["value"] += "\n" + example["input"]
    messages.append({"from": "gpt", "value": example["output"]})
    return {"conversations": messages}
```

#### apply_chat_template – зачем и как использовать?

Каждая модель (Llama, Qwen, Mistral) ожидает свой формат разметки диалога. Применение `tokenizer.apply_chat_template()` гарантирует, что специальные токены (например, `<|start_header_id|>`, `<|eot_id|>`) будут расставлены правильно. Это **критически важно**, иначе модель будет путать роли.

**Пример использования:**
```python
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is fine-tuning?"},
    {"role": "assistant", "content": "Fine-tuning is..."},
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False)
# После этого можно токенизировать и подавать в модель.
```

---

### 7.4. Квантование и BitsAndBytes

**BitsAndBytes** – это библиотека, которая позволяет загружать модели в 4‑битном формате, что радикально снижает потребление памяти. Она используется в QLoRA.

**Что такое NF4 (NormalFloat 4)?** Это специальный тип 4‑битного квантования, разработанный для нормального распределения весов. Он показывает лучшее качество, чем стандартный FP4, за счёт оптимального разбиения диапазона значений.

**Что такое Double Quantization?** Дополнительная техника, при которой мы квантуем не только веса, но и константы квантования (масштабы). Это даёт дополнительную экономию памяти (около 0.5‑1 ГБ для 7B модели) без потери качества.

**Конфигурация QLoRA (пример с BitsAndBytes):**
```python
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)
```

---

### 7.5. Аппаратные требования – памятка

Вот минимальные рекомендации по видеопамяти (VRAM) для разных моделей и методов. Значения приведены для батч‑сайза 1 и длины последовательности 2048 (консервативная оценка).

| GPU (VRAM) | Что можно запустить |
| :--- | :--- |
| **T4 / L4 (16 GB)** | QLoRA для 7B модели (4‑бит + LoRA) – **да**. LoRA в полной точности для 7B – **нет** (нужно ~15 ГБ, а у вас 16, но с учётом накладных расходов не влезет). Можно также QLoRA для 3B модели. |
| **RTX 3090 / 4090 (24 GB)** | LoRA для 7B (bf16) – **да** (~15 ГБ). QLoRA для 13B – **да**. QLoRA для 70B – нет (не влезет, нужно ~40 ГБ). Можно также Full FT для 3B (с аккумуляцией градиентов). |
| **A100 / H100 (40 GB)** | QLoRA для 70B – **да** (~40 ГБ). LoRA для 13B – **да**. Full FT для 7B с использованием DeepSpeed ZeRO‑3 – **да** (при правильной конфигурации). |
| **A100 / H100 (80 GB)** | LoRA для 70B – **да** (~140 ГБ не влезет, но QLoRA – да). Full FT для 13B – **да** (с ZeRO). Полная настройка 7B – **да** без проблем. |

**Примечания:**
- Для T4 с 16GB QLoRA 7B работает, но нужно включать gradient checkpointing и устанавливать маленький batch size (1–2).
- Для 70B моделей QLoRA требуется около 40 ГБ, что подходит для A100 40GB или 80GB.
- Всегда проверяйте фактическое потребление с помощью `nvidia-smi` в процессе настройки.

---

### 7.6. Схема стека технологий (Mermaid)

Визуализируем, как все инструменты связаны друг с другом:

```mermaid
flowchart LR
    subgraph Data
        DS[Hugging Face Datasets]
        FMT[Форматы: Alpaca, ShareGPT, ChatML]
        TEMPL[apply_chat_template]
    end

    subgraph Core
        TRANS[Transformers]
        PEFT[PEFT: LoRA, AdaLoRA]
        TRL[SFTTrainer, DPOTrainer]
        ACCEL[Accelerate]
    end

    subgraph Optimization
        BNB[BitsAndBytes: QLoRA]
        UNSLOTH[Unsloth: быстрая LoRA]
        LIGER[Liger Kernel: ускоренные ядра]
        TORCHAO[TorchAO: автоквантование]
        DEEPSPEED[DeepSpeed: ZeRO]
    end

    subgraph Management
        AXOLOTL[Axolotl: конфиги]
        WANDB[WandB: логирование]
    end

    DS --> FMT --> TEMPL --> TRL
    TRANS --> PEFT --> TRL
    PEFT --> UNSLOTH
    BNB --> PEFT
    TRL --> ACCEL --> DEEPSPEED
    AXOLOTL --> TRL
    TRL --> WANDB
```

---

### 7.7. Заключение – как собрать свой стек

1. **Для быстрых экспериментов на локальном GPU (24GB):**
   - Используйте `transformers` + `peft` + `bitsandbytes` (QLoRA).
   - Для ускорения добавьте `unsloth` – замените загрузку модели.
   - Логируйте метрики через WandB.

2. **Для воспроизводимых экспериментов в команде:**
   - Возьмите `axolotl` – описывайте всё в YAML, храните конфиги в Git.
   - Используйте `accelerate` для распределённого обучения.

3. **Для больших кластеров (A100/H100):**
   - Подключите `DeepSpeed ZeRO-3` для шардинга весов.
   - Используйте `liger-kernel` для дополнительного ускорения.

4. **Для всех проектов:**
   - Всегда применяйте `apply_chat_template` при работе с чат‑моделями.
   - Всегда проверяйте, что установлен `pad_token`.

Теперь у вас есть полный арсенал инструментов. В следующей теме мы перейдём к практическому заданию – вы примените эти знания, чтобы запустить свою первую тонкую настройку с LoRA.


## Тема 8. Заключение и домашнее задание

Мы прошли долгий путь – от первых определений до выбора инструментов. Теперь вы знаете:

- Что такое тонкая настройка и чем она отличается от предобучения.
- Когда она нужна, а когда лучше обойтись промптами или RAG.
- Как классифицируются методы (Full FT, PEFT, без изменений) и как их сравнивать.
- Какие проблемы (забывание, переобучение, галлюцинации) подстерегают вас – и как с ними бороться.
- Как устроены LoRA, QLoRA и другие PEFT‑методы.
- Как оценивать качество на всех этапах – от бейзлайна до продакшена.
- Какой стек инструментов использовать для эффективной работы.

Теперь пришло время применить эти знания на практике. Эта тема подводит итог и даёт вам чёткое домашнее задание, которое станет фундаментом для вашего первого проекта по тонкой настройке.

---

### 8.1. Схема выбора метода: от задачи к инструменту

Чтобы вы легко могли ориентироваться, я предлагаю простую **текстовую схему принятия решений**. Пройдите по шагам и выберите свой путь:

```
┌─────────────────────────────────────────────────────────────┐
│                      Задача / вопрос                        │
└─────────────────────────────────────────────────────────────┘
                              │
                              ▼
              ┌──────────────────────────────┐
              │ Есть ли у вас > 500 качественных │
              │ размеченных примеров?          │
              └──────────────────────────────┘
                     /                \
                   Да                 Нет
                    │                  │
                    ▼                  ▼
    ┌───────────────────────────┐  ┌─────────────────────────────────┐
    │ Нужно ли добавлять новые   │  │ Используйте промпты              │
    │ факты или база знаний     │  │ (Zero‑shot, Few‑shot, CoT)      │
    │ часто обновляется?        │  │ Если качество < 90% – переходите │
    └───────────────────────────┘  │ к RAG или сбору данных.          │
           /                \       └─────────────────────────────────┘
         Да                 Нет
          │                  │
          ▼                  ▼
┌──────────────────┐  ┌─────────────────────────────────────┐
│  RAG             │  │ Требуется изменить стиль, тон,      │
│  (подключайте    │  │ логику или формат вывода?          │
│  внешнюю БД)     │  └─────────────────────────────────────┘
└──────────────────┘         /                \
                           Да                 Нет
                            │                  │
                            ▼                  ▼
                ┌────────────────────┐  ┌─────────────────────┐
                │ Доступно ли GPU    │  │ Остановитесь на     │
                │ с 24+ ГБ VRAM?     │  │ промптах – они      │
                └────────────────────┘  │ вас устраивают.     │
                       /  \            └─────────────────────┘
                     Да   Нет
                      │     │
                      │     └─────────┐
                      ▼               ▼
         ┌──────────────────┐  ┌─────────────────────────┐
         │  LoRA (или Full  │  │ QLoRA (4‑бит)           │
         │  FT, если       │  │ для 7B–13B на T4/       │
         │  датасет >100K) │  │ 24GB GPU                │
         └──────────────────┘  └─────────────────────────┘
```

**Краткое резюме схемы:**

- **Малые данные (<500)** → не торопитесь с обучением. Начните с промптов. Если не хватает – собирайте больше данных.
- **Динамические факты** → RAG.
- **Изменение стиля, формата, логики** → PEFT (LoRA/QLoRA).
- **Большой датасет (>100K) и ресурсы** → можно рассмотреть Full FT.
- **Если бюджет ограничен** → QLoRA на облачном GPU.

---

### 8.2. Обязательное домашнее задание

Это задание поможет вам закрепить теорию и подготовиться к практическим экспериментам в следующих лекциях. Выполните все пункты по порядку.

#### Шаг 1. Изучите теорию

- Прочитайте **оригинальную статью о LoRA** (Hu et al., 2021): [LoRA: Low-Rank Adaptation of Large Language Models](https://arxiv.org/abs/2106.09685). Обратите внимание на разделы, посвящённые математике и гиперпараметрам.
- Также ознакомьтесь со статьёй о **QLoRA** (Dettmers et al., 2023): [QLoRA: Efficient Finetuning of Quantized LLMs](https://arxiv.org/abs/2305.14314) – хотя бы раздел о 4‑битном квантовании и NF4.

#### Шаг 2. Подготовьте датасет

- Соберите **100–200 пар «вопрос–ответ»** в вашей предметной области. Это может быть:
  - Техническая поддержка (вопросы клиентов и ответы операторов).
  - Генерация резюме или переформулировка текстов.
  - Классификация (вопрос → категория или тональность).
  - Любая задача, где вы хотите улучшить стиль или структуру ответа.
- Сохраните датасет в формате JSON (например, `{"instruction": "...", "output": "..."}`) или CSV.
- **Обязательно** выделите валидационную выборку (10–20%) – она пригодится для оценки во время обучения.

#### Шаг 3. Протестируйте промпты

- Напишите **Zero‑shot** промпт для вашей задачи и оцените его на 20 случайных примерах из валидационной выборки.
- Напишите **Few‑shot** (3–5 примеров) и снова оцените.
- Сравните результаты: зафиксируйте, насколько Few‑shot улучшил качество (по вашей метрике – точность, оценка судьи и т.д.).
- Если возможно, получите «потолок» с помощью GPT‑4 или Claude – это даст вам ориентир.

#### Шаг 4. Составьте план эксперимента

На основе того, что вы узнали в лекциях, разработайте план для первого запуска тонкой настройки:

- **Метод:** LoRA или QLoRA? (если у вас ограниченный GPU – однозначно QLoRA).
- **Ранг `r`:** начните с 8 или 16.
- **`lora_alpha`:** выберите как `2 * r`.
- **Целевые слои:** минимум `q_proj` и `v_proj`; добавьте `k_proj`, `o_proj` для более сильной адаптации.
- **Скорость обучения (LR):** для QLoRA – 2e‑4; для LoRA в полной точности – 5e‑5.
- **Количество эпох:** 2–3 (с ранней остановкой).
- **Батч‑сайз:** подберите в зависимости от VRAM (начните с 4–8).
- **Метрики:** что будете отслеживать? (eval loss, точность, ROUGE, оценка LLM).

Запишите все параметры в таблицу – это станет основой вашего отчёта.

#### Шаг 5. Оцените теоретическую производительность

Прикиньте, сколько токенов в секунду будет генерировать ваша модель после слияния адаптера (базовая скорость) и сколько памяти она будет занимать. Используйте формулы из лекции (тема 1.6). Это поможет вам понять, что вы получите на своём оборудовании.

---

### 8.3. Дополнительное задание (звёздочка)

Для тех, кто хочет попробовать код до следующей лекции:

1. **Установите стек:**
   ```bash
   pip install transformers peft bitsandbytes accelerate trl datasets wandb unsloth
   ```

2. **Загрузите базовую модель** (например, `Qwen/Qwen2.5-3B` или `meta-llama/Llama-3.2-3B`) с квантованием (для QLoRA).

3. **Настройте LoRA** (используйте конфиг с r=8, alpha=16, target_modules=["q_proj","v_proj"]).

4. **Обучите модель** на вашем датасете (хотя бы 1 эпоху) с помощью `SFTTrainer` из TRL. Если не хватает времени, можете обучить на подвыборке (50 примеров).

5. **Сохраните адаптер** и протестируйте инференс на нескольких примерах из валидации.

6. **Сравните** качество до и после – подтвердилось ли улучшение?

Если вы выполнили это задание, вы уже на шаг впереди! В следующей лекции мы разберём все эти шаги в коде, но собственный опыт даст вам гораздо больше понимания.

---

### 8.4. Требования к отчёту

Ваш отчёт по обязательному домашнему заданию должен содержать следующие разделы:

1. **Описание задачи** – что вы хотите улучшить в модели, почему выбрали именно эту задачу.
2. **Датасет** – сколько примеров, как собирали, какие форматы, как разбили на трейн/валидацию. Приведите 3–4 примера.
3. **Результаты промптов** – таблица с метриками для Zero‑shot и Few‑shot, а также потолок (GPT‑4). Укажите, как именно оценивали.
4. **План эксперимента** – заполненная таблица гиперпараметров (по шагу 4) с обоснованием каждого выбора.
5. **Оценка ресурсов** – сколько памяти займёт ваша модель, какая ожидаемая скорость инференса, сколько времени займёт обучение.
6. **Выводы** – что вы узнали, какие у вас ожидания от тонкой настройки, какой метод вы выбрали и почему.

**Критерии оценки (максимум 10 баллов):**
- Полнота датасета и его качество (2 балла).
- Корректность промптов и их оценка (2 балла).
- Обоснованность плана эксперимента (3 балла).
- Глубина анализа ожидаемых ресурсов и рисков (2 балла).
- Чёткость и структурированность отчёта (1 балл).

Отчёт присылайте в формате Markdown или PDF. Объём – 2–3 страницы (не более 5). Дедлайн – перед следующей лекцией.

---

### 8.5. Финальное резюме всей лекции

Мы заложили **системный фундамент** для работы с тонкой настройкой LLM. Теперь вы:

- Понимаете, чем отличается обучение от промптов и RAG.
- Можете принимать взвешенные решения: когда и какой метод использовать.
- Знаете о подводных камнях (забывание, галлюцинации, переобучение) и умеете с ними бороться.
- Освоили основные PEFT‑методы и готовы работать с LoRA.
- Владеете инструментами оценки и мониторинга.
- Имеете список инструментов для быстрого старта.

Следующие лекции будут полностью практическими – мы реализуем LoRA и QLoRA, обучим модель на реальном датасете, проанализируем результаты и внедрим систему мониторинга.


# Лекция 4.2. LoRA и QLoRA: практическая параметрически-эффективная настройка

## Тема 1. Математические основы LoRA

**Цель:** глубоко, но доступно разобрать математику LoRA — от расчёта памяти до тонкостей инициализации. Мы используем оригинальную статью (Hu et al., 2021) как основной источник и дополняем её числовыми примерами.

---

### 1.1. Проблема полной настройки: почему это дорого?

Полная тонкая настройка (Full Fine-Tuning) обновляет все параметры модели. Для этого необходимо хранить в видеопамяти GPU следующие компоненты:

#### Расчёт памяти для модели с \(P\) параметрами в bf16 (2 байта)

| Компонент | Формула | Для 7B (P=7·10⁹) | Почему |
| :--- | :--- | :--- | :--- |
| **Веса модели** | \(P \times 2\) байт | 14 ГБ | Нужны для forward/backward |
| **Градиенты** | \(P \times 2\) байт | 14 ГБ | Для каждого веса нужно хранить градиент |
| **Состояния Adam** | \(2 \times P \times 4\) байт | 56 ГБ | Adam хранит два fp32 момента: среднее и квадрат градиента |
| **Активации** | Зависят от batch size | ~10–20 ГБ | Промежуточные значения (можно уменьшить через checkpointing) |
| **Итого** | \(P \times (2+2+8) +\) активации \(\approx 12P\) байт | **~84–100 ГБ** | Для 7B нужно минимум 80 ГБ VRAM |

#### Почему Adam требует 8 байт на параметр?

Adam хранит два момента для каждого параметра:
- \(m_t = \beta_1 m_{t-1} + (1-\beta_1)g_t\) — экспоненциально взвешенное среднее градиентов
- \(v_t = \beta_2 v_{t-1} + (1-\beta_2)g_t^2\) — экспоненциально взвешенное среднее квадратов градиентов

Оба момента хранятся в fp32 (4 байта) для численной стабильности → \(2 \times 4 = 8\) байт на параметр.

#### Что это означает на практике — данные из статьи LoRA:

В оригинальной статье авторы пишут:

> *"On GPT-3 175B, we reduce the VRAM consumption during training from 1.2TB to 350GB. With \(r = 4\) and only the query and value projection matrices being adapted, the checkpoint size is reduced by roughly \(10,000\times\) (from 350GB to 35MB)."*

| Модель | Полная настройка | LoRA (r=4) | Экономия |
| :--- | :--- | :--- | :--- |
| **GPT-3 175B** | 1.2 TB VRAM | 350 GB VRAM | **×3.4** |
| **GPT-3 175B** | 350 GB чекпоинт | 35 MB чекпоинт | **×10 000** |

**Почему это дорого:**

- **Оборудование:** для 7B нужен A100 80GB (и то с аккумуляцией). Для 13B — 2–4 GPU. Для 70B — кластер из 8+ GPU. Для 175B — десятки GPU.
- **Время и стоимость:** обучение 7B на 100K примеров — 10–20 часов на A100. Аренда A100 стоит $2–4/час → **$30–80 за эксперимент**.
- **Эксперименты:** подбор гиперпараметров требует множества прогонов → затраты растут экспоненциально.

**Именно поэтому нужен LoRA:** он снижает требования к памяти и делает дообучение доступным на обычных GPU.

---

### 1.2. Гипотеза низкой внутренней размерности

**Откуда берётся идея LoRA?**

В 2018–2020 годах исследователи обнаружили важный эмпирический факт:

> **Предобученные модели живут на низкой внутренней размерности.** Их можно эффективно адаптировать к новым задачам, изменяя лишь небольшое подмножество «направлений» в пространстве весов.

В статье LoRA авторы ссылаются на работы:
- Li et al. (2018a) — измерение внутренней размерности ландшафта потерь
- Aghajanyan et al. (2020) — объяснение эффективности тонкой настройки через низкую внутреннюю размерность

**Гипотеза LoRA:**

> **Изменения весов \(\Delta W = W - W_0\) при адаптации к новой задаче имеют низкий «внутренний ранг» (intrinsic rank).**

Формально: пусть \(W_0 \in \mathbb{R}^{d \times k}\) — веса предобученной модели. После адаптации мы получаем \(W = W_0 + \Delta W\). Авторы LoRA показали, что матрица \(\Delta W\) может быть хорошо аппроксимирована произведением двух матриц малого размера:

\[
\Delta W \approx B \cdot A, \quad B \in \mathbb{R}^{d \times r}, \; A \in \mathbb{R}^{r \times k}, \; r \ll \min(d,k).
\]

**Почему это так?** Предобученная модель уже содержит богатые представления языка, фактов и рассуждений. Адаптация к конкретной задаче требует лишь **небольших корректировок** — она затрагивает ограниченное число «направлений» в пространстве весов. Это похоже на то, как опытный музыкант, разучивая новое произведение, корректирует лишь несколько движений, а не переучивается игре с нуля.

#### Эмпирическое подтверждение из статьи (Таблица 6):

| Ранг \(r\) | Trainable Params | WikiSQL Accuracy |
| :--- | :--- | :--- |
| **1** | 4.7M | 73.4% |
| **2** | 9.4M | 73.3% |
| **4** | 18.8M | 73.7% |
| **8** | 37.7M | 73.8% |
| **64** | 301.9M | 73.5% |

На GPT-3 175B, даже при \(r = 1\), LoRA достигает качества, близкого к полной настройке (73.8% при \(r=8\)). Это прямое доказательство: **ранг адаптации действительно очень низкий!**

Авторы также исследовали перекрытие подпространств между \(A_{r=8}\) и \(A_{r=64}\):

> *"Directions corresponding to the top singular vector overlap significantly between \(A_{r=8}\) and \(A_{r=64}\), while others do not... providing an explanation of why \(r = 1\) performs quite well."*

---

### 1.3. Формула LoRA: \(W' = W_0 + B \cdot A\)

Основная идея LoRA — заморозить предобученные веса \(W_0\) и вместо их обновления вводить **обучаемые матрицы малого ранга** \(A\) и \(B\):

\[
W' = W_0 + \Delta W = W_0 + B \cdot A,
\]

где:
- \(W_0 \in \mathbb{R}^{d \times k}\) — замороженные веса (не обучаются).
- \(A \in \mathbb{R}^{r \times k}\) — обучаемая матрица.
- \(B \in \mathbb{R}^{d \times r}\) — обучаемая матрица.
- \(r \ll \min(d,k)\) — ранг.

**Как выглядит проход вперёд?**

Для входного вектора \(x \in \mathbb{R}^{k}\):

\[
h = W_0 x + \Delta W x = W_0 x + B A x.
\]

То есть к выходу основного слоя добавляется адаптивный член.

#### Инициализация: почему \(B\) нулевая, а \(A\) случайная?

| Матрица | Инициализация | Почему |
| :--- | :--- | :--- |
| **A** | Случайная (Гаусс \(\mathcal{N}(0, \sigma^2)\)) | Даёт разные направления для градиентов, предотвращает симметрию |
| **B** | Нулевая | В начале \(B \cdot A = 0\), значит \(W' = W_0\). Модель стартует с предобученных весов |

**Почему это важно?** Если бы \(B\) тоже была случайной, то адаптер вносил бы случайные изменения на старте, что могло бы привести к нестабильности и скачкам loss. А так — обучение начинается с уже хорошо работающих весов.

В статье авторы пишут:

> *"We use a random Gaussian initialization for A and zero for B, so \(\Delta W = BA\) is zero at the beginning of training."*

**Что происходит с градиентами?**

При обучении мы обновляем только \(A\) и \(B\). Градиенты вычисляются через цепное правило:

\[
\frac{\partial \mathcal{L}}{\partial B} = \frac{\partial \mathcal{L}}{\partial (B A)} A^T, \quad
\frac{\partial \mathcal{L}}{\partial A} = B^T \frac{\partial \mathcal{L}}{\partial (B A)}.
\]

Так как \(B\) начинается с нуля, \(\partial \mathcal{L}/\partial A = 0\) на первых шагах. Это значит, что \(A\) не обновляется, пока \(B\) не станет ненулевым. На практике градиенты всё равно идут, потому что \(B\) обновляется первым и начинает «протаскивать» градиент к \(A\).

#### Масштабирование \(\alpha / r\)

На практике выход адаптера умножается на коэффициент:

\[
W' = W_0 + \frac{\alpha}{r} B \cdot A,
\]

где \(\alpha\) — константа (lora_alpha), обычно равная \(2 \times r\).

**Зачем это нужно?**

При смене ранга \(r\) мы хотим сохранить примерно одинаковый масштаб обновлений. Если \(r\) увеличивается вдвое, количество параметров удваивается. Без масштабирования градиенты могли бы изменить свою величину. Коэффициент \(\alpha/r\) компенсирует это: чем больше \(r\), тем меньше масштаб, и наоборот.

В статье авторы пишут:

> *"We then scale \(\Delta W x\) by \(\frac{\alpha}{r}\), where \(\alpha\) is a constant in \(r\). When optimizing with Adam, tuning \(\alpha\) is roughly the same as tuning the learning rate if we scale the initialization appropriately."*

**Практическое правило:** \(\alpha = 2 \times r\) (например, r=16 → alpha=32). Это даёт стабильные градиенты.

---

### 1.4. Что такое ранг \(r\) и как его выбрать?

**Ранг \(r\)** — размерность подпространства, в котором мы ищем изменения весов. Чем больше \(r\), тем больше «свободы» у адаптера, но и тем больше параметров нужно обучать.

#### Какой ранг выбрать? — данные из статьи

В статье авторы исследовали влияние ранга на качество:

**Таблица 6 (GPT-3 175B, адаптация \(W_q, W_v\)):**

| Ранг \(r\) | Trainable Params | WikiSQL | MNLI |
| :--- | :--- | :--- | :--- |
| **1** | 4.7M | 73.4% | 91.3% |
| **2** | 9.4M | 73.3% | 91.4% |
| **4** | 18.8M | 73.7% | 91.3% |
| **8** | 37.7M | 73.8% | 91.6% |
| **64** | 301.9M | 73.5% | 91.4% |

**Вывод:** качество практически не растёт после \(r=4\)–\(8\), а количество параметров при \(r=64\) в 64 раза больше, чем при \(r=1\).

| Ранг \(r\) | Параметров на слой (\(d=4096\)) | Рекомендация |
| :--- | :--- | :--- |
| **1** | \(2 \cdot 4096 \cdot 1 = 8\,192\) | Минимальный тест (иногда работает!) |
| **4** | \(2 \cdot 4096 \cdot 4 = 32\,768\) | Простые задачи |
| **8** | \(2 \cdot 4096 \cdot 8 = 65\,536\) | Базовый уровень |
| **16** | \(2 \cdot 4096 \cdot 16 = 131\,072\) | **Стандарт по умолчанию** |
| **32** | \(2 \cdot 4096 \cdot 32 = 262\,144\) | Сложные задачи |
| **64** | \(2 \cdot 4096 \cdot 64 = 524\,288\) | Очень сложные задачи |

**Почему убывающая отдача после 64?**

Из Таблицы 6 видно, что при \(r=64\) качество не улучшается, а параметров становится на порядок больше. Исследование подпространств (Рисунок 3 в статье) показывает, что направления, соответствующие топ-1 сингулярному вектору, перекрываются между \(r=1\) и \(r=64\), а остальные содержат в основном шум.

> *"This suggests that the top singular-vector directions of \(A_{r=8}\) and \(A_{r=64}\) are the most useful, while other directions potentially contain mostly random noises accumulated during training."*

**Рекомендация:** начинайте с \(r=16\). Если качество не устраивает — попробуйте \(r=32\) или \(r=64\). Больше 64 — редко когда нужно.

---

### 1.5. Расчёт количества параметров LoRA

Для одного слоя размером \(d \times d\) (квадратная матрица, как в трансформерах):

\[
\text{Params}_{\text{LoRA}} = 2 \cdot d \cdot r.
\]

**Почему \(2 \cdot d \cdot r\)?** — потому что две матрицы: \(A\) размером \(r \times d\) и \(B\) размером \(d \times r\).

#### Сравнение с полным слоем (\(d^2\)):

| Размер слоя \(d\) | Полный слой \(d^2\) | LoRA (\(r=16\)) | Экономия |
| :--- | :--- | :--- | :--- |
| **1024** | 1 048 576 | 32 768 | **×32** |
| **4096** | 16 777 216 | 131 072 | **×128** |
| **8192** | 67 108 864 | 262 144 | **×256** |

#### Для всей модели GPT-3 175B (из статьи):

В статье авторы пишут:

> *"With \(r = 4\) and only the query and value projection matrices being adapted, the checkpoint size is reduced by roughly \(10,000\times\) (from 350GB to 35MB)."*

**Расчёт параметров:**

- Полная модель: **175 млрд** параметров.
- LoRA (\(r=4\), адаптируем \(W_q\) и \(W_v\), 96 слоёв):
  - На слой: \(2 \cdot d \cdot r = 2 \cdot 12\,288 \cdot 4 = 98\,304\) параметров
  - На 2 проекции: \(2 \times 98\,304 = 196\,608\) параметров на слой
  - На 96 слоёв: \(96 \times 196\,608 \approx 18.9\) млн параметров
- Сравнение: \(18.9\) млн vs \(175\) млрд → **в 9 260 раз меньше!**

#### Для 7B модели:

Если адаптировать только \(W_q\) и \(W_v\) (2 проекции на слой, 32 слоя, \(d=4096\), \(r=16\)):

\[
\text{Params}_{\text{LoRA}} = 2 \cdot 2 \cdot 32 \cdot 4096 \cdot 16 = 8\,388\,608 \approx 8.4\text{ млн}.
\]

Это всего **0.12%** от 7B модели!

---

### 1.6. Ключевые свойства LoRA (из статьи)

#### 1. Нулевая задержка инференса

> *"Our simple linear design allows us to merge the trainable matrices with the frozen weights when deployed, introducing no inference latency compared to a fully fine-tuned model."*

После обучения мы вычисляем \(W = W_0 + BA\) и используем её как обычную матрицу — никаких дополнительных слоёв, никакой задержки.

#### 2. Быстрое переключение задач

> *"When we need to switch to another downstream task, we can recover \(W_0\) by subtracting \(BA\) and then adding a different \(B'A'\), a quick operation with very little memory overhead."*

```python
# Храним одну базовую модель + множество адаптеров
base_model = load_model("gpt-3-175b")  # 350 GB

# Задача 1: техподдержка (4.7M параметров)
adapter_1 = load_adapter("support_task")

# Задача 2: юридические консультации (4.7M параметров)
adapter_2 = load_adapter("legal_task")

# Переключение — быстрая операция!
adapter_1.unload()
adapter_2.load()
```

#### 3. Экономия памяти до 3 раз

> *"Compared to GPT-3 175B fine-tuned with Adam, LoRA can reduce the number of trainable parameters by 10,000 times and the GPU memory requirement by 3 times."*

#### 4. Качество лучше или на уровне полной настройки

Из Таблицы 4 (GPT-3 175B):

| Метод | Trainable Params | MNLI Accuracy |
| :--- | :--- | :--- |
| Full Fine-Tune | 175B | 89.5% |
| **LoRA** (r=8) | **4.7M** | **91.7%** |

LoRA **превосходит** полную настройку, используя в 37 000 раз меньше параметров!

---

### 1.7. Полный числовой пример: от матриц до обучения

#### Исходные данные

Пусть у нас есть слой трансформера с размерностью \(d = 4\) (для простоты). Предобученная матрица:

\[
W_0 = \begin{pmatrix}
1 & 2 & 3 & 4 \\
5 & 6 & 7 & 8 \\
9 & 10 & 11 & 12 \\
13 & 14 & 15 & 16
\end{pmatrix}
\]

Мы хотим адаптировать этот слой под новую задачу с помощью LoRA. Выбираем ранг \(r = 2\).

#### Шаг 1: Инициализация

Инициализируем матрицы \(A\) и \(B\):

**A (случайная, из Гаусса):**

\[
A = \begin{pmatrix}
0.2 & -0.1 & 0.3 & 0.0 \\
0.1 & 0.4 & -0.2 & 0.5
\end{pmatrix}
\]

**B (нулевая):**

\[
B = \begin{pmatrix}
0 & 0 \\
0 & 0 \\
0 & 0 \\
0 & 0
\end{pmatrix}
\]

На старте \(\Delta W = B \cdot A = 0\), значит \(W' = W_0\). Адаптер не вносит изменений.

#### Шаг 2: Обучение (один шаг)

Предположим, после первого батча градиенты обновили \(A\) и \(B\). Теперь они выглядят так:

**A (после обновления):**

\[
A = \begin{pmatrix}
0.25 & -0.15 & 0.35 & 0.05 \\
0.15 & 0.45 & -0.25 & 0.55
\end{pmatrix}
\]

**B (после обновления):**

\[
B = \begin{pmatrix}
0.2 & 0.1 \\
0.3 & 0.2 \\
0.1 & 0.4 \\
0.5 & 0.3
\end{pmatrix}
\]

#### Шаг 3: Вычисление \(\Delta W = B \cdot A\)

\[
\Delta W = B \cdot A =
\begin{pmatrix}
0.2 & 0.1 \\
0.3 & 0.2 \\
0.1 & 0.4 \\
0.5 & 0.3
\end{pmatrix}
\cdot
\begin{pmatrix}
0.25 & -0.15 & 0.35 & 0.05 \\
0.15 & 0.45 & -0.25 & 0.55
\end{pmatrix}
\]

Выполним умножение (элемент \((i,j)\) = сумма произведений):

\[
\Delta W =
\begin{pmatrix}
0.2 \cdot 0.25 + 0.1 \cdot 0.15 & 0.2 \cdot (-0.15) + 0.1 \cdot 0.45 & 0.2 \cdot 0.35 + 0.1 \cdot (-0.25) & 0.2 \cdot 0.05 + 0.1 \cdot 0.55 \\
0.3 \cdot 0.25 + 0.2 \cdot 0.15 & 0.3 \cdot (-0.15) + 0.2 \cdot 0.45 & 0.3 \cdot 0.35 + 0.2 \cdot (-0.25) & 0.3 \cdot 0.05 + 0.2 \cdot 0.55 \\
0.1 \cdot 0.25 + 0.4 \cdot 0.15 & 0.1 \cdot (-0.15) + 0.4 \cdot 0.45 & 0.1 \cdot 0.35 + 0.4 \cdot (-0.25) & 0.1 \cdot 0.05 + 0.4 \cdot 0.55 \\
0.5 \cdot 0.25 + 0.3 \cdot 0.15 & 0.5 \cdot (-0.15) + 0.3 \cdot 0.45 & 0.5 \cdot 0.35 + 0.3 \cdot (-0.25) & 0.5 \cdot 0.05 + 0.3 \cdot 0.55
\end{pmatrix}
\]

\[
\Delta W =
\begin{pmatrix}
0.05 + 0.015 & -0.03 + 0.045 & 0.07 - 0.025 & 0.01 + 0.055 \\
0.075 + 0.03 & -0.045 + 0.09 & 0.105 - 0.05 & 0.015 + 0.11 \\
0.025 + 0.06 & -0.015 + 0.18 & 0.035 - 0.10 & 0.005 + 0.22 \\
0.125 + 0.045 & -0.075 + 0.135 & 0.175 - 0.075 & 0.025 + 0.165
\end{pmatrix}
\]

\[
\Delta W =
\begin{pmatrix}
0.065 & 0.015 & 0.045 & 0.065 \\
0.105 & 0.045 & 0.055 & 0.125 \\
0.085 & 0.165 & -0.065 & 0.225 \\
0.170 & 0.060 & 0.100 & 0.190
\end{pmatrix}
\]

#### Шаг 4: Применение масштабирования \(\alpha / r\)

Пусть \(\alpha = 4\) (по правилу \(\alpha = 2 \times r = 2 \times 2 = 4\)), тогда \(\alpha/r = 4/2 = 2\).

\[
\Delta W_{\text{scaled}} = \frac{\alpha}{r} \cdot \Delta W = 2 \cdot \Delta W =
\begin{pmatrix}
0.130 & 0.030 & 0.090 & 0.130 \\
0.210 & 0.090 & 0.110 & 0.250 \\
0.170 & 0.330 & -0.130 & 0.450 \\
0.340 & 0.120 & 0.200 & 0.380
\end{pmatrix}
\]

#### Шаг 5: Обновлённая матрица весов

\[
W' = W_0 + \Delta W_{\text{scaled}} =
\begin{pmatrix}
1.130 & 2.030 & 3.090 & 4.130 \\
5.210 & 6.090 & 7.110 & 8.250 \\
9.170 & 10.330 & 10.870 & 12.450 \\
13.340 & 14.120 & 15.200 & 16.380
\end{pmatrix}
\]

#### Шаг 6: Сравнение количества параметров

| Параметр | Количество |
| :--- | :--- |
| **Полный слой** (\(d^2 = 4 \times 4\)) | **16** параметров |
| **LoRA** (\(2 \cdot d \cdot r = 2 \cdot 4 \cdot 2\)) | **16** параметров (\(A\): 8, \(B\): 8) |
| **Обучается** | только 16 параметров LoRA |
| **Заморожено** | 16 параметров \(W_0\) |

Для реальных моделей (\(d=4096, r=16\)):

| Параметр | Количество |
| :--- | :--- |
| **Полный слой** (\(d^2\)) | **16.7M** параметров |
| **LoRA** (\(2 \cdot d \cdot r\)) | **131K** параметров |
| **Экономия** | **×128** |

---

### Итоговые выводы по теме 1

1. **Полная настройка** требует **~84 ГБ VRAM** для 7B и стоит $30–80/эксперимент.
2. **LoRA** основан на гипотезе о низкой размерности изменений весов (\(\Delta W\) имеет низкий ранг).
3. Формула: \(W' = W_0 + \frac{\alpha}{r} B A\), где \(B=0\) в начале, \(A\) случайна.
4. Масштабирование \(\alpha/r\) — для стабильности градиентов при смене ранга.
5. **Ранг \(r\)** — начинайте с 16, увеличивайте до 32–64 при необходимости.
6. LoRA добавляет всего **0.12%** параметров, но даёт качество близкое к Full FT.
7. **Свойства:** нулевая задержка, быстрое переключение задач, экономия до 10 000 раз.

**Помните слова из статьи:**
> *"We show that a very low rank (i.e., \(r\) can be one or two) suffices even when the full rank (\(d\)) is as high as 12,288."*

Именно это делает LoRA революционным методом. Теперь, когда вы понимаете математику, переходим к практике — как настроить LoRA в коде.


## Тема 2. Конфигурация LoRA – ключевые гиперпараметры

---

### Введение: почему LoRA — это не «чёрный ящик»

LoRA даёт нам контроль над адаптацией модели через небольшой набор гиперпараметров. Правильный выбор этих параметров — ключ к успеху. В этой теме мы разберём каждый гиперпараметр: **почему** мы выбираем именно так, **как** это влияет на обучение и **какие значения** работают на практике.

---

### 2.1. Целевые слои (target modules): что адаптировать?

В трансформере есть несколько типов матриц весов, к которым можно применить LoRA:

| Слой | Обозначение | Роль | Приоритет |
| :--- | :--- | :--- | :--- |
| **Query** | `W_q` | Проекция запросов в self-attention | ⭐⭐⭐⭐⭐ |
| **Key** | `W_k` | Проекция ключей в self-attention | ⭐⭐⭐ |
| **Value** | `W_v` | Проекция значений в self-attention | ⭐⭐⭐⭐⭐ |
| **Output** | `W_o` | Выходная проекция self-attention | ⭐⭐⭐⭐ |
| **MLP** | `W_ffn` | Проекции в feed-forward слое | ⭐⭐ |

#### Почему начинают с `W_q` и `W_v`?

Это не просто эмпирическое правило — оно основано на эксперименте из оригинальной статьи LoRA.

**Таблица 5 (GPT-3 175B, 18M параметров):**

| Адаптируемые слои | Ранг | WikiSQL | MultiNLI |
| :--- | :--- | :--- | :--- |
| Только `W_q` | 8 | 70.4% | 91.0% |
| Только `W_k` | 8 | 70.0% | 90.8% |
| Только `W_v` | 8 | 73.0% | 91.0% |
| Только `W_o` | 8 | 73.2% | 91.3% |
| `W_q` + `W_v` | 4 | **73.7%** | **91.3%** |
| `W_q` + `W_k` + `W_v` + `W_o` | 2 | 73.7% | 91.7% |

**Выводы из таблицы:**

1. **`W_q` и `W_v` дают наибольший прирост.** Адаптация `W_q` и `W_v` даёт лучшее качество, чем адаптация одного слоя с более высоким рангом.

2. **Адаптация `W_q` + `W_v` (r=4) лучше, чем только `W_q` (r=8).** Это означает, что важно адаптировать *больше типов слоёв*, даже с меньшим рангом.

3. **Добавление `W_k`, `W_o` даёт небольшой дополнительный прирост** (+0.4% на MultiNLI).

#### Практические рекомендации по выбору целевых слоёв:

| Уровень | target_modules | Когда применять |
| :--- | :--- | :--- |
| **Минимальный** | `["q_proj", "v_proj"]` | Начальные эксперименты, ограниченный бюджет |
| **Стандартный** | `["q_proj", "k_proj", "v_proj", "o_proj"]` | **Рекомендуемый для большинства задач** |
| **Расширенный** | `["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]` | Сложные задачи (изменение стиля, логики) |

**Важно:** названия слоёв зависят от модели!

| Модель | Проекции внимания | MLP-слои |
| :--- | :--- | :--- |
| **Llama 3** | `q_proj, k_proj, v_proj, o_proj` | `gate_proj, up_proj, down_proj` |
| **Mistral** | `q_proj, k_proj, v_proj, o_proj` | `gate_proj, up_proj, down_proj` |
| **Qwen 2.5** | `q_proj, k_proj, v_proj, o_proj` | `gate_proj, up_proj, down_proj` |
| **GPT-2** | `c_attn, c_proj` | `c_fc, c_proj` |
| **RoBERTa** | `query, key, value, output` | `intermediate, output` |

---

### 2.2. Гиперпараметры LoRA: значения по умолчанию

#### 2.2.1. Ранг `r`

| Значение | Параметров (d=4096) | Когда применять |
| :--- | :--- | :--- |
| **4** | ~65K | Очень простые задачи, быстрые эксперименты |
| **8** | ~131K | Простые задачи (классификация, базовое форматирование) |
| **16** | ~262K | **Стандартное значение по умолчанию** |
| **32** | ~524K | Сложные задачи (изменение стиля, сложная логика) |
| **64** | ~1M | Очень сложные задачи, приближение к Full FT |

**Рекомендация:** начинайте с **r=16** — это лучший баланс качества и скорости.

#### 2.2.2. Масштабирующий фактор `lora_alpha`

**Практическое правило:** `lora_alpha = 2 × r`

| r | lora_alpha | alpha/r |
| :--- | :--- | :--- |
| 4 | 8 | 2.0 |
| 8 | 16 | 2.0 |
| **16** | **32** | **2.0** |
| 32 | 64 | 2.0 |
| 64 | 128 | 2.0 |

**Почему `alpha = 2 * r`?** Это эмпирическое правило, которое обеспечивает стабильность градиентов. В оригинальной статье авторы пишут:

> *"We then scale ΔWx by α/r, where α is a constant in r. When optimizing with Adam, tuning α is roughly the same as tuning the learning rate if we scale the initialization appropriately."*

**Тонкая настройка:** если после обучения качество слишком низкое — попробуйте увеличить `alpha` (до 4×r). Если модель начинает галлюцинировать — уменьшите `alpha` (до 1×r).

#### 2.2.3. Dropout: `lora_dropout`

| Значение | Рекомендация |
| :--- | :--- |
| **0.0** | Если датасет большой (>10K примеров) |
| **0.05** | **Стандартное значение** |
| **0.1** | Если датасет маленький (<1K примеров) |

**Когда увеличивать dropout:** при малом размере датасета (риск переобучения).

**Когда уменьшать:** при большом датасете — dropout только замедляет обучение.

#### 2.2.4. Bias: `bias`

| Значение | Рекомендация |
| :--- | :--- |
| **"none"** | **Всегда!** Экономит память, не влияет на качество |
| "all" | Обучает все bias (больше параметров, почти без прироста) |
| "lora_only" | Обучает только bias в LoRA-слоях |

**Рекомендация:** всегда используйте `bias="none"`. Bias-параметры составляют ничтожную долю от всех весов, и их обучение не даёт значимого прироста, но увеличивает память.

---

### 2.3. Learning Rate: почему для LoRA выше?

| Режим | Learning Rate | Почему |
| :--- | :--- | :--- |
| **Full Fine-Tune** | 1e-5 – 5e-5 | Обновляются все веса — нужна осторожность |
| **LoRA (полная точность)** | **5e-5** | Адаптер учится с нуля, нужна более высокая LR |
| **QLoRA (4-bit)** | **2e-4** | Квантование требует более агрессивного обучения |

**Почему LoRA требует более высокую LR?**

1. **Адаптер инициализируется нулями.** В начале обучения адаптер не вносит изменений. Нужна более высокая LR, чтобы он начал обучаться.

2. **Меньше параметров.** Меньше параметров — меньше риск переобучения, можно использовать более высокую LR.

3. **Масштабирование α/r.** Коэффициент α/r уменьшает величину обновлений, поэтому LR можно делать выше.

**Почему QLoRA требует ещё более высокую LR (2e-4)?**

При 4-битном квантовании веса модели хранятся с меньшей точностью. Градиенты становятся более «шумными». Более высокая LR помогает преодолеть этот шум и эффективно обучать адаптер.

#### Практический совет по настройке LR:

```python
# Для LoRA (полная точность)
training_args = TrainingArguments(
    learning_rate=5e-5,      # Начать с 5e-5
)

# Для QLoRA (4-bit)
training_args = TrainingArguments(
    learning_rate=2e-4,      # Начать с 2e-4
)
```

**Если Loss не снижается** → увеличьте LR в 2–3 раза.

**Если Loss взлетает (NaN)** → уменьшите LR в 10 раз.

---

### 2.4. Количество эпох: почему 1–3?

**Почему так мало?**

1. **LoRA обучает мало параметров (<1%).** Они быстро адаптируются к данным.

2. **После 3 эпох риск переобучения резко возрастает.**

3. **Исследования показывают:** большинство задач сходятся за 1–2 эпохи.

#### Как следить за eval loss?

**Лучший индикатор — eval loss:**

```python
from transformers import TrainingArguments

training_args = TrainingArguments(
    evaluation_strategy="steps",  # Оценивать каждые N шагов
    eval_steps=100,               # Оценка каждые 100 шагов
    save_strategy="steps",
    save_steps=100,
    load_best_model_at_end=True,  # Загрузить лучший чекпоинт
    metric_for_best_model="eval_loss",
    greater_is_better=False,      # Меньше — лучше
)
```

**Как интерпретировать eval loss:**

| Ситуация | Что делать |
| :--- | :--- |
| **eval_loss падает** | ✅ Всё идёт по плану |
| **eval_loss растёт, train_loss падает** | ⚠️ Переобучение → остановить обучение |
| **eval_loss не падает** | ⚠️ Проблема: LR слишком низкий или данные плохие |

#### Когда можно увеличить число эпох?

1. **Очень маленький датасет (<500 примеров)** → риск переобучения, лучше не увеличивать.
2. **Очень сложная задача** → можно попробовать 5 эпох, но внимательно следить за eval loss.
3. **Мультизадачное обучение** → может потребоваться больше эпох.

**Рекомендация:** начните с **2 эпох** и смотрите на eval loss. Если он продолжает падать — добавьте ещё одну эпоху (максимум 3).

---

### 2.5. Полная шпаргалка по гиперпараметрам

| Параметр | Рекомендуемое значение | Когда менять |
| :--- | :--- | :--- |
| **target_modules** | `["q_proj", "v_proj"]` | Добавить `k_proj`, `o_proj`, MLP для сложных задач |
| **r (ранг)** | **16** | 8 — для простых, 32–64 — для сложных |
| **lora_alpha** | **32** (2 × r) | Увеличить при низком качестве, уменьшить при галлюцинациях |
| **lora_dropout** | **0.05** | 0.1 для малых данных, 0.0 для больших |
| **bias** | **"none"** | Всегда! |
| **learning_rate (LoRA)** | **5e-5** | Увеличить при медленном обучении, уменьшить при NaN |
| **learning_rate (QLoRA)** | **2e-4** | Увеличить при медленном обучении, уменьшить при NaN |
| **num_epochs** | **2** | 1–3, не больше! |
| **evaluation_strategy** | `"steps"` | Не отключайте — eval loss — главный индикатор |

---

### 2.6. Пример конфигурации (готовый код)

```python
from peft import LoraConfig, get_peft_model
from transformers import TrainingArguments

# 1. Конфигурация LoRA
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# 2. Конфигурация обучения
training_args = TrainingArguments(
    output_dir="./checkpoints",
    learning_rate=2e-4,        # QLoRA: 2e-4, LoRA: 5e-5
    num_train_epochs=2,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    evaluation_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=10,
    weight_decay=0.01,
    max_grad_norm=1.0,
)

# 3. Применяем LoRA
peft_model = get_peft_model(model, lora_config)
peft_model.print_trainable_parameters()
# Вывод: trainable params: 8.4M || all params: 7B || trainable%: 0.12%
```

---

### Итоговые выводы по теме 2

1. **Целевые слои:** начинайте с `["q_proj", "v_proj"]`. Для сложных задач добавляйте `k_proj`, `o_proj` и MLP.

2. **Гиперпараметры:**
   - `r=16` — стандарт
   - `lora_alpha=32` (2 × r)
   - `lora_dropout=0.05`
   - `bias="none"`

3. **Learning Rate:** LoRA требует более высокой LR (5e-5) чем Full FT. QLoRA — ещё выше (2e-4).

4. **Эпохи:** 1–3 эпохи. Следите за eval loss — это главный индикатор переобучения.

**Золотое правило:** если вы не знаете, с чего начать — используйте значения по умолчанию из этой шпаргалки. В 90% случаев они будут работать хорошо.